15_model_competition_ensemble_xai.ipynb

15.1  Freeze evaluation protocol
15.2  Base model expansion
15.3  Base model comparison
15.4  Tune Random Forest
15.5  Tune XGBoost
15.6  Compare base + tuned models
15.7  Select top two
15.8  Equal-weight ensemble
15.9  Optimize ensemble weights
15.10 Individual vs ensemble
15.11 SHAP — Model 1
15.12 SHAP — Model 2
15.13 SHAP — Ensemble interpretation
15.14 Final untouched holdout evaluation
15.15 Freeze final model + ensemble
15.16 Phase 15 checkpoint

In [ ]:
# ============================================================
# PHASE 15.1 — CORRECTED TRAIN / VALIDATION / FINAL HOLDOUT
# ============================================================
#
# Protocol:
#   70% chronological TRAIN
#   15% chronological VALIDATION
#   15% FINAL HOLDOUT
#
# IMPORTANT:
#   A true 1-hour embargo is enforced at both boundaries.
#
# The row immediately before each boundary is excluded from
# Train/Validation so that AQI_t_plus_1 cannot cross into the
# next partition.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from google.colab import drive


# ------------------------------------------------------------
# 1. MOUNT DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

REPORT_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports"
)

SPLIT_DATA_PATH = os.path.join(
    REPORT_DIR,
    "phase15_1_time_split_assignments.parquet"
)

SPLIT_CONFIG_PATH = os.path.join(
    REPORT_DIR,
    "phase15_1_evaluation_protocol.json"
)

os.makedirs(
    REPORT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}"
    )

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = (
    df
    .dropna(
        subset=[
            "Datetime",
            "AQI_t_plus_1"
        ]
    )
    .sort_values(
        [
            "Datetime",
            "StationId"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. BASIC CHECKS
# ------------------------------------------------------------

if "StationId" not in df.columns:
    raise ValueError(
        "StationId column missing."
    )

if "AQI_t_plus_1" not in df.columns:
    raise ValueError(
        "AQI_t_plus_1 target missing."
    )

station_count = df[
    "StationId"
].nunique()

if station_count != 10:
    raise ValueError(
        f"Expected 10 stations, found {station_count}."
    )


# ------------------------------------------------------------
# 5. UNIQUE TIMESTAMP SPLIT
# ------------------------------------------------------------

unique_times = np.sort(
    df["Datetime"]
    .dropna()
    .unique()
)

n_times = len(
    unique_times
)

train_boundary_idx = int(
    n_times * 0.70
)

validation_boundary_idx = int(
    n_times * 0.85
)

train_boundary = pd.Timestamp(
    unique_times[
        train_boundary_idx - 1
    ]
)

validation_boundary = pd.Timestamp(
    unique_times[
        validation_boundary_idx - 1
    ]
)


# ------------------------------------------------------------
# 6. TRUE 1-HOUR EMBARGO
# ------------------------------------------------------------

GAP = pd.Timedelta(
    hours=1
)

# Train stops BEFORE the boundary hour
train_end = (
    train_boundary - GAP
)

# Validation starts AFTER the boundary hour
validation_start = (
    train_boundary + GAP
)

# Validation stops BEFORE its boundary hour
validation_end = (
    validation_boundary - GAP
)

# Final holdout starts AFTER the boundary hour
final_holdout_start = (
    validation_boundary + GAP
)


# ------------------------------------------------------------
# 7. ASSIGN SPLITS
# ------------------------------------------------------------

df["Split"] = "UNUSED_GAP"

df.loc[
    df["Datetime"] <= train_end,
    "Split"
] = "TRAIN"

df.loc[
    (
        df["Datetime"] >= validation_start
    )
    &
    (
        df["Datetime"] <= validation_end
    ),
    "Split"
] = "VALIDATION"

df.loc[
    df["Datetime"] >= final_holdout_start,
    "Split"
] = "FINAL_HOLDOUT"


# ------------------------------------------------------------
# 8. EXTRACT SPLITS
# ------------------------------------------------------------

train_df = df[
    df["Split"] == "TRAIN"
].copy()

valid_df = df[
    df["Split"] == "VALIDATION"
].copy()

test_df = df[
    df["Split"] == "FINAL_HOLDOUT"
].copy()

gap_df = df[
    df["Split"] == "UNUSED_GAP"
].copy()


# ------------------------------------------------------------
# 9. TIMESTAMP OVERLAP CHECK
# ------------------------------------------------------------

train_times = set(
    train_df["Datetime"]
)

valid_times = set(
    valid_df["Datetime"]
)

test_times = set(
    test_df["Datetime"]
)

if train_times & valid_times:
    raise ValueError(
        "TRAIN and VALIDATION timestamps overlap."
    )

if valid_times & test_times:
    raise ValueError(
        "VALIDATION and FINAL HOLDOUT timestamps overlap."
    )

if train_times & test_times:
    raise ValueError(
        "TRAIN and FINAL HOLDOUT timestamps overlap."
    )


# ------------------------------------------------------------
# 10. VERIFY THE ACTUAL EMBARGO
# ------------------------------------------------------------

actual_train_gap = (
    valid_df["Datetime"].min()
    - train_df["Datetime"].max()
)

actual_validation_gap = (
    test_df["Datetime"].min()
    - valid_df["Datetime"].max()
)

print(
    "Train → Validation separation:",
    actual_train_gap
)

print(
    "Validation → Holdout separation:",
    actual_validation_gap
)

if actual_train_gap < pd.Timedelta(
    hours=1
):
    raise ValueError(
        "Train/validation embargo is less than 1 hour."
    )

if actual_validation_gap < pd.Timedelta(
    hours=1
):
    raise ValueError(
        "Validation/holdout embargo is less than 1 hour."
    )


# ------------------------------------------------------------
# 11. SPLIT SUMMARY
# ------------------------------------------------------------

def summarize_split(
    data,
    name
):

    if len(data) == 0:

        return {
            "Split": name,
            "Rows": 0,
            "Stations": 0,
            "Start": None,
            "End": None
        }

    return {
        "Split": name,
        "Rows": len(data),
        "Stations": data[
            "StationId"
        ].nunique(),
        "Start": str(
            data["Datetime"].min()
        ),
        "End": str(
            data["Datetime"].max()
        )
    }


summary = pd.DataFrame([
    summarize_split(
        train_df,
        "TRAIN"
    ),
    summarize_split(
        valid_df,
        "VALIDATION"
    ),
    summarize_split(
        test_df,
        "FINAL_HOLDOUT"
    ),
    summarize_split(
        gap_df,
        "UNUSED_GAP"
    )
])


# ------------------------------------------------------------
# 12. VERIFY ALL ROWS ACCOUNTED FOR
# ------------------------------------------------------------

assigned_rows = (
    len(train_df)
    + len(valid_df)
    + len(test_df)
    + len(gap_df)
)

if assigned_rows != len(df):
    raise ValueError(
        "Some rows were not assigned to a split."
    )


# ------------------------------------------------------------
# 13. SAVE PROTOCOL
# ------------------------------------------------------------

protocol = {

    "phase": "15",

    "step": "15.1",

    "split_method":
        "Chronological split by unique Datetime",

    "train_fraction":
        0.70,

    "validation_fraction":
        0.15,

    "final_holdout_fraction":
        0.15,

    "target":
        "AQI_t_plus_1",

    "stations":
        int(station_count),

    "temporal_embargo":
        "1 hour",

    "train": {
        "rows":
            int(len(train_df)),
        "start":
            str(
                train_df["Datetime"].min()
            ),
        "end":
            str(
                train_df["Datetime"].max()
            )
    },

    "validation": {
        "rows":
            int(len(valid_df)),
        "start":
            str(
                valid_df["Datetime"].min()
            ),
        "end":
            str(
                valid_df["Datetime"].max()
            )
    },

    "final_holdout": {
        "rows":
            int(len(test_df)),
        "start":
            str(
                test_df["Datetime"].min()
            ),
        "end":
            str(
                test_df["Datetime"].max()
            ),
        "status":
            "FROZEN — NOT USED FOR MODEL SELECTION"
    },

    "unused_gap": {
        "rows":
            int(len(gap_df)),
        "start":
            (
                str(
                    gap_df["Datetime"].min()
                )
                if len(gap_df)
                else None
            ),
        "end":
            (
                str(
                    gap_df["Datetime"].max()
                )
                if len(gap_df)
                else None
            )
    }
}


# ------------------------------------------------------------
# 14. SAVE SPLIT ASSIGNMENTS
# ------------------------------------------------------------

df[
    [
        "StationId",
        "Datetime",
        "Split"
    ]
].to_parquet(
    SPLIT_DATA_PATH,
    index=False
)


# ------------------------------------------------------------
# 15. SAVE PROTOCOL
# ------------------------------------------------------------

with open(
    SPLIT_CONFIG_PATH,
    "w"
) as f:
    json.dump(
        protocol,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 16. DISPLAY
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 15.1 — CORRECTED EVALUATION PROTOCOL")
print("=" * 75)

print(
    "Total rows       :",
    len(df)
)

print(
    "Unique timestamps:",
    n_times
)

print(
    "Stations         :",
    station_count
)

print(
    "Target           :",
    "AQI_t_plus_1"
)

print("\nSplit summary:")
display(summary)

print("\nActual embargo:")
print(
    "Train → Validation:",
    actual_train_gap
)
print(
    "Validation → Holdout:",
    actual_validation_gap
)

print("\nFinal holdout:")
print(
    "FROZEN — DO NOT USE FOR MODEL SELECTION"
)

print("\nSaved:")
print(SPLIT_DATA_PATH)

print(SPLIT_CONFIG_PATH)

print("=" * 75)
print("PHASE 15.1 CORRECTED AND COMPLETED")
print("NEXT: PHASE 15.2 — BASE MODEL EXPANSION")
print("=" * 75)

In [ ]:
# ============================================================
# PHASE 15.2 — BASE MODEL EXPANSION
# ============================================================
#
# Base models:
#   1. Linear Regression
#   2. Decision Tree
#   3. Random Forest
#   4. Extra Trees
#   5. Gradient Boosting
#   6. HistGradientBoosting
#   7. AdaBoost
#   8. XGBoost
#
# Evaluation:
#   TRAIN -> fit
#   VALIDATION -> compare
#   FINAL_HOLDOUT -> NEVER TOUCHED
#
# Resume-safe:
#   Results are saved after EVERY completed model.
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# XGBoost
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False


warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_2_base_model_results.csv"
)


# ------------------------------------------------------------
# 2. VERIFY FILES
# ------------------------------------------------------------

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Feature-engineered dataset not found:\n{DATA_PATH}"
    )

if not os.path.exists(SPLIT_PATH):
    raise FileNotFoundError(
        f"Phase 15.1 split assignments not found:\n{SPLIT_PATH}"
    )


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 4. LOAD FROZEN PHASE 15.1 SPLIT
# ------------------------------------------------------------

split_df = pd.read_parquet(
    SPLIT_PATH
)

split_df["Datetime"] = pd.to_datetime(
    split_df["Datetime"],
    errors="coerce"
)

# Remove any existing split column in case it exists
if "Split" in df.columns:
    df = df.drop(
        columns=["Split"]
    )

# Merge the frozen assignment
df = df.merge(
    split_df,
    on=[
        "StationId",
        "Datetime"
    ],
    how="left",
    validate="one_to_one"
)

if df["Split"].isna().any():
    missing_split_rows = int(
        df["Split"].isna().sum()
    )

    raise ValueError(
        f"{missing_split_rows} rows did not receive "
        "a Phase 15.1 split assignment."
    )


# ------------------------------------------------------------
# 5. TARGET
# ------------------------------------------------------------

TARGET = "AQI_t_plus_1"


if TARGET not in df.columns:
    raise ValueError(
        f"Target column missing: {TARGET}"
    )


# ------------------------------------------------------------
# 6. FINAL 73-FEATURE CONTRACT
# ------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in current_features
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

MODEL_FEATURES = list(
    dict.fromkeys(
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + event_features
        + station_features
    )
)

if len(MODEL_FEATURES) != 73:
    raise ValueError(
        f"Expected 73 model features, found "
        f"{len(MODEL_FEATURES)}."
    )


# ------------------------------------------------------------
# 7. VERIFY MODEL FEATURES
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "Missing model features:\n"
        + "\n".join(missing_features)
    )


# ------------------------------------------------------------
# 8. CREATE TRAIN / VALIDATION DATA
# ------------------------------------------------------------

train_df = (
    df[
        df["Split"] == "TRAIN"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

valid_df = (
    df[
        df["Split"] == "VALIDATION"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

# Final holdout is deliberately NOT loaded into any model
# training/evaluation operation in this step.

print("=" * 75)
print("PHASE 15.2 — BASE MODEL EXPANSION")
print("=" * 75)

print(
    "Train rows      :",
    len(train_df)
)

print(
    "Validation rows :",
    len(valid_df)
)

print(
    "Final holdout   : FROZEN / UNTOUCHED"
)

print(
    "Features        :",
    len(MODEL_FEATURES)
)

print(
    "Target          :",
    TARGET
)


# ------------------------------------------------------------
# 9. REMOVE INVALID TARGETS
# ------------------------------------------------------------

train_df = train_df[
    pd.to_numeric(
        train_df[TARGET],
        errors="coerce"
    ).notna()
].copy()

valid_df = valid_df[
    pd.to_numeric(
        valid_df[TARGET],
        errors="coerce"
    ).notna()
].copy()


# ------------------------------------------------------------
# 10. X / y
# ------------------------------------------------------------

X_train = train_df[
    MODEL_FEATURES
].copy()

y_train = pd.to_numeric(
    train_df[TARGET],
    errors="coerce"
)

X_valid = valid_df[
    MODEL_FEATURES
].copy()

y_valid = pd.to_numeric(
    valid_df[TARGET],
    errors="coerce"
)


# ------------------------------------------------------------
# 11. FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "StationId"
]

numeric_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in categorical_features
]


# ------------------------------------------------------------
# 12. COMMON PREPROCESSOR
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# 13. BASE MODEL BANK
# ------------------------------------------------------------
#
# These are BASE configurations, not the tuned models.
# Tuning is performed later in Phase 15.4 / 15.5.
# ------------------------------------------------------------

models = {

    "Linear Regression":
        LinearRegression(),

    "Decision Tree":
        DecisionTreeRegressor(
            random_state=42,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1
        ),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=1.0
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features=1.0
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),

    "HistGradientBoosting":
        HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.0,
            random_state=42
        ),

    "AdaBoost":
        AdaBoostRegressor(
            n_estimators=100,
            learning_rate=0.05,
            random_state=42
        )
}

# Add XGBoost only if installed
if XGBOOST_AVAILABLE:

    models["XGBoost"] = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

else:

    print(
        "\nWARNING: XGBoost is not currently installed."
    )

    print(
        "XGBoost will be handled separately before "
        "the tuning stage."
    )


# ------------------------------------------------------------
# 14. LOAD EXISTING RESULTS IF AVAILABLE
# ------------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    results_df = pd.read_csv(
        RESULTS_PATH
    )

    completed_models = set(
        results_df["Model"].astype(str)
    )

    print(
        "\nExisting completed models:",
        len(completed_models)
    )

else:

    results_df = pd.DataFrame()

    completed_models = set()


# ------------------------------------------------------------
# 15. MODEL EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_base_model(
    model_name,
    estimator
):

    print("\n" + "=" * 65)
    print(
        "TRAINING BASE MODEL:",
        model_name
    )
    print("=" * 65)

    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            estimator
        )
    ])

    # Fit ONLY on training data
    pipeline.fit(
        X_train,
        y_train
    )

    # Predict ONLY validation data
    predictions = pipeline.predict(
        X_valid
    )

    actual = y_valid.to_numpy()
    predictions = np.asarray(
        predictions
    )

    valid_mask = (
        np.isfinite(actual)
        &
        np.isfinite(predictions)
    )

    if valid_mask.sum() == 0:
        raise ValueError(
            f"No valid validation predictions for "
            f"{model_name}."
        )

    actual = actual[
        valid_mask
    ]

    predictions = predictions[
        valid_mask
    ]

    mae = mean_absolute_error(
        actual,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predictions
        )
    )

    r2 = r2_score(
        actual,
        predictions
    )

    accuracy_10 = (
        np.mean(
            np.abs(
                actual - predictions
            ) <= 10
        ) * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(
                actual - predictions
            ) <= 20
        ) * 100
    )

    result = {
        "Model": model_name,
        "Feature Count": len(
            MODEL_FEATURES
        ),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
        "Accuracy ±10 AQI": float(
            accuracy_10
        ),
        "Accuracy ±20 AQI": float(
            accuracy_20
        ),
        "Train Records": int(
            len(actual)
            if False
            else len(X_train)
        ),
        "Validation Records": int(
            len(actual)
        )
    }

    print(
        f"MAE              : {mae:.6f}"
    )

    print(
        f"RMSE             : {rmse:.6f}"
    )

    print(
        f"R²               : {r2:.6f}"
    )

    print(
        f"Accuracy ±10 AQI : {accuracy_10:.4f}%"
    )

    print(
        f"Accuracy ±20 AQI : {accuracy_20:.4f}%"
    )

    return result


# ------------------------------------------------------------
# 16. TRAIN ALL BASE MODELS
# ------------------------------------------------------------

for model_name, estimator in models.items():

    if model_name in completed_models:

        print(
            f"\nSKIPPING already completed: "
            f"{model_name}"
        )

        continue

    result = evaluate_base_model(
        model_name,
        estimator
    )

    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    # SAVE IMMEDIATELY
    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    completed_models.add(
        model_name
    )

    print(
        f"\n✓ Saved result for {model_name}"
    )


# ------------------------------------------------------------
# 17. FINAL BASE MODEL LEADERBOARD
# ------------------------------------------------------------

if len(results_df) > 0:

    results_df = (
        results_df
        .sort_values(
            by="RMSE",
            ascending=True
        )
        .reset_index(drop=True)
    )

    results_df["Rank"] = (
        np.arange(
            1,
            len(results_df) + 1
        )
    )

    rank = results_df.pop(
        "Rank"
    )

    results_df.insert(
        0,
        "Rank",
        rank
    )

    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    print("\n" + "=" * 75)
    print("PHASE 15.2 — BASE MODEL LEADERBOARD")
    print("=" * 75)

    display(
        results_df[
            [
                "Rank",
                "Model",
                "Feature Count",
                "MAE",
                "RMSE",
                "R2",
                "Accuracy ±10 AQI",
                "Accuracy ±20 AQI",
                "Validation Records"
            ]
        ]
    )


# ------------------------------------------------------------
# 18. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.2 COMPLETED")
print("=" * 75)

print(
    "Base-model results saved to:"
)

print(
    RESULTS_PATH
)

print(
    "\nFinal holdout status:"
)

print(
    "FROZEN — NOT USED"
)

print("=" * 75)

In [ ]:
# ============================================================
# PHASE 15.3 — BASE MODEL LEADERBOARD ANALYSIS
# CORRECTED / RESTART-SAFE VERSION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

BASE_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_2_base_model_results.csv"
)

LEADERBOARD_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_3_base_model_leaderboard.csv"
)

CONFIG_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_3_model_competition_config.json"
)

# ------------------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------------------

for path in [
    DATA_PATH,
    SPLIT_PATH,
    BASE_RESULTS_PATH
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

# ------------------------------------------------------------
# 3. LOAD PHASE 15.2 RESULTS
# ------------------------------------------------------------

base_results = pd.read_csv(
    BASE_RESULTS_PATH
)

required_columns = [
    "Model",
    "Feature Count",
    "MAE",
    "RMSE",
    "R2",
    "Accuracy ±10 AQI",
    "Accuracy ±20 AQI",
    "Validation Records"
]

missing_columns = [
    col
    for col in required_columns
    if col not in base_results.columns
]

if missing_columns:
    raise ValueError(
        "Missing expected columns in Phase 15.2 results:\n"
        + "\n".join(missing_columns)
    )

# Remove old rank/index columns from previous outputs
for col in [
    "Rank",
    "Unnamed: 0"
]:
    if col in base_results.columns:
        base_results = base_results.drop(
            columns=[col]
        )

# ------------------------------------------------------------
# 4. LOAD DATA + FROZEN SPLIT
# ------------------------------------------------------------

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

split_df = pd.read_parquet(
    SPLIT_PATH
)

split_df["Datetime"] = pd.to_datetime(
    split_df["Datetime"],
    errors="coerce"
)

# Remove any old Split column
if "Split" in df.columns:
    df = df.drop(
        columns=["Split"]
    )

df = df.merge(
    split_df,
    on=[
        "StationId",
        "Datetime"
    ],
    how="left",
    validate="one_to_one"
)

if df["Split"].isna().any():
    raise ValueError(
        "Some rows do not have a Phase 15.1 split assignment."
    )

TARGET = "AQI_t_plus_1"

# ------------------------------------------------------------
# 5. CALCULATE PERSISTENCE BASELINE
# ------------------------------------------------------------

validation_df = (
    df[
        df["Split"] == "VALIDATION"
    ]
    .copy()
)

validation_df[TARGET] = pd.to_numeric(
    validation_df[TARGET],
    errors="coerce"
)

validation_df["AQI"] = pd.to_numeric(
    validation_df["AQI"],
    errors="coerce"
)

persistence_df = (
    validation_df[
        [TARGET, "AQI"]
    ]
    .dropna()
)

actual = persistence_df[
    TARGET
].to_numpy()

predicted = persistence_df[
    "AQI"
].to_numpy()

if len(actual) == 0:
    raise ValueError(
        "No valid observations for persistence baseline."
    )

persistence_result = {
    "Model": "Persistence Baseline",
    "Feature Count": 1,
    "MAE": float(
        mean_absolute_error(
            actual,
            predicted
        )
    ),
    "RMSE": float(
        np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )
    ),
    "R2": float(
        r2_score(
            actual,
            predicted
        )
    ),
    "Accuracy ±10 AQI": float(
        np.mean(
            np.abs(
                actual - predicted
            ) <= 10
        ) * 100
    ),
    "Accuracy ±20 AQI": float(
        np.mean(
            np.abs(
                actual - predicted
            ) <= 20
        ) * 100
    ),
    "Validation Records": int(
        len(actual)
    )
}

# ------------------------------------------------------------
# 6. COMBINE PERSISTENCE + BASE MODELS
# ------------------------------------------------------------

leaderboard = pd.concat(
    [
        pd.DataFrame(
            [persistence_result]
        ),
        base_results
    ],
    ignore_index=True
)

# Remove duplicate models if any
leaderboard = (
    leaderboard
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. SORT BY PRIMARY METRIC
# ------------------------------------------------------------

leaderboard = (
    leaderboard
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

# IMPORTANT:
# Remove Rank if it somehow survived the previous steps.
if "Rank" in leaderboard.columns:
    leaderboard = leaderboard.drop(
        columns=["Rank"]
    )

# Add a fresh rank
leaderboard.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(leaderboard) + 1
    )
)

# ------------------------------------------------------------
# 8. DISPLAY FULL LEADERBOARD
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 15.3 — BASE MODEL LEADERBOARD")
print("=" * 75)

display(
    leaderboard[
        [
            "Rank",
            "Model",
            "Feature Count",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI",
            "Validation Records"
        ]
    ]
)

# ------------------------------------------------------------
# 9. MACHINE-LEARNING-ONLY LEADERBOARD
# ------------------------------------------------------------

ml_only = (
    leaderboard[
        leaderboard["Model"]
        != "Persistence Baseline"
    ]
    .copy()
)

ml_only = (
    ml_only
    .sort_values(
        "RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("TOP MACHINE-LEARNING BASE MODELS")
print("=" * 75)

display(
    ml_only[
        [
            "Rank",
            "Model",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI"
        ]
    ]
)

# ------------------------------------------------------------
# 10. MODELS TO TUNE
# ------------------------------------------------------------

models_to_tune = [
    "Random Forest",
    "XGBoost"
]

# Current base-model competitors retained as benchmarks
benchmark_models = [
    model
    for model in ml_only["Model"].tolist()
    if model not in models_to_tune
]

# ------------------------------------------------------------
# 11. SAVE LEADERBOARD
# ------------------------------------------------------------

leaderboard.to_csv(
    LEADERBOARD_PATH,
    index=False
)

# ------------------------------------------------------------
# 12. SAVE COMPETITION CONFIG
# ------------------------------------------------------------

competition_config = {
    "phase": "15",
    "step": "15.3",

    "model_feature_count": 73,

    "primary_metric": "RMSE",

    "secondary_metrics": [
        "MAE",
        "R2",
        "Accuracy ±10 AQI",
        "Accuracy ±20 AQI"
    ],

    "base_models": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Extra Trees",
        "Gradient Boosting",
        "HistGradientBoosting",
        "AdaBoost",
        "XGBoost"
    ],

    "persistence_baseline":
        "Included as naive reference",

    "models_to_tune":
        models_to_tune,

    "benchmark_models":
        benchmark_models,

    "final_holdout":
        "FROZEN — NOT USED FOR MODEL SELECTION"
}

with open(
    CONFIG_PATH,
    "w"
) as f:
    json.dump(
        competition_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. SUMMARY
# ------------------------------------------------------------

best_base = ml_only.iloc[0]

print("\n" + "=" * 75)
print("PHASE 15.3 SUMMARY")
print("=" * 75)

print(
    "Best base model by RMSE :",
    best_base["Model"]
)

print(
    "Best base RMSE         :",
    round(
        float(best_base["RMSE"]),
        6
    )
)

print(
    "\nModels to tune:"
)

for model in models_to_tune:
    print(
        "  -",
        model
    )

print(
    "\nFinal holdout:",
    "FROZEN / UNTOUCHED"
)

print(
    "\nLeaderboard saved:"
)

print(
    LEADERBOARD_PATH
)

print(
    "\nCompetition config saved:"
)

print(
    CONFIG_PATH
)

print("=" * 75)
print("PHASE 15.3 COMPLETED")
print("NEXT: PHASE 15.4 — TUNED RANDOM FOREST")
print("=" * 75)

In [ ]:
# ============================================================
# PHASE 15 CHECKPOINT — STOPPED AFTER STEP 15.3
# ============================================================

import os
import json
from datetime import datetime

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "phase": "Phase 15 — Expanded Model Competition, Ensemble & XAI",

    "completed_until": "15.3",
    "resume_from": "15.4",

    "target": "AQI_t_plus_1",
    "stations": 10,
    "features": 73,

    "evaluation_protocol": {
        "train": "70% chronological",
        "validation": "15% chronological",
        "final_holdout": "15% chronological",
        "temporal_embargo": "1 hour"
    },

    "completed_steps": [
        "15.1",
        "15.2",
        "15.3"
    ],

    "best_base_model": "Extra Trees",
    "best_base_rmse": 8.538368,

    "models_to_tune": [
        "Random Forest",
        "XGBoost"
    ],

    "final_holdout_status":
        "FROZEN — NOT USED FOR MODEL SELECTION",

    "status":
        "STOPPED AFTER 15.3",

    "timestamp":
        datetime.now().isoformat()
}

CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "phase15_checkpoint_step15_3.json"
)

SESSION_STATUS_PATH = os.path.join(
    CHECKPOINT_DIR,
    "session_status.json"
)

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(checkpoint, f, indent=4)

with open(SESSION_STATUS_PATH, "w") as f:
    json.dump(checkpoint, f, indent=4)

print("=" * 70)
print("PHASE 15 CHECKPOINT SAVED")
print("=" * 70)
print("Completed through :", "15.3")
print("Resume from       :", "15.4")
print("Best base model   :", "Extra Trees")
print("Base RMSE         :", 8.538368)
print("Next              :", "Tuned Random Forest")
print("=" * 70)
print("SAFE TO DISCONNECT COLAB")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 15.4 — NEW RUNTIME DRIVE CONNECTION
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

print("=" * 70)
print("GOOGLE DRIVE CONNECTED")
print("=" * 70)
print("Project root:", PROJECT_ROOT)
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 15.4 — FAST TARGETED RANDOM FOREST TUNING
# ============================================================
#
# Strategy:
#   6 targeted RF configurations
#   1 frozen chronological VALIDATION set
#
# FINAL_HOLDOUT remains completely untouched.
#
# Search stage:
#   100 trees per candidate for speed
#
# Winner:
#   Refit using 300 trees on complete TRAIN
#
# Primary selection metric:
#   Validation RMSE
#
# Supporting:
#   MAE
#   R²
#   Accuracy ±10 AQI
#   Accuracy ±20 AQI
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_4_fast_rf_tuning_results.csv"
)

PARAMS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_4_best_rf_params.json"
)

VALIDATION_RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_4_tuned_rf_validation_result.json"
)

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "phase15_4_tuned_random_forest.joblib"
)

os.makedirs(
    os.path.dirname(RESULTS_PATH),
    exist_ok=True
)

os.makedirs(
    os.path.dirname(MODEL_PATH),
    exist_ok=True
)


# ------------------------------------------------------------
# 2. VERIFY FILES
# ------------------------------------------------------------

for path in [
    DATA_PATH,
    SPLIT_PATH
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 4. LOAD FROZEN PHASE 15.1 SPLIT
# ------------------------------------------------------------

split_df = pd.read_parquet(
    SPLIT_PATH
)

split_df["Datetime"] = pd.to_datetime(
    split_df["Datetime"],
    errors="coerce"
)

if "Split" in df.columns:
    df = df.drop(
        columns=["Split"]
    )

df = df.merge(
    split_df,
    on=[
        "StationId",
        "Datetime"
    ],
    how="left",
    validate="one_to_one"
)

if df["Split"].isna().any():
    raise ValueError(
        "Some rows do not have a Phase 15.1 split assignment."
    )


# ------------------------------------------------------------
# 5. TARGET
# ------------------------------------------------------------

TARGET = "AQI_t_plus_1"


# ------------------------------------------------------------
# 6. FINAL 73-FEATURE CONTRACT
# ------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in current_features
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

MODEL_FEATURES = list(
    dict.fromkeys(
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + event_features
        + station_features
    )
)

if len(MODEL_FEATURES) != 73:
    raise ValueError(
        f"Expected 73 features, found {len(MODEL_FEATURES)}."
    )


# ------------------------------------------------------------
# 7. TRAIN / VALIDATION ONLY
# ------------------------------------------------------------

train_df = (
    df[
        df["Split"] == "TRAIN"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

valid_df = (
    df[
        df["Split"] == "VALIDATION"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

# Remove invalid target rows only
train_df = train_df[
    pd.to_numeric(
        train_df[TARGET],
        errors="coerce"
    ).notna()
].copy()

valid_df = valid_df[
    pd.to_numeric(
        valid_df[TARGET],
        errors="coerce"
    ).notna()
].copy()


# ------------------------------------------------------------
# 8. X / y
# ------------------------------------------------------------

X_train = train_df[
    MODEL_FEATURES
].copy()

y_train = pd.to_numeric(
    train_df[TARGET],
    errors="coerce"
)

X_valid = valid_df[
    MODEL_FEATURES
].copy()

y_valid = pd.to_numeric(
    valid_df[TARGET],
    errors="coerce"
)


# ------------------------------------------------------------
# 9. FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "StationId"
]

numeric_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in categorical_features
]


# ------------------------------------------------------------
# 10. PIPELINE BUILDER
# ------------------------------------------------------------

def build_rf_pipeline(params):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])


# ------------------------------------------------------------
# 11. FAST TARGETED SEARCH SPACE
# ------------------------------------------------------------
#
# 100 trees during search.
#
# Configuration 2 is the previous strong Phase 13B setting.
# ------------------------------------------------------------

rf_candidates = [

    {
        "Candidate ID": 1,
        "n_estimators": 100,
        "max_depth": 18,
        "min_samples_split": 2,
        "min_samples_leaf": 10,
        "max_features": 0.7
    },

    {
        "Candidate ID": 2,
        "n_estimators": 100,
        "max_depth": 22,
        "min_samples_split": 2,
        "min_samples_leaf": 10,
        "max_features": 0.7
    },

    {
        "Candidate ID": 3,
        "n_estimators": 100,
        "max_depth": 26,
        "min_samples_split": 2,
        "min_samples_leaf": 10,
        "max_features": 0.7
    },

    {
        "Candidate ID": 4,
        "n_estimators": 100,
        "max_depth": 22,
        "min_samples_split": 2,
        "min_samples_leaf": 5,
        "max_features": 0.7
    },

    {
        "Candidate ID": 5,
        "n_estimators": 100,
        "max_depth": 22,
        "min_samples_split": 2,
        "min_samples_leaf": 15,
        "max_features": 0.7
    },

    {
        "Candidate ID": 6,
        "n_estimators": 100,
        "max_depth": 22,
        "min_samples_split": 2,
        "min_samples_leaf": 10,
        "max_features": 0.5
    }
]


# ------------------------------------------------------------
# 12. RESUME-SAFE RESULT STORAGE
# ------------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    tuning_results = pd.read_csv(
        RESULTS_PATH
    )

else:

    tuning_results = pd.DataFrame()

if (
    len(tuning_results) > 0
    and "Candidate ID" in tuning_results.columns
):

    completed_ids = set(
        tuning_results[
            "Candidate ID"
        ]
        .astype(int)
        .tolist()
    )

else:

    completed_ids = set()


# ------------------------------------------------------------
# 13. RUN FAST SEARCH
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 15.4 — FAST TARGETED RANDOM FOREST TUNING")
print("=" * 75)

print(
    "Training rows   :",
    len(X_train)
)

print(
    "Validation rows  :",
    len(X_valid)
)

print(
    "Features         :",
    len(MODEL_FEATURES)
)

print(
    "Candidates       :",
    len(rf_candidates)
)

print(
    "Trees per search candidate: 100"
)

print(
    "Final holdout    : FROZEN / UNTOUCHED"
)

print("=" * 75)


for candidate in rf_candidates:

    candidate_id = candidate[
        "Candidate ID"
    ]

    if candidate_id in completed_ids:

        print(
            f"\nSKIPPING candidate "
            f"{candidate_id} — already completed"
        )

        continue

    params = {
        key: value
        for key, value in candidate.items()
        if key != "Candidate ID"
    }

    print("\n" + "=" * 65)

    print(
        f"CANDIDATE {candidate_id}/"
        f"{len(rf_candidates)}"
    )

    print(
        "Parameters:",
        params
    )

    print("=" * 65)

    pipeline = build_rf_pipeline(
        params
    )

    # Fit ONLY on TRAIN
    pipeline.fit(
        X_train,
        y_train
    )

    # Predict VALIDATION
    prediction = pipeline.predict(
        X_valid
    )

    actual = y_valid.to_numpy()

    prediction = np.asarray(
        prediction
    )

    mask = (
        np.isfinite(actual)
        &
        np.isfinite(prediction)
    )

    if mask.sum() == 0:
        raise ValueError(
            f"No valid predictions for candidate {candidate_id}."
        )

    actual_valid = actual[
        mask
    ]

    prediction_valid = prediction[
        mask
    ]

    mae = mean_absolute_error(
        actual_valid,
        prediction_valid
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_valid,
            prediction_valid
        )
    )

    r2 = r2_score(
        actual_valid,
        prediction_valid
    )

    accuracy_10 = (
        np.mean(
            np.abs(
                actual_valid
                - prediction_valid
            ) <= 10
        ) * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(
                actual_valid
                - prediction_valid
            ) <= 20
        ) * 100
    )

    result = {
        "Candidate ID":
            candidate_id,

        "n_estimators":
            params["n_estimators"],

        "max_depth":
            params["max_depth"],

        "min_samples_split":
            params["min_samples_split"],

        "min_samples_leaf":
            params["min_samples_leaf"],

        "max_features":
            params["max_features"],

        "MAE":
            float(mae),

        "RMSE":
            float(rmse),

        "R2":
            float(r2),

        "Accuracy ±10 AQI":
            float(accuracy_10),

        "Accuracy ±20 AQI":
            float(accuracy_20),

        "Validation Records":
            int(len(actual_valid))
    }

    tuning_results = pd.concat(
        [
            tuning_results,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    # SAVE IMMEDIATELY
    tuning_results.to_csv(
        RESULTS_PATH,
        index=False
    )

    completed_ids.add(
        candidate_id
    )

    print(
        f"\n✓ Candidate {candidate_id} completed"
    )

    print(
        f"  MAE              : {mae:.6f}"
    )

    print(
        f"  RMSE             : {rmse:.6f}"
    )

    print(
        f"  R²               : {r2:.6f}"
    )

    print(
        f"  Accuracy ±10 AQI : {accuracy_10:.4f}%"
    )

    print(
        f"  Accuracy ±20 AQI : {accuracy_20:.4f}%"
    )


# ------------------------------------------------------------
# 14. RANK SEARCH RESULTS
# ------------------------------------------------------------

tuning_results = (
    tuning_results
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

tuning_results["Rank"] = np.arange(
    1,
    len(tuning_results) + 1
)

# Move Rank to first column
rank_col = tuning_results.pop(
    "Rank"
)

tuning_results.insert(
    0,
    "Rank",
    rank_col
)

tuning_results.to_csv(
    RESULTS_PATH,
    index=False
)


# ------------------------------------------------------------
# 15. DISPLAY SEARCH LEADERBOARD
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.4 — RF SEARCH LEADERBOARD")
print("=" * 75)

display(
    tuning_results[
        [
            "Rank",
            "Candidate ID",
            "n_estimators",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI"
        ]
    ]
)


# ------------------------------------------------------------
# 16. SELECT BEST CONFIGURATION
# ------------------------------------------------------------

best = (
    tuning_results
    .sort_values(
        "RMSE",
        ascending=True
    )
    .iloc[0]
)

best_params = {
    "n_estimators":
        int(best["n_estimators"]),

    "max_depth":
        (
            None
            if pd.isna(best["max_depth"])
            else int(best["max_depth"])
        ),

    "min_samples_split":
        int(best["min_samples_split"]),

    "min_samples_leaf":
        int(best["min_samples_leaf"]),

    "max_features":
        float(best["max_features"])
}


# ------------------------------------------------------------
# 17. REFIT WINNER WITH 300 TREES
# ------------------------------------------------------------
#
# The fast search used 100 trees only to identify the
# structural configuration.
#
# The selected configuration is now refit at higher capacity.
# ------------------------------------------------------------

final_rf_params = best_params.copy()

final_rf_params[
    "n_estimators"
] = 300

print("\n" + "=" * 75)
print("REFITTING BEST RF CONFIGURATION WITH 300 TREES")
print("=" * 75)

print(
    "Selected search candidate:",
    int(best["Candidate ID"])
)

print(
    "Structural parameters:"
)

for key, value in final_rf_params.items():
    print(
        f"  {key}: {value}"
    )

final_pipeline = build_rf_pipeline(
    final_rf_params
)

final_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 18. FINAL VALIDATION EVALUATION
# ------------------------------------------------------------

final_prediction = final_pipeline.predict(
    X_valid
)

actual = y_valid.to_numpy()

final_prediction = np.asarray(
    final_prediction
)

mask = (
    np.isfinite(actual)
    &
    np.isfinite(final_prediction)
)

actual = actual[
    mask
]

final_prediction = final_prediction[
    mask
]

final_mae = mean_absolute_error(
    actual,
    final_prediction
)

final_rmse = np.sqrt(
    mean_squared_error(
        actual,
        final_prediction
    )
)

final_r2 = r2_score(
    actual,
    final_prediction
)

final_accuracy_10 = (
    np.mean(
        np.abs(
            actual - final_prediction
        ) <= 10
    ) * 100
)

final_accuracy_20 = (
    np.mean(
        np.abs(
            actual - final_prediction
        ) <= 20
    ) * 100
)


# ------------------------------------------------------------
# 19. LOAD BASE RF RESULT
# ------------------------------------------------------------

BASE_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_2_base_model_results.csv"
)

base_results = pd.read_csv(
    BASE_RESULTS_PATH
)

base_rf = base_results[
    base_results["Model"]
    == "Random Forest"
]

if len(base_rf) == 1:

    base_rf = base_rf.iloc[0]

    base_rmse = float(
        base_rf["RMSE"]
    )

    base_mae = float(
        base_rf["MAE"]
    )

    base_accuracy_10 = float(
        base_rf["Accuracy ±10 AQI"]
    )

else:

    base_rmse = np.nan
    base_mae = np.nan
    base_accuracy_10 = np.nan


# ------------------------------------------------------------
# 20. SAVE BEST PARAMETERS
# ------------------------------------------------------------

params_record = {
    "phase": "15",
    "step": "15.4",

    "model": "Tuned Random Forest",

    "search_strategy":
        "Fast targeted validation search",

    "search_candidates":
        len(rf_candidates),

    "search_trees":
        100,

    "final_trees":
        300,

    "best_candidate_id":
        int(best["Candidate ID"]),

    "best_search_rmse":
        float(best["RMSE"]),

    "best_params":
        final_rf_params,

    "final_holdout_status":
        "FROZEN — NOT USED"
}

with open(
    PARAMS_PATH,
    "w"
) as f:

    json.dump(
        params_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 21. SAVE VALIDATION RESULT
# ------------------------------------------------------------

validation_record = {

    "phase": "15",
    "step": "15.4",

    "model":
        "Tuned Random Forest",

    "feature_count":
        len(MODEL_FEATURES),

    "validation_records":
        int(len(actual)),

    "validation_mae":
        float(final_mae),

    "validation_rmse":
        float(final_rmse),

    "validation_r2":
        float(final_r2),

    "validation_accuracy_plus_minus_10":
        float(final_accuracy_10),

    "validation_accuracy_plus_minus_20":
        float(final_accuracy_20),

    "base_random_forest": {

        "mae":
            base_mae,

        "rmse":
            base_rmse,

        "accuracy_plus_minus_10":
            base_accuracy_10
    },

    "tuned_minus_base": {

        "mae":
            (
                final_mae - base_mae
                if np.isfinite(base_mae)
                else None
            ),

        "rmse":
            (
                final_rmse - base_rmse
                if np.isfinite(base_rmse)
                else None
            ),

        "accuracy_plus_minus_10":
            (
                final_accuracy_10
                - base_accuracy_10
                if np.isfinite(
                    base_accuracy_10
                )
                else None
            )
    },

    "final_holdout_status":
        "FROZEN — NOT USED"
}

with open(
    VALIDATION_RESULT_PATH,
    "w"
) as f:

    json.dump(
        validation_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 22. SAVE MODEL
# ------------------------------------------------------------

joblib.dump(
    final_pipeline,
    MODEL_PATH
)


# ------------------------------------------------------------
# 23. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.4 — TUNED RANDOM FOREST COMPLETED")
print("=" * 75)

print(
    "Selected candidate:",
    int(best["Candidate ID"])
)

print(
    "\nFinal RF parameters:"
)

for key, value in final_rf_params.items():

    print(
        f"  {key}: {value}"
    )

print("\nValidation performance:")

print(
    f"  MAE              : "
    f"{final_mae:.6f}"
)

print(
    f"  RMSE             : "
    f"{final_rmse:.6f}"
)

print(
    f"  R²               : "
    f"{final_r2:.6f}"
)

print(
    f"  Accuracy ±10 AQI : "
    f"{final_accuracy_10:.4f}%"
)

print(
    f"  Accuracy ±20 AQI : "
    f"{final_accuracy_20:.4f}%"
)

print("\nBase RF → Tuned RF:")

print(
    f"  Base RMSE  : {base_rmse:.6f}"
)

print(
    f"  Tuned RMSE : {final_rmse:.6f}"
)

print(
    f"  RMSE change: "
    f"{final_rmse - base_rmse:.6f}"
)

print("\nFinal holdout:")
print(
    "FROZEN / UNTOUCHED"
)

print("\nSaved tuning results:")
print(
    RESULTS_PATH
)

print("\nSaved best parameters:")
print(
    PARAMS_PATH
)

print("\nSaved tuned RF model:")
print(
    MODEL_PATH
)

print("=" * 75)
print("NEXT: PHASE 15.5 — FAST TARGETED EXTRA TREES TUNING")
print("=" * 75)

In [ ]:
# ============================================================
# PHASE 15.5 — NEW RUNTIME DRIVE CONNECTION
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Delhi-AQI-Prediction"

print("=" * 70)
print("GOOGLE DRIVE CONNECTED")
print("Ready for Phase 15.5")
print("=" * 70)

In [ ]:
# ============================================================
# VERIFY PHASE 15.4 ARTIFACTS
# ============================================================

import os

required_files = [
    "outputs/reports/phase15_1_time_split_assignments.parquet",
    "outputs/reports/phase15_2_base_model_results.csv",
    "outputs/reports/phase15_3_base_model_leaderboard.csv",
    "outputs/reports/phase15_4_fast_rf_tuning_results.csv",
    "outputs/reports/phase15_4_best_rf_params.json",
    "models/phase15_4_tuned_random_forest.joblib"
]

print("=" * 70)
print("PHASE 15.4 ARTIFACT CHECK")
print("=" * 70)

all_ok = True

for relative_path in required_files:

    full_path = os.path.join(
        PROJECT_ROOT,
        relative_path
    )

    if os.path.exists(full_path):
        print("✓", relative_path)
    else:
        print("✗", relative_path)
        all_ok = False

print("=" * 70)

if not all_ok:
    raise FileNotFoundError(
        "One or more required Phase 15 artifacts are missing."
    )

print("ALL REQUIRED ARTIFACTS FOUND")
print("Proceed to Phase 15.5")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 15.5 — FAST TARGETED EXTRA TREES TUNING
# ============================================================
#
# Strategy:
#   6 targeted Extra Trees configurations
#   1 frozen chronological VALIDATION set
#
# FINAL_HOLDOUT remains completely untouched.
#
# Search stage:
#   100 trees per candidate
#
# Winner:
#   Refit selected structure with 300 trees
#
# Primary metric:
#   Validation RMSE
#
# Supporting:
#   MAE
#   R²
#   Accuracy ±10 AQI
#   Accuracy ±20 AQI
#
# Resume-safe:
#   Results saved after EVERY completed candidate.
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

BASE_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_2_base_model_results.csv"
)

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_5_fast_et_tuning_results.csv"
)

PARAMS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_5_best_et_params.json"
)

VALIDATION_RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_5_tuned_et_validation_result.json"
)

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "phase15_5_tuned_extra_trees.joblib"
)

os.makedirs(
    os.path.dirname(RESULTS_PATH),
    exist_ok=True
)

os.makedirs(
    os.path.dirname(MODEL_PATH),
    exist_ok=True
)


# ------------------------------------------------------------
# 2. VERIFY
# ------------------------------------------------------------

for path in [
    DATA_PATH,
    SPLIT_PATH,
    BASE_RESULTS_PATH
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 4. LOAD FROZEN PHASE 15.1 SPLIT
# ------------------------------------------------------------

split_df = pd.read_parquet(
    SPLIT_PATH
)

split_df["Datetime"] = pd.to_datetime(
    split_df["Datetime"],
    errors="coerce"
)

if "Split" in df.columns:
    df = df.drop(
        columns=["Split"]
    )

df = df.merge(
    split_df,
    on=[
        "StationId",
        "Datetime"
    ],
    how="left",
    validate="one_to_one"
)

if df["Split"].isna().any():
    raise ValueError(
        "Some rows do not have Phase 15.1 split assignments."
    )


# ------------------------------------------------------------
# 5. TARGET
# ------------------------------------------------------------

TARGET = "AQI_t_plus_1"


# ------------------------------------------------------------
# 6. FINAL 73-FEATURE CONTRACT
# ------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in current_features
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

MODEL_FEATURES = list(
    dict.fromkeys(
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + event_features
        + station_features
    )
)

if len(MODEL_FEATURES) != 73:
    raise ValueError(
        f"Expected 73 features, found {len(MODEL_FEATURES)}."
    )


# ------------------------------------------------------------
# 7. TRAIN / VALIDATION
# ------------------------------------------------------------

train_df = (
    df[
        df["Split"] == "TRAIN"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

valid_df = (
    df[
        df["Split"] == "VALIDATION"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

train_df = train_df[
    pd.to_numeric(
        train_df[TARGET],
        errors="coerce"
    ).notna()
].copy()

valid_df = valid_df[
    pd.to_numeric(
        valid_df[TARGET],
        errors="coerce"
    ).notna()
].copy()


X_train = train_df[
    MODEL_FEATURES
].copy()

y_train = pd.to_numeric(
    train_df[TARGET],
    errors="coerce"
)

X_valid = valid_df[
    MODEL_FEATURES
].copy()

y_valid = pd.to_numeric(
    valid_df[TARGET],
    errors="coerce"
)


# ------------------------------------------------------------
# 8. FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "StationId"
]

numeric_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in categorical_features
]


# ------------------------------------------------------------
# 9. PIPELINE BUILDER
# ------------------------------------------------------------

def build_et_pipeline(params):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])


# ------------------------------------------------------------
# 10. FAST ET SEARCH SPACE
# ------------------------------------------------------------
#
# Targeted around the successful base Extra Trees model.
#
# Base ET from Phase 15.2:
#   n_estimators = 100
#   max_depth = None
#   min_samples_split = 2
#   min_samples_leaf = 1
#   max_features = 1.0
#
# Search uses 100 trees.
# Winner is refit with 300 trees.
# ------------------------------------------------------------

et_candidates = [

    {
        "Candidate ID": 1,
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },

    {
        "Candidate ID": 2,
        "n_estimators": 100,
        "max_depth": 22,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },

    {
        "Candidate ID": 3,
        "n_estimators": 100,
        "max_depth": 26,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },

    {
        "Candidate ID": 4,
        "n_estimators": 100,
        "max_depth": 26,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.7
    },

    {
        "Candidate ID": 5,
        "n_estimators": 100,
        "max_depth": 22,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.7
    },

    {
        "Candidate ID": 6,
        "n_estimators": 100,
        "max_depth": 26,
        "min_samples_split": 2,
        "min_samples_leaf": 5,
        "max_features": 0.7
    }
]


# ------------------------------------------------------------
# 11. LOAD PREVIOUS RESULTS
# ------------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    tuning_results = pd.read_csv(
        RESULTS_PATH
    )

else:

    tuning_results = pd.DataFrame()


if (
    len(tuning_results) > 0
    and "Candidate ID" in tuning_results.columns
):

    completed_ids = set(
        tuning_results[
            "Candidate ID"
        ]
        .astype(int)
        .tolist()
    )

else:

    completed_ids = set()


# ------------------------------------------------------------
# 12. START SEARCH
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 15.5 — FAST TARGETED EXTRA TREES TUNING")
print("=" * 75)

print(
    "Training rows   :",
    len(X_train)
)

print(
    "Validation rows :",
    len(X_valid)
)

print(
    "Features        :",
    len(MODEL_FEATURES)
)

print(
    "Candidates      :",
    len(et_candidates)
)

print(
    "Trees per search candidate:",
    100
)

print(
    "Final holdout   : FROZEN / UNTOUCHED"
)

print("=" * 75)


# ------------------------------------------------------------
# 13. EVALUATE CANDIDATES
# ------------------------------------------------------------

for candidate in et_candidates:

    candidate_id = candidate[
        "Candidate ID"
    ]

    if candidate_id in completed_ids:

        print(
            f"\nSKIPPING candidate "
            f"{candidate_id} — already completed"
        )

        continue

    params = {
        key: value
        for key, value in candidate.items()
        if key != "Candidate ID"
    }

    print("\n" + "=" * 65)

    print(
        f"CANDIDATE {candidate_id}/"
        f"{len(et_candidates)}"
    )

    print(
        "Parameters:",
        params
    )

    print("=" * 65)

    pipeline = build_et_pipeline(
        params
    )

    # Fit ONLY on TRAIN
    pipeline.fit(
        X_train,
        y_train
    )

    # Predict ONLY VALIDATION
    prediction = pipeline.predict(
        X_valid
    )

    actual = y_valid.to_numpy()

    prediction = np.asarray(
        prediction
    )

    mask = (
        np.isfinite(actual)
        &
        np.isfinite(prediction)
    )

    if mask.sum() == 0:
        raise ValueError(
            f"No valid predictions for candidate {candidate_id}."
        )

    actual_valid = actual[
        mask
    ]

    prediction_valid = prediction[
        mask
    ]

    mae = mean_absolute_error(
        actual_valid,
        prediction_valid
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_valid,
            prediction_valid
        )
    )

    r2 = r2_score(
        actual_valid,
        prediction_valid
    )

    accuracy_10 = (
        np.mean(
            np.abs(
                actual_valid
                - prediction_valid
            ) <= 10
        ) * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(
                actual_valid
                - prediction_valid
            ) <= 20
        ) * 100
    )

    result = {
        "Candidate ID":
            candidate_id,

        "n_estimators":
            params["n_estimators"],

        "max_depth":
            params["max_depth"],

        "min_samples_split":
            params["min_samples_split"],

        "min_samples_leaf":
            params["min_samples_leaf"],

        "max_features":
            params["max_features"],

        "MAE":
            float(mae),

        "RMSE":
            float(rmse),

        "R2":
            float(r2),

        "Accuracy ±10 AQI":
            float(accuracy_10),

        "Accuracy ±20 AQI":
            float(accuracy_20),

        "Validation Records":
            int(len(actual_valid))
    }

    tuning_results = pd.concat(
        [
            tuning_results,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    # Save immediately
    tuning_results.to_csv(
        RESULTS_PATH,
        index=False
    )

    completed_ids.add(
        candidate_id
    )

    print(
        f"\n✓ Candidate {candidate_id} completed"
    )

    print(
        f"  MAE              : {mae:.6f}"
    )

    print(
        f"  RMSE             : {rmse:.6f}"
    )

    print(
        f"  R²               : {r2:.6f}"
    )

    print(
        f"  Accuracy ±10 AQI : {accuracy_10:.4f}%"
    )

    print(
        f"  Accuracy ±20 AQI : {accuracy_20:.4f}%"
    )


# ------------------------------------------------------------
# 14. RANK RESULTS
# ------------------------------------------------------------

tuning_results = (
    tuning_results
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

if "Rank" in tuning_results.columns:
    tuning_results = tuning_results.drop(
        columns=["Rank"]
    )

tuning_results.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(tuning_results) + 1
    )
)

tuning_results.to_csv(
    RESULTS_PATH,
    index=False
)


# ------------------------------------------------------------
# 15. DISPLAY LEADERBOARD
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.5 — EXTRA TREES SEARCH LEADERBOARD")
print("=" * 75)

display(
    tuning_results[
        [
            "Rank",
            "Candidate ID",
            "n_estimators",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI"
        ]
    ]
)


# ------------------------------------------------------------
# 16. SELECT BEST STRUCTURE
# ------------------------------------------------------------

best = (
    tuning_results
    .sort_values(
        "RMSE",
        ascending=True
    )
    .iloc[0]
)

best_params = {
    "n_estimators":
        int(best["n_estimators"]),

    "max_depth":
        (
            None
            if pd.isna(best["max_depth"])
            else int(best["max_depth"])
        ),

    "min_samples_split":
        int(best["min_samples_split"]),

    "min_samples_leaf":
        int(best["min_samples_leaf"]),

    "max_features":
        float(best["max_features"])
}


# ------------------------------------------------------------
# 17. REFIT WINNER WITH 300 TREES
# ------------------------------------------------------------

final_et_params = best_params.copy()

final_et_params[
    "n_estimators"
] = 300

print("\n" + "=" * 75)
print("REFITTING BEST ET CONFIGURATION WITH 300 TREES")
print("=" * 75)

print(
    "Selected search candidate:",
    int(best["Candidate ID"])
)

print(
    "Final ET parameters:"
)

for key, value in final_et_params.items():

    print(
        f"  {key}: {value}"
    )


final_pipeline = build_et_pipeline(
    final_et_params
)

final_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 18. VALIDATION EVALUATION
# ------------------------------------------------------------

final_prediction = (
    final_pipeline.predict(
        X_valid
    )
)

actual = y_valid.to_numpy()

final_prediction = np.asarray(
    final_prediction
)

mask = (
    np.isfinite(actual)
    &
    np.isfinite(final_prediction)
)

actual = actual[
    mask
]

final_prediction = final_prediction[
    mask
]

final_mae = mean_absolute_error(
    actual,
    final_prediction
)

final_rmse = np.sqrt(
    mean_squared_error(
        actual,
        final_prediction
    )
)

final_r2 = r2_score(
    actual,
    final_prediction
)

final_accuracy_10 = (
    np.mean(
        np.abs(
            actual
            - final_prediction
        ) <= 10
    ) * 100
)

final_accuracy_20 = (
    np.mean(
        np.abs(
            actual
            - final_prediction
        ) <= 20
    ) * 100
)


# ------------------------------------------------------------
# 19. LOAD BASE ET RESULT
# ------------------------------------------------------------

base_results = pd.read_csv(
    BASE_RESULTS_PATH
)

base_et = base_results[
    base_results["Model"]
    == "Extra Trees"
]

if len(base_et) != 1:
    raise ValueError(
        "Base Extra Trees result not found uniquely."
    )

base_et = base_et.iloc[0]

base_rmse = float(
    base_et["RMSE"]
)

base_mae = float(
    base_et["MAE"]
)

base_accuracy_10 = float(
    base_et["Accuracy ±10 AQI"]
)


# ------------------------------------------------------------
# 20. SAVE BEST PARAMETERS
# ------------------------------------------------------------

params_record = {

    "phase": "15",

    "step": "15.5",

    "model":
        "Tuned Extra Trees",

    "search_strategy":
        "Fast targeted validation search",

    "search_candidates":
        len(et_candidates),

    "search_trees":
        100,

    "final_trees":
        300,

    "best_candidate_id":
        int(best["Candidate ID"]),

    "best_search_rmse":
        float(best["RMSE"]),

    "best_params":
        final_et_params,

    "final_holdout_status":
        "FROZEN — NOT USED"
}

with open(
    PARAMS_PATH,
    "w"
) as f:

    json.dump(
        params_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 21. SAVE VALIDATION RESULT
# ------------------------------------------------------------

validation_record = {

    "phase": "15",

    "step": "15.5",

    "model":
        "Tuned Extra Trees",

    "feature_count":
        len(MODEL_FEATURES),

    "validation_records":
        int(len(actual)),

    "validation_mae":
        float(final_mae),

    "validation_rmse":
        float(final_rmse),

    "validation_r2":
        float(final_r2),

    "validation_accuracy_plus_minus_10":
        float(final_accuracy_10),

    "validation_accuracy_plus_minus_20":
        float(final_accuracy_20),

    "base_extra_trees": {

        "mae":
            base_mae,

        "rmse":
            base_rmse,

        "accuracy_plus_minus_10":
            base_accuracy_10
    },

    "tuned_minus_base": {

        "mae":
            float(
                final_mae
                - base_mae
            ),

        "rmse":
            float(
                final_rmse
                - base_rmse
            ),

        "accuracy_plus_minus_10":
            float(
                final_accuracy_10
                - base_accuracy_10
            )
    },

    "final_holdout_status":
        "FROZEN — NOT USED"
}

with open(
    VALIDATION_RESULT_PATH,
    "w"
) as f:

    json.dump(
        validation_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 22. SAVE TUNED ET MODEL
# ------------------------------------------------------------

joblib.dump(
    final_pipeline,
    MODEL_PATH
)


# ------------------------------------------------------------
# 23. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.5 — TUNED EXTRA TREES COMPLETED")
print("=" * 75)

print(
    "Selected candidate:",
    int(best["Candidate ID"])
)

print(
    "\nFinal ET parameters:"
)

for key, value in final_et_params.items():

    print(
        f"  {key}: {value}"
    )

print("\nValidation performance:")

print(
    f"  MAE              : "
    f"{final_mae:.6f}"
)

print(
    f"  RMSE             : "
    f"{final_rmse:.6f}"
)

print(
    f"  R²               : "
    f"{final_r2:.6f}"
)

print(
    f"  Accuracy ±10 AQI : "
    f"{final_accuracy_10:.4f}%"
)

print(
    f"  Accuracy ±20 AQI : "
    f"{final_accuracy_20:.4f}%"
)

print("\nBase ET → Tuned ET:")

print(
    f"  Base RMSE  : "
    f"{base_rmse:.6f}"
)

print(
    f"  Tuned RMSE : "
    f"{final_rmse:.6f}"
)

print(
    f"  RMSE change: "
    f"{final_rmse - base_rmse:.6f}"
)

print(
    "\nFinal holdout:",
    "FROZEN / UNTOUCHED"
)

print("\nSaved tuning results:")
print(
    RESULTS_PATH
)

print("\nSaved best parameters:")
print(
    PARAMS_PATH
)

print("\nSaved tuned ET model:")
print(
    MODEL_PATH
)

print("=" * 75)
print("PHASE 15.5 COMPLETED")
print("NEXT: PHASE 15.6 — FAST TARGETED XGBOOST TUNING")
print("=" * 75)

In [ ]:
# ============================================================
# PHASE 15.6 — FAST TARGETED XGBOOST TUNING
# ============================================================
#
# Strategy:
#   6 targeted XGBoost configurations
#   TRAIN -> fit
#   VALIDATION -> select
#   FINAL_HOLDOUT -> untouched
#
# Search:
#   100 boosting rounds per candidate
#
# Winner:
#   Refit with 300 boosting rounds
#
# Primary metric:
#   RMSE
#
# Supporting:
#   MAE
#   R²
#   Accuracy ±10 AQI
#   Accuracy ±20 AQI
#
# Resume-safe:
#   Save every completed candidate immediately.
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError(
        "XGBoost is not installed. Run:\n"
        "!pip install -q xgboost"
    )


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

BASE_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_2_base_model_results.csv"
)

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_6_fast_xgb_tuning_results.csv"
)

PARAMS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_6_best_xgb_params.json"
)

VALIDATION_RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_6_tuned_xgb_validation_result.json"
)

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "phase15_6_tuned_xgboost.joblib"
)

os.makedirs(
    os.path.dirname(RESULTS_PATH),
    exist_ok=True
)

os.makedirs(
    os.path.dirname(MODEL_PATH),
    exist_ok=True
)


# ------------------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------------------

for path in [
    DATA_PATH,
    SPLIT_PATH,
    BASE_RESULTS_PATH
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

df = pd.read_parquet(
    DATA_PATH
)

df["Datetime"] = pd.to_datetime(
    df["Datetime"],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 4. LOAD FROZEN PHASE 15.1 SPLIT
# ------------------------------------------------------------

split_df = pd.read_parquet(
    SPLIT_PATH
)

split_df["Datetime"] = pd.to_datetime(
    split_df["Datetime"],
    errors="coerce"
)

if "Split" in df.columns:
    df = df.drop(
        columns=["Split"]
    )

df = df.merge(
    split_df,
    on=[
        "StationId",
        "Datetime"
    ],
    how="left",
    validate="one_to_one"
)

if df["Split"].isna().any():
    raise ValueError(
        "Some rows do not have Phase 15.1 split assignments."
    )


# ------------------------------------------------------------
# 5. TARGET
# ------------------------------------------------------------

TARGET = "AQI_t_plus_1"


# ------------------------------------------------------------
# 6. FINAL 73-FEATURE CONTRACT
# ------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in current_features
]

rolling_features = [
    "AQI_rolling_mean_3h",
    "AQI_rolling_std_3h",
    "AQI_rolling_mean_12h",
    "AQI_rolling_std_12h",

    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_std_3h",
    "PM2.5_rolling_mean_12h",
    "PM2.5_rolling_std_12h",

    "PM10_rolling_mean_3h",
    "PM10_rolling_std_3h",
    "PM10_rolling_mean_12h",
    "PM10_rolling_std_12h",

    "NO2_rolling_mean_3h",
    "NO2_rolling_std_3h",
    "NO2_rolling_mean_12h",
    "NO2_rolling_std_12h"
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

station_features = [
    "StationId"
]

MODEL_FEATURES = list(
    dict.fromkeys(
        current_features
        + lag_features
        + rolling_features
        + calendar_features
        + event_features
        + station_features
    )
)

if len(MODEL_FEATURES) != 73:
    raise ValueError(
        f"Expected 73 features, found {len(MODEL_FEATURES)}."
    )


# ------------------------------------------------------------
# 7. TRAIN / VALIDATION
# ------------------------------------------------------------

train_df = (
    df[
        df["Split"] == "TRAIN"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

valid_df = (
    df[
        df["Split"] == "VALIDATION"
    ]
    .sort_values(
        ["Datetime", "StationId"]
    )
    .reset_index(drop=True)
)

train_df = train_df[
    pd.to_numeric(
        train_df[TARGET],
        errors="coerce"
    ).notna()
].copy()

valid_df = valid_df[
    pd.to_numeric(
        valid_df[TARGET],
        errors="coerce"
    ).notna()
].copy()


X_train = train_df[
    MODEL_FEATURES
].copy()

y_train = pd.to_numeric(
    train_df[TARGET],
    errors="coerce"
)

X_valid = valid_df[
    MODEL_FEATURES
].copy()

y_valid = pd.to_numeric(
    valid_df[TARGET],
    errors="coerce"
)


# ------------------------------------------------------------
# 8. FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "StationId"
]

numeric_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in categorical_features
]


# ------------------------------------------------------------
# 9. PIPELINE BUILDER
# ------------------------------------------------------------

def build_xgb_pipeline(params):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )

    model = XGBRegressor(
        **params,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])


# ------------------------------------------------------------
# 10. FAST TARGETED XGBOOST SEARCH
# ------------------------------------------------------------
#
# 100 boosting rounds during search.
#
# The search focuses on:
#   learning_rate
#   max_depth
#   min_child_weight
#   subsample
#   colsample_bytree
# ------------------------------------------------------------

xgb_candidates = [

    {
        "Candidate ID": 1,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 4,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "Candidate ID": 2,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "Candidate ID": 3,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 8,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "Candidate ID": 4,
        "n_estimators": 100,
        "learning_rate": 0.10,
        "max_depth": 6,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "Candidate ID": 5,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "Candidate ID": 6,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 1,
        "subsample": 1.0,
        "colsample_bytree": 1.0
    }
]


# ------------------------------------------------------------
# 11. LOAD PREVIOUS RESULTS IF THEY EXIST
# ------------------------------------------------------------

if os.path.exists(RESULTS_PATH):

    tuning_results = pd.read_csv(
        RESULTS_PATH
    )

else:

    tuning_results = pd.DataFrame()


if (
    len(tuning_results) > 0
    and "Candidate ID" in tuning_results.columns
):

    completed_ids = set(
        tuning_results[
            "Candidate ID"
        ]
        .astype(int)
        .tolist()
    )

else:

    completed_ids = set()


# ------------------------------------------------------------
# 12. START SEARCH
# ------------------------------------------------------------

print("=" * 75)
print("PHASE 15.6 — FAST TARGETED XGBOOST TUNING")
print("=" * 75)

print(
    "Training rows   :",
    len(X_train)
)

print(
    "Validation rows :",
    len(X_valid)
)

print(
    "Features        :",
    len(MODEL_FEATURES)
)

print(
    "Candidates      :",
    len(xgb_candidates)
)

print(
    "Trees per search candidate:",
    100
)

print(
    "Final holdout   : FROZEN / UNTOUCHED"
)

print("=" * 75)


# ------------------------------------------------------------
# 13. EVALUATE CANDIDATES
# ------------------------------------------------------------

for candidate in xgb_candidates:

    candidate_id = candidate[
        "Candidate ID"
    ]

    if candidate_id in completed_ids:

        print(
            f"\nSKIPPING candidate "
            f"{candidate_id} — already completed"
        )

        continue

    params = {
        key: value
        for key, value in candidate.items()
        if key != "Candidate ID"
    }

    print("\n" + "=" * 65)

    print(
        f"CANDIDATE {candidate_id}/"
        f"{len(xgb_candidates)}"
    )

    print(
        "Parameters:",
        params
    )

    print("=" * 65)

    pipeline = build_xgb_pipeline(
        params
    )

    # Fit ONLY on TRAIN
    pipeline.fit(
        X_train,
        y_train
    )

    # Predict ONLY VALIDATION
    prediction = pipeline.predict(
        X_valid
    )

    actual = y_valid.to_numpy()

    prediction = np.asarray(
        prediction
    )

    mask = (
        np.isfinite(actual)
        &
        np.isfinite(prediction)
    )

    if mask.sum() == 0:
        raise ValueError(
            f"No valid predictions for candidate {candidate_id}."
        )

    actual_valid = actual[
        mask
    ]

    prediction_valid = prediction[
        mask
    ]

    mae = mean_absolute_error(
        actual_valid,
        prediction_valid
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_valid,
            prediction_valid
        )
    )

    r2 = r2_score(
        actual_valid,
        prediction_valid
    )

    accuracy_10 = (
        np.mean(
            np.abs(
                actual_valid
                - prediction_valid
            ) <= 10
        ) * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(
                actual_valid
                - prediction_valid
            ) <= 20
        ) * 100
    )

    result = {
        "Candidate ID":
            candidate_id,

        "n_estimators":
            params["n_estimators"],

        "learning_rate":
            params["learning_rate"],

        "max_depth":
            params["max_depth"],

        "min_child_weight":
            params["min_child_weight"],

        "subsample":
            params["subsample"],

        "colsample_bytree":
            params["colsample_bytree"],

        "MAE":
            float(mae),

        "RMSE":
            float(rmse),

        "R2":
            float(r2),

        "Accuracy ±10 AQI":
            float(accuracy_10),

        "Accuracy ±20 AQI":
            float(accuracy_20),

        "Validation Records":
            int(len(actual_valid))
    }

    tuning_results = pd.concat(
        [
            tuning_results,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    # Save immediately
    tuning_results.to_csv(
        RESULTS_PATH,
        index=False
    )

    completed_ids.add(
        candidate_id
    )

    print(
        f"\n✓ Candidate {candidate_id} completed"
    )

    print(
        f"  MAE              : {mae:.6f}"
    )

    print(
        f"  RMSE             : {rmse:.6f}"
    )

    print(
        f"  R²               : {r2:.6f}"
    )

    print(
        f"  Accuracy ±10 AQI : {accuracy_10:.4f}%"
    )

    print(
        f"  Accuracy ±20 AQI : {accuracy_20:.4f}%"
    )


# ------------------------------------------------------------
# 14. RANK RESULTS
# ------------------------------------------------------------

tuning_results = (
    tuning_results
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

if "Rank" in tuning_results.columns:
    tuning_results = tuning_results.drop(
        columns=["Rank"]
    )

tuning_results.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(tuning_results) + 1
    )
)

tuning_results.to_csv(
    RESULTS_PATH,
    index=False
)


# ------------------------------------------------------------
# 15. DISPLAY SEARCH LEADERBOARD
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.6 — XGBOOST SEARCH LEADERBOARD")
print("=" * 75)

display(
    tuning_results[
        [
            "Rank",
            "Candidate ID",
            "n_estimators",
            "learning_rate",
            "max_depth",
            "min_child_weight",
            "subsample",
            "colsample_bytree",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI"
        ]
    ]
)


# ------------------------------------------------------------
# 16. SELECT BEST STRUCTURE
# ------------------------------------------------------------

best = (
    tuning_results
    .sort_values(
        "RMSE",
        ascending=True
    )
    .iloc[0]
)

best_params = {
    "n_estimators":
        int(best["n_estimators"]),

    "learning_rate":
        float(best["learning_rate"]),

    "max_depth":
        int(best["max_depth"]),

    "min_child_weight":
        int(best["min_child_weight"]),

    "subsample":
        float(best["subsample"]),

    "colsample_bytree":
        float(best["colsample_bytree"])
}


# ------------------------------------------------------------
# 17. REFIT BEST STRUCTURE WITH 300 TREES
# ------------------------------------------------------------

final_xgb_params = best_params.copy()

final_xgb_params[
    "n_estimators"
] = 300

print("\n" + "=" * 75)
print("REFITTING BEST XGBOOST CONFIGURATION WITH 300 TREES")
print("=" * 75)

print(
    "Selected search candidate:",
    int(best["Candidate ID"])
)

print(
    "Final XGBoost parameters:"
)

for key, value in final_xgb_params.items():

    print(
        f"  {key}: {value}"
    )


final_pipeline = build_xgb_pipeline(
    final_xgb_params
)

final_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 18. FINAL VALIDATION EVALUATION
# ------------------------------------------------------------

final_prediction = (
    final_pipeline.predict(
        X_valid
    )
)

actual = y_valid.to_numpy()

final_prediction = np.asarray(
    final_prediction
)

mask = (
    np.isfinite(actual)
    &
    np.isfinite(final_prediction)
)

actual = actual[
    mask
]

final_prediction = final_prediction[
    mask
]

final_mae = mean_absolute_error(
    actual,
    final_prediction
)

final_rmse = np.sqrt(
    mean_squared_error(
        actual,
        final_prediction
    )
)

final_r2 = r2_score(
    actual,
    final_prediction
)

final_accuracy_10 = (
    np.mean(
        np.abs(
            actual
            - final_prediction
        ) <= 10
    ) * 100
)

final_accuracy_20 = (
    np.mean(
        np.abs(
            actual
            - final_prediction
        ) <= 20
    ) * 100
)


# ------------------------------------------------------------
# 19. BASE XGBOOST COMPARISON
# ------------------------------------------------------------

base_results = pd.read_csv(
    BASE_RESULTS_PATH
)

base_xgb = base_results[
    base_results["Model"]
    == "XGBoost"
]

if len(base_xgb) != 1:
    raise ValueError(
        "Base XGBoost result not found uniquely."
    )

base_xgb = base_xgb.iloc[0]

base_rmse = float(
    base_xgb["RMSE"]
)

base_mae = float(
    base_xgb["MAE"]
)

base_accuracy_10 = float(
    base_xgb["Accuracy ±10 AQI"]
)


# ------------------------------------------------------------
# 20. SAVE BEST PARAMETERS
# ------------------------------------------------------------

params_record = {

    "phase": "15",

    "step": "15.6",

    "model":
        "Tuned XGBoost",

    "search_strategy":
        "Fast targeted validation search",

    "search_candidates":
        len(xgb_candidates),

    "search_trees":
        100,

    "final_trees":
        300,

    "best_candidate_id":
        int(best["Candidate ID"]),

    "best_search_rmse":
        float(best["RMSE"]),

    "best_params":
        final_xgb_params,

    "final_holdout_status":
        "FROZEN — NOT USED"
}

with open(
    PARAMS_PATH,
    "w"
) as f:

    json.dump(
        params_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 21. SAVE VALIDATION RESULT
# ------------------------------------------------------------

validation_record = {

    "phase": "15",

    "step": "15.6",

    "model":
        "Tuned XGBoost",

    "feature_count":
        len(MODEL_FEATURES),

    "validation_records":
        int(len(actual)),

    "validation_mae":
        float(final_mae),

    "validation_rmse":
        float(final_rmse),

    "validation_r2":
        float(final_r2),

    "validation_accuracy_plus_minus_10":
        float(final_accuracy_10),

    "validation_accuracy_plus_minus_20":
        float(final_accuracy_20),

    "base_xgboost": {

        "mae":
            base_mae,

        "rmse":
            base_rmse,

        "accuracy_plus_minus_10":
            base_accuracy_10
    },

    "tuned_minus_base": {

        "mae":
            float(
                final_mae
                - base_mae
            ),

        "rmse":
            float(
                final_rmse
                - base_rmse
            ),

        "accuracy_plus_minus_10":
            float(
                final_accuracy_10
                - base_accuracy_10
            )
    },

    "final_holdout_status":
        "FROZEN — NOT USED"
}

with open(
    VALIDATION_RESULT_PATH,
    "w"
) as f:

    json.dump(
        validation_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 22. SAVE TUNED XGBOOST MODEL
# ------------------------------------------------------------

joblib.dump(
    final_pipeline,
    MODEL_PATH
)


# ------------------------------------------------------------
# 23. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 15.6 — TUNED XGBOOST COMPLETED")
print("=" * 75)

print(
    "Selected candidate:",
    int(best["Candidate ID"])
)

print(
    "\nFinal XGBoost parameters:"
)

for key, value in final_xgb_params.items():

    print(
        f"  {key}: {value}"
    )

print("\nValidation performance:")

print(
    f"  MAE              : "
    f"{final_mae:.6f}"
)

print(
    f"  RMSE             : "
    f"{final_rmse:.6f}"
)

print(
    f"  R²               : "
    f"{final_r2:.6f}"
)

print(
    f"  Accuracy ±10 AQI : "
    f"{final_accuracy_10:.4f}%"
)

print(
    f"  Accuracy ±20 AQI : "
    f"{final_accuracy_20:.4f}%"
)

print("\nBase XGBoost → Tuned XGBoost:")

print(
    f"  Base RMSE  : "
    f"{base_rmse:.6f}"
)

print(
    f"  Tuned RMSE : "
    f"{final_rmse:.6f}"
)

print(
    f"  RMSE change: "
    f"{final_rmse - base_rmse:.6f}"
)

print(
    "\nFinal holdout:",
    "FROZEN / UNTOUCHED"
)

print("\nSaved tuning results:")
print(
    RESULTS_PATH
)

print("\nSaved best parameters:")
print(
    PARAMS_PATH
)

print("\nSaved tuned XGBoost model:")
print(
    MODEL_PATH
)

print("=" * 75)
print("NEXT: PHASE 15.7 — FINAL MODEL COMPARISON + TOP-TWO SELECTION")
print("=" * 75)

In [ ]:
# ============================================================
# PHASE 15.7 — FINAL MODEL COMPARISON + TOP-TWO SELECTION
# ============================================================
#
# Compare:
#   Persistence Baseline
#   8 Base ML models
#   Tuned Random Forest
#   Tuned Extra Trees
#   Tuned XGBoost
#
# Selection:
#   Primary   -> Lowest Validation RMSE
#   Secondary -> Lowest Validation MAE
#
# FINAL HOLDOUT:
#   Completely untouched
#
# Output:
#   1. Complete candidate leaderboard
#   2. Selected Top-2 models
#   3. Saved Top-2 configuration
# ============================================================

import os
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Delhi-AQI-Prediction"
)

BASE_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_2_base_model_results.csv"
)

RF_RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_4_tuned_rf_validation_result.json"
)

ET_RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_5_tuned_et_validation_result.json"
)

XGB_RESULT_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_6_tuned_xgb_validation_result.json"
)

SAVE_LEADERBOARD_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_7_final_model_comparison.csv"
)

SAVE_TOP2_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "reports",
    "phase15_7_top_two_models.json"
)


# ------------------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------------------

required_files = [
    BASE_RESULTS_PATH,
    RF_RESULT_PATH,
    ET_RESULT_PATH,
    XGB_RESULT_PATH
]

for path in required_files:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Required artifact not found:\n{path}"
        )


# ------------------------------------------------------------
# 3. LOAD BASE MODEL RESULTS
# ------------------------------------------------------------

base_results = pd.read_csv(
    BASE_RESULTS_PATH
)

# Remove previously generated ranking/index columns
for col in [
    "Rank",
    "Unnamed: 0"
]:

    if col in base_results.columns:

        base_results = base_results.drop(
            columns=[col]
        )


# ------------------------------------------------------------
# 4. LOAD TUNED MODEL RESULTS
# ------------------------------------------------------------

with open(
    RF_RESULT_PATH,
    "r"
) as f:

    rf_data = json.load(f)

with open(
    ET_RESULT_PATH,
    "r"
) as f:

    et_data = json.load(f)

with open(
    XGB_RESULT_PATH,
    "r"
) as f:

    xgb_data = json.load(f)


# ------------------------------------------------------------
# 5. EXTRACT TUNED MODEL METRICS
# ------------------------------------------------------------

tuned_results = pd.DataFrame([

    {
        "Model":
            "Tuned Random Forest",

        "Model Type":
            "Tuned",

        "Feature Count":
            int(
                rf_data["feature_count"]
            ),

        "MAE":
            float(
                rf_data["validation_mae"]
            ),

        "RMSE":
            float(
                rf_data["validation_rmse"]
            ),

        "R2":
            float(
                rf_data["validation_r2"]
            ),

        "Accuracy ±10 AQI":
            float(
                rf_data[
                    "validation_accuracy_plus_minus_10"
                ]
            ),

        "Accuracy ±20 AQI":
            float(
                rf_data[
                    "validation_accuracy_plus_minus_20"
                ]
            ),

        "Validation Records":
            int(
                rf_data[
                    "validation_records"
                ]
            )
    },

    {
        "Model":
            "Tuned Extra Trees",

        "Model Type":
            "Tuned",

        "Feature Count":
            int(
                et_data["feature_count"]
            ),

        "MAE":
            float(
                et_data["validation_mae"]
            ),

        "RMSE":
            float(
                et_data["validation_rmse"]
            ),

        "R2":
            float(
                et_data["validation_r2"]
            ),

        "Accuracy ±10 AQI":
            float(
                et_data[
                    "validation_accuracy_plus_minus_10"
                ]
            ),

        "Accuracy ±20 AQI":
            float(
                et_data[
                    "validation_accuracy_plus_minus_20"
                ]
            ),

        "Validation Records":
            int(
                et_data[
                    "validation_records"
                ]
            )
    },

    {
        "Model":
            "Tuned XGBoost",

        "Model Type":
            "Tuned",

        "Feature Count":
            int(
                xgb_data["feature_count"]
            ),

        "MAE":
            float(
                xgb_data["validation_mae"]
            ),

        "RMSE":
            float(
                xgb_data["validation_rmse"]
            ),

        "R2":
            float(
                xgb_data["validation_r2"]
            ),

        "Accuracy ±10 AQI":
            float(
                xgb_data[
                    "validation_accuracy_plus_minus_10"
                ]
            ),

        "Accuracy ±20 AQI":
            float(
                xgb_data[
                    "validation_accuracy_plus_minus_20"
                ]
            ),

        "Validation Records":
            int(
                xgb_data[
                    "validation_records"
                ]
            )
    }
])


# ------------------------------------------------------------
# 6. ADD MODEL TYPE TO BASE RESULTS
# ------------------------------------------------------------

base_results["Model Type"] = "Base"


# Persistence is a baseline, not a candidate ML model.
base_results.loc[
    base_results["Model"]
    == "Persistence Baseline",
    "Model Type"
] = "Baseline"


# ------------------------------------------------------------
# 7. STANDARDIZE COLUMN ORDER
# ------------------------------------------------------------

standard_columns = [
    "Model",
    "Model Type",
    "Feature Count",
    "MAE",
    "RMSE",
    "R2",
    "Accuracy ±10 AQI",
    "Accuracy ±20 AQI",
    "Validation Records"
]

base_results = base_results[
    [
        col
        for col in standard_columns
        if col in base_results.columns
    ]
]

tuned_results = tuned_results[
    standard_columns
]


# ------------------------------------------------------------
# 8. COMBINE ALL CANDIDATES
# ------------------------------------------------------------

all_results = pd.concat(
    [
        base_results,
        tuned_results
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 9. REMOVE DUPLICATES
# ------------------------------------------------------------

all_results = (
    all_results
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. NUMERIC SAFETY CHECK
# ------------------------------------------------------------

metric_columns = [
    "MAE",
    "RMSE",
    "R2",
    "Accuracy ±10 AQI",
    "Accuracy ±20 AQI"
]

for col in metric_columns:

    all_results[col] = pd.to_numeric(
        all_results[col],
        errors="coerce"
    )


if all_results[
    "RMSE"
].isna().any():

    bad_models = all_results.loc[
        all_results["RMSE"].isna(),
        "Model"
    ].tolist()

    raise ValueError(
        "Missing RMSE for:\n"
        + "\n".join(bad_models)
    )


# ------------------------------------------------------------
# 11. SORT ALL MODELS
# ------------------------------------------------------------
#
# Primary:
#   RMSE ascending
#
# Secondary:
#   MAE ascending
# ------------------------------------------------------------

all_results = (
    all_results
    .sort_values(
        by=[
            "RMSE",
            "MAE"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. FRESH RANK
# ------------------------------------------------------------

if "Rank" in all_results.columns:

    all_results = all_results.drop(
        columns=["Rank"]
    )

all_results.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(all_results) + 1
    )
)


# ------------------------------------------------------------
# 13. DISPLAY COMPLETE COMPETITION
# ------------------------------------------------------------

print("=" * 80)
print("PHASE 15.7 — COMPLETE MODEL COMPETITION")
print("=" * 80)

display(
    all_results[
        [
            "Rank",
            "Model",
            "Model Type",
            "Feature Count",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI",
            "Validation Records"
        ]
    ]
)


# ------------------------------------------------------------
# 14. SELECT ML MODELS ONLY
# ------------------------------------------------------------

ml_results = all_results[
    all_results["Model Type"]
    != "Baseline"
].copy()

ml_results = (
    ml_results
    .sort_values(
        by=[
            "RMSE",
            "MAE"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. SELECT TOP TWO
# ------------------------------------------------------------

if len(ml_results) < 2:

    raise ValueError(
        "Fewer than two ML models are available "
        "for Top-2 selection."
    )

top_two = (
    ml_results
    .head(2)
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 16. DISPLAY TOP TWO
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 15.7 — SELECTED TOP TWO MODELS")
print("=" * 80)

display(
    top_two[
        [
            "Rank",
            "Model",
            "Model Type",
            "Feature Count",
            "MAE",
            "RMSE",
            "R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI"
        ]
    ]
)


# ------------------------------------------------------------
# 17. TOP-TWO GAP
# ------------------------------------------------------------

top1 = top_two.iloc[0]
top2 = top_two.iloc[1]

rmse_gap = (
    float(top2["RMSE"])
    - float(top1["RMSE"])
)

mae_gap = (
    float(top2["MAE"])
    - float(top1["MAE"])
)

accuracy_gap = (
    float(top1["Accuracy ±10 AQI"])
    - float(top2["Accuracy ±10 AQI"])
)


# ------------------------------------------------------------
# 18. SAVE COMPLETE LEADERBOARD
# ------------------------------------------------------------

all_results.to_csv(
    SAVE_LEADERBOARD_PATH,
    index=False
)


# ------------------------------------------------------------
# 19. SAVE TOP-TWO CONFIGURATION
# ------------------------------------------------------------

top_two_config = {

    "phase":
        "15",

    "step":
        "15.7",

    "selection_rule": {

        "primary_metric":
            "Validation RMSE",

        "secondary_metric":
            "Validation MAE",

        "supporting_metrics": [
            "Validation R2",
            "Accuracy ±10 AQI",
            "Accuracy ±20 AQI"
        ]
    },

    "top_two": [

        {
            "rank": 1,

            "model":
                str(
                    top1["Model"]
                ),

            "model_type":
                str(
                    top1["Model Type"]
                ),

            "feature_count":
                int(
                    top1["Feature Count"]
                ),

            "MAE":
                float(
                    top1["MAE"]
                ),

            "RMSE":
                float(
                    top1["RMSE"]
                ),

            "R2":
                float(
                    top1["R2"]
                ),

            "accuracy_plus_minus_10":
                float(
                    top1["Accuracy ±10 AQI"]
                ),

            "accuracy_plus_minus_20":
                float(
                    top1["Accuracy ±20 AQI"]
                )
        },

        {
            "rank": 2,

            "model":
                str(
                    top2["Model"]
                ),

            "model_type":
                str(
                    top2["Model Type"]
                ),

            "feature_count":
                int(
                    top2["Feature Count"]
                ),

            "MAE":
                float(
                    top2["MAE"]
                ),

            "RMSE":
                float(
                    top2["RMSE"]
                ),

            "R2":
                float(
                    top2["R2"]
                ),

            "accuracy_plus_minus_10":
                float(
                    top2["Accuracy ±10 AQI"]
                ),

            "accuracy_plus_minus_20":
                float(
                    top2["Accuracy ±20 AQI"]
                )
        }
    ],

    "top_two_rmse_gap":
        float(rmse_gap),

    "top_two_mae_gap":
        float(mae_gap),

    "top_two_accuracy_plus_minus_10_gap":
        float(accuracy_gap),

    "feature_count_for_both":
        [
            int(top1["Feature Count"]),
            int(top2["Feature Count"])
        ],

    "final_holdout_status":
        "FROZEN — NOT USED FOR MODEL SELECTION",

    "next_step":
        "Phase 15.8 — Generate Top-2 Validation Predictions",

    "status":
        "TOP TWO SELECTED FOR ENSEMBLE TESTING"
}


with open(
    SAVE_TOP2_PATH,
    "w"
) as f:

    json.dump(
        top_two_config,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 20. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 15.7 SUMMARY")
print("=" * 80)

print(
    "Top 1:",
    top1["Model"]
)

print(
    "Top 1 RMSE:",
    f"{float(top1['RMSE']):.6f}"
)

print(
    "\nTop 2:",
    top2["Model"]
)

print(
    "Top 2 RMSE:",
    f"{float(top2['RMSE']):.6f}"
)

print(
    "\nRMSE gap:",
    f"{rmse_gap:.6f}"
)

print(
    "\nFinal holdout:",
    "FROZEN / UNTOUCHED"
)

print(
    "\nComplete leaderboard saved:"
)

print(
    SAVE_LEADERBOARD_PATH
)

print(
    "\nTop-two configuration saved:"
)

print(
    SAVE_TOP2_PATH
)

print("=" * 80)
print("PHASE 15.7 COMPLETED")
print("NEXT: PHASE 15.8 — TOP-TWO VALIDATION PREDICTIONS")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.8 — TOP-TWO VALIDATION PREDICTIONS
# =============================================================================
# Purpose:
# 1. Restore the Phase 15.7 top-two model selection.
# 2. Restore the frozen Phase 15.1 train/validation/holdout split.
# 3. Load the saved tuned ET and RF artifacts selected by Phase 15.7.
# 4. Generate predictions ONLY on the validation set.
# 5. Recalculate validation metrics for verification.
# 6. Save top-two predictions for Phase 15.9 ensemble weight optimization.
#
# Restart-safe: does not depend on previous notebook variables.
# Final holdout remains untouched.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------------------------------------------------------------
# 1. PATHS
# -----------------------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data/processed/delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_1_time_split_assignments.parquet"
)

TOP_TWO_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_7_top_two_models.json"
)

PREDICTION_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_8_top_two_validation_predictions.parquet"
)

METRICS_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_8_top_two_validation_metrics.csv"
)

AGREEMENT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_8_top_two_prediction_agreement.json"
)

os.makedirs(os.path.dirname(PREDICTION_FILE), exist_ok=True)

# -----------------------------------------------------------------------------
# 2. CHECK REQUIRED FILES
# -----------------------------------------------------------------------------

required_files = {
    "Feature engineered data": FEATURE_FILE,
    "Frozen split assignments": SPLIT_FILE,
    "Top-two configuration": TOP_TWO_FILE,
}

for label, path in required_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} not found:\n{path}")

# -----------------------------------------------------------------------------
# 3. LOAD TOP-TWO CONFIGURATION
# -----------------------------------------------------------------------------

with open(TOP_TWO_FILE, "r") as f:
    top_two_config = json.load(f)

# Support either:
#   {"top_two_models": [...]}
# or direct list-like structures.
if isinstance(top_two_config, dict):
    if "top_two_models" in top_two_config:
        top_two = top_two_config["top_two_models"]
    elif "top_two" in top_two_config:
        top_two = top_two_config["top_two"]
    else:
        # Try common fallback keys
        possible = [
            v for v in top_two_config.values()
            if isinstance(v, list)
        ]
        if not possible:
            raise ValueError(
                "Could not find the top-two model list in phase15_7_top_two_models.json"
            )
        top_two = possible[0]
else:
    top_two = top_two_config

if len(top_two) != 2:
    raise ValueError(f"Expected exactly 2 selected models, found {len(top_two)}")

# Normalize model names
def extract_model_name(item):
    if isinstance(item, str):
        return item

    if isinstance(item, dict):
        for key in ["Model", "model", "model_name", "Model Name", "name"]:
            if key in item:
                return str(item[key])

    raise ValueError(f"Could not determine model name from:\n{item}")

selected_model_names = [extract_model_name(x) for x in top_two]

print("=" * 80)
print("PHASE 15.8 — TOP-TWO VALIDATION PREDICTIONS")
print("=" * 80)
print("Selected models from Phase 15.7:")
for i, name in enumerate(selected_model_names, start=1):
    print(f"{i}. {name}")

# -----------------------------------------------------------------------------
# 4. MAP MODEL NAMES TO SAVED ARTIFACTS
# -----------------------------------------------------------------------------

MODEL_PATH_MAP = {
    "Tuned Extra Trees": os.path.join(
        BASE_DIR,
        "models/phase15_5_tuned_extra_trees.joblib"
    ),
    "Tuned Random Forest": os.path.join(
        BASE_DIR,
        "models/phase15_4_tuned_random_forest.joblib"
    ),
    "Tuned XGBoost": os.path.join(
        BASE_DIR,
        "models/phase15_6_tuned_xgboost.joblib"
    ),
}

model_paths = {}

for model_name in selected_model_names:
    if model_name not in MODEL_PATH_MAP:
        raise ValueError(
            f"No saved artifact mapping found for selected model: {model_name}"
        )

    path = MODEL_PATH_MAP[model_name]

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Model artifact for '{model_name}' not found:\n{path}"
        )

    model_paths[model_name] = path

# -----------------------------------------------------------------------------
# 5. LOAD FEATURE-ENGINEERED DATA
# -----------------------------------------------------------------------------

df = pd.read_parquet(FEATURE_FILE)

TARGET = "AQI_t_plus_1"
DATETIME_COL = "Datetime"
STATION_COL = "StationId"

if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' not found.")

if DATETIME_COL not in df.columns:
    raise KeyError(f"Datetime column '{DATETIME_COL}' not found.")

# Ensure datetime type is consistent
df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce")

if df[DATETIME_COL].isna().any():
    raise ValueError("Datetime contains invalid/null values after parsing.")

# -----------------------------------------------------------------------------
# 6. DEFINE / VERIFY THE 73-FEATURE CONTRACT
# -----------------------------------------------------------------------------

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
]

pollutant_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in pollutant_features
]

rolling_source_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2",
]

rolling_features = []

for feature in rolling_source_features:
    for window in [3, 12]:
        rolling_features.append(f"{feature}_rolling_mean_{window}h")
        rolling_features.append(f"{feature}_rolling_std_{window}h")

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow",
]

station_features = [
    STATION_COL
]

feature_cols = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + station_features
)

print("\nFeature contract:")
print(f"Expected feature count: 73")
print(f"Actual feature count:   {len(feature_cols)}")

if len(feature_cols) != 73:
    raise ValueError(
        f"Feature contract generated {len(feature_cols)} features, expected 73."
    )

missing_features = [c for c in feature_cols if c not in df.columns]

if missing_features:
    raise KeyError(
        "The following expected model features are missing from the dataset:\n"
        + "\n".join(missing_features)
    )

# -----------------------------------------------------------------------------
# 7. LOAD FROZEN PHASE 15.1 SPLIT ASSIGNMENTS
# -----------------------------------------------------------------------------

split_df = pd.read_parquet(SPLIT_FILE)

split_df.columns = [str(c) for c in split_df.columns]

print("\nSplit assignment columns:")
print(split_df.columns.tolist())

# Find split label column robustly
split_candidates = [
    c for c in split_df.columns
    if c.lower() in {
        "split",
        "dataset",
        "partition",
        "set",
        "subset",
        "split_name",
        "data_split",
    }
]

if not split_candidates:
    # Try columns containing "split"
    split_candidates = [
        c for c in split_df.columns
        if "split" in c.lower()
    ]

if not split_candidates:
    raise ValueError(
        "Could not identify the split-label column in the Phase 15.1 assignment file."
    )

split_col = split_candidates[0]

split_df[DATETIME_COL] = pd.to_datetime(
    split_df[DATETIME_COL],
    errors="coerce"
)

if split_df[DATETIME_COL].isna().any():
    raise ValueError("Split assignment file contains invalid Datetime values.")

# -----------------------------------------------------------------------------
# 8. ATTACH SPLIT LABEL TO MODEL-READY DATA
# -----------------------------------------------------------------------------

# Prefer StationId + Datetime if both exist.
if STATION_COL in split_df.columns:

    split_keys = [STATION_COL, DATETIME_COL]

    # Check whether key is unique in split file
    if split_df.duplicated(split_keys).any():
        raise ValueError(
            "Split assignment file contains duplicate StationId-Datetime keys."
        )

    split_lookup = split_df[split_keys + [split_col]].copy()

    model_df = df.merge(
        split_lookup,
        on=split_keys,
        how="left",
        validate="one_to_one"
    )

else:

    # Split was defined at timestamp level.
    if split_df[DATETIME_COL].duplicated().any():
        raise ValueError(
            "Timestamp-only split assignment file contains duplicate timestamps."
        )

    split_lookup = split_df[[DATETIME_COL, split_col]].copy()

    model_df = df.merge(
        split_lookup,
        on=DATETIME_COL,
        how="left",
        validate="many_to_one"
    )

# Verify every model-ready row received a split assignment
unassigned = model_df[split_col].isna().sum()

if unassigned > 0:
    raise ValueError(
        f"{unassigned:,} model-ready rows have no split assignment."
    )

print("\nSplit distribution:")
print(model_df[split_col].value_counts(dropna=False))

# -----------------------------------------------------------------------------
# 9. ISOLATE VALIDATION SET ONLY
# -----------------------------------------------------------------------------

# Handle common capitalization variants
normalized_split = model_df[split_col].astype(str).str.upper()

validation_mask = normalized_split.isin({
    "VALIDATION",
    "VAL",
    "VALID",
})

val_df = model_df.loc[validation_mask].copy()

if val_df.empty:
    raise ValueError("Validation set is empty.")

# Sort deterministically
sort_cols = [DATETIME_COL]
if STATION_COL in val_df.columns:
    sort_cols.append(STATION_COL)

val_df = val_df.sort_values(sort_cols).reset_index(drop=True)

# Remove rows without target
val_df = val_df.loc[val_df[TARGET].notna()].reset_index(drop=True)

print("\nValidation data:")
print(f"Rows:         {len(val_df):,}")
print(f"Stations:     {val_df[STATION_COL].nunique() if STATION_COL in val_df.columns else 'N/A'}")
print(f"Start:        {val_df[DATETIME_COL].min()}")
print(f"End:          {val_df[DATETIME_COL].max()}")

# Expected from Phase 15.1/15.2/15.7
EXPECTED_VALIDATION_ROWS = 69097

if len(val_df) != EXPECTED_VALIDATION_ROWS:
    print(
        f"WARNING: Expected {EXPECTED_VALIDATION_ROWS:,} validation rows "
        f"but found {len(val_df):,}."
    )

# -----------------------------------------------------------------------------
# 10. PREPARE X / y
# -----------------------------------------------------------------------------

X_val = val_df[feature_cols].copy()
y_val = val_df[TARGET].astype(float).to_numpy()

# -----------------------------------------------------------------------------
# 11. ROBUST MODEL ARTIFACT LOADER
# -----------------------------------------------------------------------------

def predict_from_artifact(artifact, X):
    """
    Supports:
    - sklearn Pipeline / estimator with .predict()
    - dict containing 'pipeline'
    - dict containing 'model' + 'preprocessor'/'transformer'
    """

    # Case 1: direct predictor / pipeline
    if hasattr(artifact, "predict"):
        return artifact.predict(X)

    # Case 2: dictionary artifact
    if isinstance(artifact, dict):

        # Full pipeline stored inside dictionary
        for key in ["pipeline", "estimator", "predictor"]:
            if key in artifact and hasattr(artifact[key], "predict"):
                return artifact[key].predict(X)

        # Separate model + preprocessing
        model_obj = None
        preprocessor = None

        for key in ["model", "regressor", "estimator"]:
            if key in artifact and hasattr(artifact[key], "predict"):
                model_obj = artifact[key]
                break

        for key in ["preprocessor", "transformer", "processor"]:
            if key in artifact and hasattr(artifact[key], "transform"):
                preprocessor = artifact[key]
                break

        if model_obj is not None:
            if preprocessor is not None:
                X_transformed = preprocessor.transform(X)
                return model_obj.predict(X_transformed)

            return model_obj.predict(X)

    raise TypeError(
        "Unsupported saved model artifact format. "
        "Expected a sklearn predictor/pipeline or a compatible dictionary."
    )

# -----------------------------------------------------------------------------
# 12. LOAD TOP-TWO MODELS + GENERATE PREDICTIONS
# -----------------------------------------------------------------------------

predictions = pd.DataFrame({
    DATETIME_COL: val_df[DATETIME_COL].values
})

if STATION_COL in val_df.columns:
    predictions[STATION_COL] = val_df[STATION_COL].values

predictions["Actual_AQI_t_plus_1"] = y_val

metrics_rows = []

for model_name in selected_model_names:

    model_path = model_paths[model_name]

    print("\n" + "-" * 80)
    print(f"Loading: {model_name}")
    print(f"Artifact: {model_path}")

    artifact = joblib.load(model_path)

    raw_pred = predict_from_artifact(artifact, X_val)

    pred = np.asarray(raw_pred, dtype=float).reshape(-1)

    if len(pred) != len(y_val):
        raise ValueError(
            f"{model_name}: prediction length {len(pred):,} "
            f"does not match validation rows {len(y_val):,}."
        )

    if not np.isfinite(pred).all():
        raise ValueError(
            f"{model_name}: predictions contain NaN or infinite values."
        )

    safe_name = (
        model_name
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

    predictions[f"{safe_name}_Prediction"] = pred

    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)

    acc10 = np.mean(np.abs(y_val - pred) <= 10) * 100
    acc20 = np.mean(np.abs(y_val - pred) <= 20) * 100

    metrics_rows.append({
        "Model": model_name,
        "Validation Records": len(y_val),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Accuracy ±10 AQI": acc10,
        "Accuracy ±20 AQI": acc20,
    })

    print(f"Validation MAE:        {mae:.6f}")
    print(f"Validation RMSE:       {rmse:.6f}")
    print(f"Validation R2:         {r2:.6f}")
    print(f"Accuracy ±10 AQI:      {acc10:.6f}%")
    print(f"Accuracy ±20 AQI:      {acc20:.6f}%")

# -----------------------------------------------------------------------------
# 13. COMPARE TWO PREDICTION SERIES
# -----------------------------------------------------------------------------

metrics_df = pd.DataFrame(metrics_rows).sort_values(
    by=["RMSE", "MAE"]
).reset_index(drop=True)

# Locate prediction columns
prediction_columns = [
    c for c in predictions.columns
    if c.endswith("_Prediction")
]

if len(prediction_columns) != 2:
    raise ValueError(
        f"Expected exactly 2 prediction columns, found {len(prediction_columns)}."
    )

pred_a = predictions[prediction_columns[0]].to_numpy()
pred_b = predictions[prediction_columns[1]].to_numpy()

prediction_correlation = float(np.corrcoef(pred_a, pred_b)[0, 1])

prediction_difference = pred_a - pred_b

agreement_summary = {
    "Model_A": selected_model_names[0],
    "Model_B": selected_model_names[1],
    "Validation_Records": int(len(predictions)),
    "Prediction_Correlation": prediction_correlation,
    "Mean_Absolute_Prediction_Difference": float(
        np.mean(np.abs(prediction_difference))
    ),
    "RMSE_Between_Model_Predictions": float(
        np.sqrt(np.mean(prediction_difference ** 2))
    ),
    "Maximum_Absolute_Prediction_Difference": float(
        np.max(np.abs(prediction_difference))
    ),
}

# -----------------------------------------------------------------------------
# 14. SAVE OUTPUTS
# -----------------------------------------------------------------------------

predictions.to_parquet(
    PREDICTION_FILE,
    index=False
)

metrics_df.to_csv(
    METRICS_FILE,
    index=False
)

with open(AGREEMENT_FILE, "w") as f:
    json.dump(
        agreement_summary,
        f,
        indent=2
    )

# -----------------------------------------------------------------------------
# 15. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 15.8 — TOP-TWO VALIDATION METRICS")
print("=" * 80)

display(
    metrics_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy ±10 AQI": "{:.6f}",
        "Accuracy ±20 AQI": "{:.6f}",
    })
)

print("\n" + "=" * 80)
print("PHASE 15.8 — PREDICTION AGREEMENT")
print("=" * 80)

print(f"Model A: {agreement_summary['Model_A']}")
print(f"Model B: {agreement_summary['Model_B']}")
print(
    f"Prediction correlation: "
    f"{agreement_summary['Prediction_Correlation']:.6f}"
)
print(
    f"Mean absolute prediction difference: "
    f"{agreement_summary['Mean_Absolute_Prediction_Difference']:.6f}"
)
print(
    f"Prediction difference RMSE: "
    f"{agreement_summary['RMSE_Between_Model_Predictions']:.6f}"
)
print(
    f"Maximum absolute prediction difference: "
    f"{agreement_summary['Maximum_Absolute_Prediction_Difference']:.6f}"
)

print("\n" + "=" * 80)
print("PHASE 15.8 — OUTPUT FILES")
print("=" * 80)

print(f"Predictions saved:")
print(PREDICTION_FILE)

print(f"\nMetrics saved:")
print(METRICS_FILE)

print(f"\nPrediction agreement saved:")
print(AGREEMENT_FILE)

print("\n" + "=" * 80)
print("PHASE 15.8 COMPLETED")
print("FINAL HOLDOUT: FROZEN / UNTOUCHED")
print("NEXT: PHASE 15.9 — OPTIMIZE TOP-TWO ENSEMBLE WEIGHT")
print("=" * 80)

In [ ]:
# =============================================================================
# DELHI AQI PROJECT — PHASE 15.8 CHECKPOINT
# =============================================================================
# Restart-safe session checkpoint.
# Completed through Phase 15.8.
# Final holdout remains frozen and untouched.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
from datetime import datetime

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

CHECKPOINT_DIR = os.path.join(
    BASE_DIR, "outputs/checkpoints"
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "current_phase": "15.8",
    "status": "PAUSED",
    "last_completed_phase": "15.8",
    "next_phase": "15.9",
    "target": "AQI_t_plus_1",
    "scope": {
        "city": "Delhi",
        "stations": 10,
        "feature_count": 73,
        "validation_records": 69097,
        "final_holdout_status": "FROZEN / UNTOUCHED"
    },

    "selected_top_two_models": [
        {
            "rank": 1,
            "model": "Tuned Extra Trees",
            "artifact": os.path.join(
                BASE_DIR,
                "models/phase15_5_tuned_extra_trees.joblib"
            ),
            "validation_mae": 3.624170,
            "validation_rmse": 8.286682,
            "validation_r2": 0.995394,
            "accuracy_pm10": 93.261647,
            "accuracy_pm20": 97.895712
        },
        {
            "rank": 2,
            "model": "Tuned Random Forest",
            "artifact": os.path.join(
                BASE_DIR,
                "models/phase15_4_tuned_random_forest.joblib"
            ),
            "validation_mae": 3.556331,
            "validation_rmse": 8.331690,
            "validation_r2": 0.995344,
            "accuracy_pm10": 93.567014,
            "accuracy_pm20": 98.001360
        }
    ],

    "prediction_agreement": {
        "correlation": 0.999867,
        "mean_absolute_prediction_difference": 1.069531,
        "prediction_difference_rmse": 1.989679,
        "maximum_absolute_prediction_difference": 90.973143
    },

    "important_files": {
        "feature_data":
            os.path.join(
                BASE_DIR,
                "data/processed/delhi_10_station_feature_engineered.parquet"
            ),

        "time_split":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_1_time_split_assignments.parquet"
            ),

        "model_competition":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_7_final_model_comparison.csv"
            ),

        "top_two_config":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_7_top_two_models.json"
            ),

        "top_two_validation_predictions":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_8_top_two_validation_predictions.parquet"
            ),

        "top_two_validation_metrics":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_8_top_two_validation_metrics.csv"
            ),

        "prediction_agreement":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_8_top_two_prediction_agreement.json"
            ),

        "tuned_extra_trees":
            os.path.join(
                BASE_DIR,
                "models/phase15_5_tuned_extra_trees.joblib"
            ),

        "tuned_random_forest":
            os.path.join(
                BASE_DIR,
                "models/phase15_4_tuned_random_forest.joblib"
            ),

        "tuned_xgboost":
            os.path.join(
                BASE_DIR,
                "models/phase15_6_tuned_xgboost.joblib"
            )
    },

    "resume_instruction": (
        "Mount Google Drive, load this checkpoint, then continue with "
        "Phase 15.9 — Optimize Top-Two Ensemble Weight. "
        "Do not retrain completed models and do not access the final holdout "
        "until the final evaluation phase."
    ),

    "checkpoint_created": datetime.now().isoformat()
}

checkpoint_file = os.path.join(
    CHECKPOINT_DIR,
    "phase15_8_checkpoint.json"
)

with open(checkpoint_file, "w") as f:
    json.dump(checkpoint, f, indent=2)

# Also update the general session status
session_status = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "last_completed_phase": "15.8",
    "current_status": "PAUSED",
    "next_phase": "15.9",
    "final_holdout": "FROZEN / UNTOUCHED",
    "checkpoint_file": checkpoint_file,
    "updated": datetime.now().isoformat()
}

session_status_file = os.path.join(
    CHECKPOINT_DIR,
    "session_status.json"
)

with open(session_status_file, "w") as f:
    json.dump(session_status, f, indent=2)

print("=" * 80)
print("DELHI AQI PROJECT — CHECKPOINT SAVED")
print("=" * 80)
print(f"Checkpoint:     {checkpoint_file}")
print(f"Session status: {session_status_file}")
print("Last completed: Phase 15.8")
print("Next phase:     Phase 15.9")
print("Holdout:        FROZEN / UNTOUCHED")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.9 — OPTIMIZE TOP-TWO ENSEMBLE WEIGHT
# =============================================================================
# Restart-safe:
# - Does NOT retrain any model
# - Loads only saved Phase 15.8 validation predictions
# - Optimizes weighted blend of the selected top two models
# - Primary optimization objective: minimum validation RMSE
# - Also evaluates MAE, R2, Accuracy ±10, Accuracy ±20
# - Compares optimized blend against both individual models and 50/50 blend
# - FINAL_HOLDOUT remains completely untouched
#
# Ensemble:
#   Prediction = w * Model_A + (1-w) * Model_B
#
# Model A = Tuned Extra Trees
# Model B = Tuned Random Forest
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =============================================================================
# 1. PATHS
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

TOP_TWO_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_7_top_two_models.json"
)

PREDICTIONS_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_8_top_two_validation_predictions.parquet"
)

OUTPUT_PREDICTIONS_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_optimized_ensemble_validation_predictions.parquet"
)

OUTPUT_METRICS_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_ensemble_validation_results.csv"
)

OUTPUT_WEIGHT_SEARCH_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_ensemble_weight_search.csv"
)

OUTPUT_CONFIG_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_optimized_ensemble_config.json"
)

# =============================================================================
# 2. VERIFY REQUIRED FILES
# =============================================================================

required_files = {
    "Phase 15.7 top-two configuration": TOP_TWO_FILE,
    "Phase 15.8 validation predictions": PREDICTIONS_FILE
}

for label, path in required_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )

# =============================================================================
# 3. LOAD TOP-TWO CONFIGURATION
# =============================================================================

with open(TOP_TWO_FILE, "r") as f:
    top_two_config = json.load(f)

if isinstance(top_two_config, dict):

    if "top_two_models" in top_two_config:
        top_two = top_two_config["top_two_models"]

    elif "top_two" in top_two_config:
        top_two = top_two_config["top_two"]

    else:
        list_values = [
            value
            for value in top_two_config.values()
            if isinstance(value, list)
        ]

        if not list_values:
            raise ValueError(
                "Could not locate top-two model information."
            )

        top_two = list_values[0]

else:
    top_two = top_two_config

if len(top_two) != 2:
    raise ValueError(
        f"Expected exactly 2 top models, found {len(top_two)}."
    )


def extract_model_name(item):

    if isinstance(item, str):
        return item

    if isinstance(item, dict):

        for key in [
            "Model",
            "model",
            "model_name",
            "Model Name",
            "name"
        ]:
            if key in item:
                return str(item[key])

    raise ValueError(
        f"Could not extract model name from:\n{item}"
    )


model_a = extract_model_name(top_two[0])
model_b = extract_model_name(top_two[1])

print("=" * 80)
print("PHASE 15.9 — OPTIMIZE TOP-TWO ENSEMBLE WEIGHT")
print("=" * 80)

print(f"\nModel A: {model_a}")
print(f"Model B: {model_b}")

# =============================================================================
# 4. LOAD PHASE 15.8 PREDICTIONS
# =============================================================================

pred_df = pd.read_parquet(
    PREDICTIONS_FILE
)

print("\nLoaded Phase 15.8 prediction file:")
print(f"Rows: {len(pred_df):,}")
print(f"Columns: {len(pred_df.columns)}")

# =============================================================================
# 5. IDENTIFY REQUIRED COLUMNS
# =============================================================================

TARGET_COL = "Actual_AQI_t_plus_1"

if TARGET_COL not in pred_df.columns:
    raise KeyError(
        f"Missing target column: {TARGET_COL}"
    )


def model_to_prediction_column(model_name):

    return (
        model_name
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        + "_Prediction"
    )


pred_a_col = model_to_prediction_column(model_a)
pred_b_col = model_to_prediction_column(model_b)

if pred_a_col not in pred_df.columns:
    raise KeyError(
        f"Prediction column not found for {model_a}:\n"
        f"{pred_a_col}"
    )

if pred_b_col not in pred_df.columns:
    raise KeyError(
        f"Prediction column not found for {model_b}:\n"
        f"{pred_b_col}"
    )

# =============================================================================
# 6. PREPARE ARRAYS
# =============================================================================

actual = pred_df[TARGET_COL].astype(float).to_numpy()

prediction_a = (
    pred_df[pred_a_col]
    .astype(float)
    .to_numpy()
)

prediction_b = (
    pred_df[pred_b_col]
    .astype(float)
    .to_numpy()
)

if not np.isfinite(actual).all():
    raise ValueError(
        "Actual AQI contains NaN or infinite values."
    )

if not np.isfinite(prediction_a).all():
    raise ValueError(
        f"{model_a} predictions contain NaN or infinite values."
    )

if not np.isfinite(prediction_b).all():
    raise ValueError(
        f"{model_b} predictions contain NaN or infinite values."
    )

print("\nValidation data verified:")
print(f"Validation records: {len(actual):,}")

# =============================================================================
# 7. METRIC FUNCTION
# =============================================================================

def calculate_metrics(actual_values, predicted_values):

    mae = mean_absolute_error(
        actual_values,
        predicted_values
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_values,
            predicted_values
        )
    )

    r2 = r2_score(
        actual_values,
        predicted_values
    )

    accuracy_10 = (
        np.mean(
            np.abs(
                actual_values - predicted_values
            ) <= 10
        ) * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(
                actual_values - predicted_values
            ) <= 20
        ) * 100
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Accuracy ±10 AQI": accuracy_10,
        "Accuracy ±20 AQI": accuracy_20
    }

# =============================================================================
# 8. INDIVIDUAL MODEL BASELINES
# =============================================================================

metrics_rows = []

metrics_a = calculate_metrics(
    actual,
    prediction_a
)

metrics_rows.append({
    "Model": model_a,
    "Model Type": "Selected Top-Two",
    "Weight Model A": 1.0,
    "Weight Model B": 0.0,
    **metrics_a
})

metrics_b = calculate_metrics(
    actual,
    prediction_b
)

metrics_rows.append({
    "Model": model_b,
    "Model Type": "Selected Top-Two",
    "Weight Model A": 0.0,
    "Weight Model B": 1.0,
    **metrics_b
})

# =============================================================================
# 9. 50/50 BASELINE ENSEMBLE
# =============================================================================

equal_weight = 0.5

prediction_5050 = (
    equal_weight * prediction_a
    + (1 - equal_weight) * prediction_b
)

metrics_5050 = calculate_metrics(
    actual,
    prediction_5050
)

metrics_rows.append({
    "Model": "50/50 Ensemble",
    "Model Type": "Equal-Weight Ensemble",
    "Weight Model A": equal_weight,
    "Weight Model B": 1 - equal_weight,
    **metrics_5050
})

# =============================================================================
# 10. ANALYTICAL RMSE-OPTIMAL WEIGHT
# =============================================================================
#
# Minimize:
#
#   SSE(w) = sum[(y - (w*A + (1-w)*B))^2]
#
# Let:
#   eA = y - A
#   eB = y - B
#
# Optimal unconstrained weight:
#
#   w = sum[eB * (eB - eA)] / sum[(eA - eB)^2]
#
# Then constrain w to [0, 1].
# =============================================================================

error_a = actual - prediction_a
error_b = actual - prediction_b

denominator = np.sum(
    (error_a - error_b) ** 2
)

if denominator == 0:

    optimal_rmse_weight = 0.5

    print(
        "\nWARNING: Both prediction series are identical."
    )

else:

    optimal_rmse_weight = (
        np.sum(
            error_b * (error_b - error_a)
        )
        / denominator
    )

    optimal_rmse_weight = float(
        np.clip(
            optimal_rmse_weight,
            0.0,
            1.0
        )
    )

optimal_rmse_weight_b = (
    1.0 - optimal_rmse_weight
)

# =============================================================================
# 11. VERIFY ANALYTICAL SOLUTION WITH FINE WEIGHT GRID
# =============================================================================

weights = np.linspace(
    0.0,
    1.0,
    1001
)

grid_results = []

best_grid_weight = None
best_grid_rmse = np.inf

for weight in weights:

    blend_prediction = (
        weight * prediction_a
        + (1 - weight) * prediction_b
    )

    blend_rmse = np.sqrt(
        mean_squared_error(
            actual,
            blend_prediction
        )
    )

    blend_mae = mean_absolute_error(
        actual,
        blend_prediction
    )

    blend_acc10 = (
        np.mean(
            np.abs(
                actual - blend_prediction
            ) <= 10
        ) * 100
    )

    blend_acc20 = (
        np.mean(
            np.abs(
                actual - blend_prediction
            ) <= 20
        ) * 100
    )

    grid_results.append({
        "Weight_Model_A": weight,
        "Weight_Model_B": 1 - weight,
        "RMSE": blend_rmse,
        "MAE": blend_mae,
        "Accuracy ±10 AQI": blend_acc10,
        "Accuracy ±20 AQI": blend_acc20
    })

    if blend_rmse < best_grid_rmse:
        best_grid_rmse = blend_rmse
        best_grid_weight = weight

grid_df = pd.DataFrame(
    grid_results
)

# =============================================================================
# 12. FINAL RMSE-OPTIMIZED ENSEMBLE PREDICTION
# =============================================================================

ensemble_prediction = (
    optimal_rmse_weight * prediction_a
    + optimal_rmse_weight_b * prediction_b
)

ensemble_metrics = calculate_metrics(
    actual,
    ensemble_prediction
)

metrics_rows.append({
    "Model": "Optimized Weighted Ensemble",
    "Model Type": "RMSE-Optimized Ensemble",
    "Weight Model A": optimal_rmse_weight,
    "Weight Model B": optimal_rmse_weight_b,
    **ensemble_metrics
})

results_df = pd.DataFrame(
    metrics_rows
)

# =============================================================================
# 13. ADD RMSE/MAE IMPROVEMENT COLUMNS
# =============================================================================

rmse_a = metrics_a["RMSE"]
mae_a = metrics_a["MAE"]

rmse_b = metrics_b["RMSE"]
mae_b = metrics_b["MAE"]

results_df["RMSE Improvement vs Model A"] = (
    rmse_a - results_df["RMSE"]
)

results_df["RMSE Improvement vs Model B"] = (
    rmse_b - results_df["RMSE"]
)

results_df["MAE Improvement vs Model A"] = (
    mae_a - results_df["MAE"]
)

results_df["MAE Improvement vs Model B"] = (
    mae_b - results_df["MAE"]
)

# =============================================================================
# 14. SAVE WEIGHT SEARCH
# =============================================================================

grid_df.to_csv(
    OUTPUT_WEIGHT_SEARCH_FILE,
    index=False
)

# =============================================================================
# 15. SAVE ENSEMBLE VALIDATION PREDICTIONS
# =============================================================================

ensemble_output = pred_df.copy()

ensemble_output["Optimized_Ensemble_Prediction"] = (
    ensemble_prediction
)

ensemble_output["Ensemble_Error"] = (
    actual - ensemble_prediction
)

ensemble_output.to_parquet(
    OUTPUT_PREDICTIONS_FILE,
    index=False
)

# =============================================================================
# 16. SAVE METRICS
# =============================================================================

results_df.to_csv(
    OUTPUT_METRICS_FILE,
    index=False
)

# =============================================================================
# 17. SAVE CONFIGURATION
# =============================================================================

ensemble_config = {

    "phase": "15.9",

    "objective": (
        "Minimize validation RMSE "
        "using weighted blend of selected top-two models"
    ),

    "formula": (
        "prediction = w * Model_A + (1-w) * Model_B"
    ),

    "model_a": model_a,

    "model_b": model_b,

    "weight_model_a": optimal_rmse_weight,

    "weight_model_b": optimal_rmse_weight_b,

    "analytical_solution_rmse": True,

    "grid_verification": {
        "grid_points": 1001,
        "best_grid_weight_model_a": float(
            best_grid_weight
        ),
        "best_grid_weight_model_b": float(
            1 - best_grid_weight
        ),
        "best_grid_rmse": float(
            best_grid_rmse
        )
    },

    "validation_records": int(len(actual)),

    "optimized_validation_metrics": ensemble_metrics,

    "model_a_validation_metrics": metrics_a,

    "model_b_validation_metrics": metrics_b,

    "equal_weight_50_50_metrics": metrics_5050,

    "final_holdout_status": (
        "FROZEN / UNTOUCHED"
    ),

    "next_phase": "15.10"
}

with open(
    OUTPUT_CONFIG_FILE,
    "w"
) as f:

    json.dump(
        ensemble_config,
        f,
        indent=2
    )

# =============================================================================
# 18. DISPLAY FINAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.9 — OPTIMIZED ENSEMBLE WEIGHT")
print("=" * 80)

print(
    f"\nModel A: {model_a}"
)

print(
    f"Model B: {model_b}"
)

print(
    f"\nOptimal Model A weight: "
    f"{optimal_rmse_weight:.6f}"
)

print(
    f"Optimal Model B weight: "
    f"{optimal_rmse_weight_b:.6f}"
)

print(
    f"\nFine-grid Model A weight: "
    f"{best_grid_weight:.3f}"
)

print(
    f"Fine-grid RMSE: "
    f"{best_grid_rmse:.6f}"
)

print(
    f"Analytical RMSE: "
    f"{ensemble_metrics['RMSE']:.6f}"
)

print("\n" + "=" * 80)
print("PHASE 15.9 — MODEL / ENSEMBLE COMPARISON")
print("=" * 80)

display(
    results_df.style.format({
        "Weight Model A": "{:.6f}",
        "Weight Model B": "{:.6f}",
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy ±10 AQI": "{:.6f}",
        "Accuracy ±20 AQI": "{:.6f}",
        "RMSE Improvement vs Model A": "{:.6f}",
        "RMSE Improvement vs Model B": "{:.6f}",
        "MAE Improvement vs Model A": "{:.6f}",
        "MAE Improvement vs Model B": "{:.6f}"
    })
)

# =============================================================================
# 19. IMPORTANT HOLDOUT SAFETY CHECK
# =============================================================================

print("\n" + "=" * 80)
print("HOLDOUT SAFETY")
print("=" * 80)

print(
    "FINAL_HOLDOUT was NOT loaded, evaluated, tuned, or used "
    "for ensemble-weight optimization."
)

print(
    "Optimization used VALIDATION data only."
)

print("\n" + "=" * 80)
print("PHASE 15.9 — OUTPUT FILES")
print("=" * 80)

print(
    f"Optimized ensemble predictions:\n"
    f"{OUTPUT_PREDICTIONS_FILE}"
)

print(
    f"\nEnsemble validation results:\n"
    f"{OUTPUT_METRICS_FILE}"
)

print(
    f"\nWeight search:\n"
    f"{OUTPUT_WEIGHT_SEARCH_FILE}"
)

print(
    f"\nEnsemble configuration:\n"
    f"{OUTPUT_CONFIG_FILE}"
)

print("\n" + "=" * 80)
print("PHASE 15.9 COMPLETED")
print("FINAL HOLDOUT: FROZEN / UNTOUCHED")
print("NEXT: PHASE 15.10 — ENSEMBLE VALIDATION & FINAL MODEL PREPARATION")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.10 — ENSEMBLE VALIDATION & FINAL MODEL PREPARATION
# =============================================================================
#
# Purpose:
# 1. Load the frozen Phase 15.9 optimized ensemble predictions.
# 2. Validate the ensemble in more detail:
#       - Overall metrics
#       - Per-station metrics
#       - AQI-range metrics
#       - Error distribution
# 3. Freeze the selected ensemble weight.
# 4. Load the tuned ET + RF artifacts.
# 5. Refit both component models on TRAIN + VALIDATION only.
# 6. Save the refitted final component models.
# 7. Save the complete final ensemble configuration.
#
# IMPORTANT:
# FINAL_HOLDOUT is never loaded into X/y for training or optimization.
# It remains reserved for the final one-time evaluation.
#
# Restart-safe: no previous Python variables are required.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =============================================================================
# 1. PATHS
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data/processed/delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_1_time_split_assignments.parquet"
)

P15_9_PREDICTIONS = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_optimized_ensemble_validation_predictions.parquet"
)

P15_9_CONFIG = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_optimized_ensemble_config.json"
)

P15_9_RESULTS = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_ensemble_validation_results.csv"
)

ET_SOURCE_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_5_tuned_extra_trees.joblib"
)

RF_SOURCE_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_4_tuned_random_forest.joblib"
)

FINAL_ET_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
)

FINAL_RF_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_random_forest_trainval.joblib"
)

OVERALL_VALIDATION_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_ensemble_validation_summary.csv"
)

STATION_VALIDATION_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_ensemble_station_validation.csv"
)

AQI_RANGE_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_ensemble_aqi_range_validation.csv"
)

ERROR_SUMMARY_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_ensemble_error_summary.json"
)

FINAL_CONFIG_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_final_ensemble_config.json"
)

os.makedirs(
    os.path.join(BASE_DIR, "models"),
    exist_ok=True
)

os.makedirs(
    os.path.join(BASE_DIR, "outputs/reports"),
    exist_ok=True
)

# =============================================================================
# 2. VERIFY REQUIRED FILES
# =============================================================================

required_files = {
    "Feature engineered data": FEATURE_FILE,
    "Frozen split assignments": SPLIT_FILE,
    "Phase 15.9 ensemble predictions": P15_9_PREDICTIONS,
    "Phase 15.9 ensemble config": P15_9_CONFIG,
    "Phase 15.9 ensemble results": P15_9_RESULTS,
    "Tuned Extra Trees artifact": ET_SOURCE_MODEL,
    "Tuned Random Forest artifact": RF_SOURCE_MODEL,
}

for label, path in required_files.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )

# =============================================================================
# 3. LOAD FROZEN ENSEMBLE CONFIGURATION
# =============================================================================

with open(P15_9_CONFIG, "r") as f:
    ensemble_config = json.load(f)

MODEL_A = ensemble_config["model_a"]
MODEL_B = ensemble_config["model_b"]

WEIGHT_A = float(
    ensemble_config["weight_model_a"]
)

WEIGHT_B = float(
    ensemble_config["weight_model_b"]
)

if not np.isclose(
    WEIGHT_A + WEIGHT_B,
    1.0,
    atol=1e-10
):
    raise ValueError(
        "Ensemble weights do not sum to 1."
    )

if MODEL_A != "Tuned Extra Trees":
    raise ValueError(
        f"Unexpected Model A: {MODEL_A}"
    )

if MODEL_B != "Tuned Random Forest":
    raise ValueError(
        f"Unexpected Model B: {MODEL_B}"
    )

print("=" * 80)
print("PHASE 15.10 — ENSEMBLE VALIDATION & FINAL MODEL PREPARATION")
print("=" * 80)

print("\nFrozen ensemble:")
print(f"Model A: {MODEL_A}")
print(f"Model B: {MODEL_B}")
print(f"Weight A: {WEIGHT_A:.6f}")
print(f"Weight B: {WEIGHT_B:.6f}")

# =============================================================================
# 4. LOAD PHASE 15.9 VALIDATION PREDICTIONS
# =============================================================================

validation_pred = pd.read_parquet(
    P15_9_PREDICTIONS
)

TARGET_PRED_COL = "Actual_AQI_t_plus_1"
ENSEMBLE_PRED_COL = "Optimized_Ensemble_Prediction"

ET_PRED_COL = (
    "Tuned_Extra_Trees_Prediction"
)

RF_PRED_COL = (
    "Tuned_Random_Forest_Prediction"
)

required_prediction_columns = [
    TARGET_PRED_COL,
    ET_PRED_COL,
    RF_PRED_COL,
    ENSEMBLE_PRED_COL
]

for col in required_prediction_columns:

    if col not in validation_pred.columns:
        raise KeyError(
            f"Missing required prediction column: {col}"
        )

if len(validation_pred) != 69097:
    raise ValueError(
        f"Expected 69,097 validation rows, "
        f"found {len(validation_pred):,}."
    )

print("\nValidation predictions loaded:")
print(f"Rows: {len(validation_pred):,}")
print(
    f"Columns: "
    f"{validation_pred.columns.tolist()}"
)

# =============================================================================
# 5. VERIFY ENSEMBLE PREDICTION REPRODUCES THE FROZEN WEIGHT
# =============================================================================

actual = (
    validation_pred[TARGET_PRED_COL]
    .astype(float)
    .to_numpy()
)

et_pred = (
    validation_pred[ET_PRED_COL]
    .astype(float)
    .to_numpy()
)

rf_pred = (
    validation_pred[RF_PRED_COL]
    .astype(float)
    .to_numpy()
)

saved_ensemble_pred = (
    validation_pred[ENSEMBLE_PRED_COL]
    .astype(float)
    .to_numpy()
)

reconstructed_ensemble_pred = (
    WEIGHT_A * et_pred
    + WEIGHT_B * rf_pred
)

max_reconstruction_difference = float(
    np.max(
        np.abs(
            saved_ensemble_pred
            - reconstructed_ensemble_pred
        )
    )
)

print(
    f"\nMaximum ensemble reconstruction difference: "
    f"{max_reconstruction_difference:.12f}"
)

if max_reconstruction_difference > 1e-8:
    raise ValueError(
        "Saved ensemble predictions do not match "
        "the frozen Phase 15.9 weights."
    )

ensemble_pred = reconstructed_ensemble_pred

# =============================================================================
# 6. METRIC FUNCTION
# =============================================================================

def compute_metrics(y_true, y_pred):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    acc10 = (
        np.mean(
            np.abs(
                y_true - y_pred
            ) <= 10
        )
        * 100
    )

    acc20 = (
        np.mean(
            np.abs(
                y_true - y_pred
            ) <= 20
        )
        * 100
    )

    return {
        "Records": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Accuracy ±10 AQI": acc10,
        "Accuracy ±20 AQI": acc20
    }

# =============================================================================
# 7. OVERALL ENSEMBLE VALIDATION
# =============================================================================

overall_metrics = compute_metrics(
    actual,
    ensemble_pred
)

overall_row = pd.DataFrame([
    {
        "Model": "Optimized Weighted Ensemble",
        **overall_metrics
    }
])

overall_row.to_csv(
    OVERALL_VALIDATION_FILE,
    index=False
)

print("\n" + "=" * 80)
print("PHASE 15.10 — OVERALL ENSEMBLE VALIDATION")
print("=" * 80)

display(
    overall_row.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy ±10 AQI": "{:.6f}",
        "Accuracy ±20 AQI": "{:.6f}"
    })
)

# =============================================================================
# 8. PER-STATION VALIDATION
# =============================================================================

if "StationId" not in validation_pred.columns:
    raise KeyError(
        "StationId not found in validation predictions."
    )

station_rows = []

for station_id, group in validation_pred.groupby(
    "StationId",
    sort=True
):

    y_station = (
        group[TARGET_PRED_COL]
        .astype(float)
        .to_numpy()
    )

    p_station = (
        group[ENSEMBLE_PRED_COL]
        .astype(float)
        .to_numpy()
    )

    metrics_station = compute_metrics(
        y_station,
        p_station
    )

    station_rows.append({
        "StationId": station_id,
        **metrics_station
    })

station_df = pd.DataFrame(
    station_rows
)

station_df = station_df.sort_values(
    "RMSE"
).reset_index(drop=True)

station_df.to_csv(
    STATION_VALIDATION_FILE,
    index=False
)

print("\n" + "=" * 80)
print("PHASE 15.10 — STATION-LEVEL ENSEMBLE VALIDATION")
print("=" * 80)

display(
    station_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy ±10 AQI": "{:.6f}",
        "Accuracy ±20 AQI": "{:.6f}"
    })
)

# =============================================================================
# 9. AQI-RANGE VALIDATION
# =============================================================================
#
# This evaluates whether performance changes across pollution severity.
#
# Bins:
#   <100
#   100-199
#   200-299
#   300-399
#   >=400
# =============================================================================

aqi_bins = [
    -np.inf,
    100,
    200,
    300,
    400,
    np.inf
]

aqi_labels = [
    "<100",
    "100–199",
    "200–299",
    "300–399",
    "≥400"
]

range_work = validation_pred.copy()

range_work["Actual_AQI_Range"] = pd.cut(
    range_work[TARGET_PRED_COL],
    bins=aqi_bins,
    labels=aqi_labels,
    right=False
)

range_rows = []

for aqi_range, group in range_work.groupby(
    "Actual_AQI_Range",
    observed=True
):

    y_range = (
        group[TARGET_PRED_COL]
        .astype(float)
        .to_numpy()
    )

    p_range = (
        group[ENSEMBLE_PRED_COL]
        .astype(float)
        .to_numpy()
    )

    metrics_range = compute_metrics(
        y_range,
        p_range
    )

    range_rows.append({
        "AQI Range": str(aqi_range),
        **metrics_range
    })

aqi_range_df = pd.DataFrame(
    range_rows
)

aqi_range_df.to_csv(
    AQI_RANGE_FILE,
    index=False
)

print("\n" + "=" * 80)
print("PHASE 15.10 — AQI-RANGE ENSEMBLE VALIDATION")
print("=" * 80)

display(
    aqi_range_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy ±10 AQI": "{:.6f}",
        "Accuracy ±20 AQI": "{:.6f}"
    })
)

# =============================================================================
# 10. ERROR DISTRIBUTION
# =============================================================================

ensemble_error = (
    actual - ensemble_pred
)

error_summary = {
    "mean_error": float(
        np.mean(ensemble_error)
    ),
    "median_error": float(
        np.median(ensemble_error)
    ),
    "std_error": float(
        np.std(ensemble_error)
    ),
    "mean_absolute_error": float(
        np.mean(np.abs(ensemble_error))
    ),
    "p90_absolute_error": float(
        np.percentile(
            np.abs(ensemble_error),
            90
        )
    ),
    "p95_absolute_error": float(
        np.percentile(
            np.abs(ensemble_error),
            95
        )
    ),
    "p99_absolute_error": float(
        np.percentile(
            np.abs(ensemble_error),
            99
        )
    ),
    "max_absolute_error": float(
        np.max(
            np.abs(ensemble_error)
        )
    )
}

with open(
    ERROR_SUMMARY_FILE,
    "w"
) as f:
    json.dump(
        error_summary,
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("PHASE 15.10 — ERROR DISTRIBUTION")
print("=" * 80)

for key, value in error_summary.items():
    print(
        f"{key:30s}: {value:.6f}"
    )

# =============================================================================
# 11. LOAD SOURCE TUNED MODELS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.10 — LOADING TUNED COMPONENT MODELS")
print("=" * 80)

et_source = joblib.load(
    ET_SOURCE_MODEL
)

rf_source = joblib.load(
    RF_SOURCE_MODEL
)

print(
    f"Loaded Tuned Extra Trees artifact: "
    f"{type(et_source)}"
)

print(
    f"Loaded Tuned Random Forest artifact: "
    f"{type(rf_source)}"
)

# =============================================================================
# 12. RECREATE EXACT 73-FEATURE CONTRACT
# =============================================================================

TARGET = "AQI_t_plus_1"
DATETIME = "Datetime"
STATION = "StationId"

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
]

pollutant_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in pollutant_features
]

rolling_source_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2",
]

rolling_features = []

for feature in rolling_source_features:
    for window in [3, 12]:
        rolling_features.append(
            f"{feature}_rolling_mean_{window}h"
        )
        rolling_features.append(
            f"{feature}_rolling_std_{window}h"
        )

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow",
]

feature_cols = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + [STATION]
)

if len(feature_cols) != 73:
    raise ValueError(
        f"Expected 73 features, generated {len(feature_cols)}."
    )

# =============================================================================
# 13. LOAD FEATURE DATA + FROZEN SPLIT
# =============================================================================

df = pd.read_parquet(
    FEATURE_FILE
)

df[DATETIME] = pd.to_datetime(
    df[DATETIME],
    errors="coerce"
)

split_df = pd.read_parquet(
    SPLIT_FILE
)

split_df[DATETIME] = pd.to_datetime(
    split_df[DATETIME],
    errors="coerce"
)

if "Split" not in split_df.columns:
    raise KeyError(
        "Expected 'Split' column in Phase 15.1 split file."
    )

# Join split using station + datetime
split_keys = [
    STATION,
    DATETIME
]

if split_df.duplicated(
    split_keys
).any():

    raise ValueError(
        "Duplicate StationId-Datetime keys in split assignments."
    )

split_lookup = split_df[
    split_keys + ["Split"]
].copy()

model_df = df.merge(
    split_lookup,
    on=split_keys,
    how="left",
    validate="one_to_one"
)

if model_df["Split"].isna().any():
    raise ValueError(
        "Some rows have no split assignment."
    )

# =============================================================================
# 14. TRAIN + VALIDATION ONLY
# =============================================================================

trainval_mask = model_df["Split"].isin([
    "TRAIN",
    "VALIDATION"
])

holdout_mask = model_df["Split"].eq(
    "FINAL_HOLDOUT"
)

unused_gap_mask = model_df["Split"].eq(
    "UNUSED_GAP"
)

trainval_df = (
    model_df.loc[trainval_mask]
    .copy()
)

print("\n" + "=" * 80)
print("PHASE 15.10 — FINAL COMPONENT MODEL TRAINING DATA")
print("=" * 80)

print(
    f"TRAIN + VALIDATION rows: "
    f"{len(trainval_df):,}"
)

print(
    f"Final holdout rows present in dataframe: "
    f"{holdout_mask.sum():,}"
)

print(
    f"Unused gap rows: "
    f"{unused_gap_mask.sum():,}"
)

# =============================================================================
# 15. HARD SAFETY CHECK — HOLDOUT NOT IN TRAINING
# =============================================================================

if trainval_df["Split"].eq(
    "FINAL_HOLDOUT"
).any():

    raise RuntimeError(
        "SAFETY FAILURE: FINAL_HOLDOUT entered training data."
    )

if len(trainval_df) != 235942:
    print(
        "NOTE: TRAIN + VALIDATION row count is "
        f"{len(trainval_df):,}."
    )

# =============================================================================
# 16. PREPARE X / y FOR FINAL COMPONENT MODELS
# =============================================================================

required_train_columns = (
    feature_cols
    + [TARGET]
)

missing_train_columns = [
    col
    for col in required_train_columns
    if col not in trainval_df.columns
]

if missing_train_columns:
    raise KeyError(
        "Missing training columns:\n"
        + "\n".join(missing_train_columns)
    )

trainval_df = trainval_df[
    trainval_df[TARGET].notna()
].copy()

X_trainval = trainval_df[
    feature_cols
].copy()

y_trainval = trainval_df[
    TARGET
].astype(float)

print(
    f"\nFinal component training rows: "
    f"{len(X_trainval):,}"
)

print(
    f"Feature count: "
    f"{X_trainval.shape[1]}"
)

# =============================================================================
# 17. CLONE + REFIT TUNED EXTRA TREES
# =============================================================================

print("\n" + "=" * 80)
print("REFITTING FINAL TUNED EXTRA TREES")
print("=" * 80)

try:

    final_et = clone(
        et_source
    )

except Exception:

    raise TypeError(
        "The saved Tuned Extra Trees artifact cannot be cloned. "
        "Inspect the artifact structure before refitting."
    )

final_et.fit(
    X_trainval,
    y_trainval
)

joblib.dump(
    final_et,
    FINAL_ET_MODEL
)

print(
    f"Final Tuned Extra Trees saved:\n"
    f"{FINAL_ET_MODEL}"
)

# =============================================================================
# 18. CLONE + REFIT TUNED RANDOM FOREST
# =============================================================================

print("\n" + "=" * 80)
print("REFITTING FINAL TUNED RANDOM FOREST")
print("=" * 80)

try:

    final_rf = clone(
        rf_source
    )

except Exception:

    raise TypeError(
        "The saved Tuned Random Forest artifact cannot be cloned. "
        "Inspect the artifact structure before refitting."
    )

final_rf.fit(
    X_trainval,
    y_trainval
)

joblib.dump(
    final_rf,
    FINAL_RF_MODEL
)

print(
    f"Final Tuned Random Forest saved:\n"
    f"{FINAL_RF_MODEL}"
)

# =============================================================================
# 19. FINAL ENSEMBLE CONFIGURATION
# =============================================================================

final_config = {

    "phase": "15.10",

    "project": "Delhi AQI t+1 Hour Forecasting",

    "target": TARGET,

    "station_count": int(
        trainval_df[STATION].nunique()
    ),

    "feature_count": len(
        feature_cols
    ),

    "component_models": [
        {
            "model": MODEL_A,
            "weight": WEIGHT_A,
            "training_data": "TRAIN + VALIDATION",
            "artifact": FINAL_ET_MODEL
        },
        {
            "model": MODEL_B,
            "weight": WEIGHT_B,
            "training_data": "TRAIN + VALIDATION",
            "artifact": FINAL_RF_MODEL
        }
    ],

    "ensemble_formula": (
        "prediction = "
        f"{WEIGHT_A:.12f} * Tuned Extra Trees + "
        f"{WEIGHT_B:.12f} * Tuned Random Forest"
    ),

    "ensemble_weights": {
        "Tuned Extra Trees": WEIGHT_A,
        "Tuned Random Forest": WEIGHT_B
    },

    "weight_objective": (
        "Minimum validation RMSE"
    ),

    "weight_selected_in_phase": "15.9",

    "validation_records": int(
        len(validation_pred)
    ),

    "validation_rmse": float(
        overall_metrics["RMSE"]
    ),

    "validation_mae": float(
        overall_metrics["MAE"]
    ),

    "validation_r2": float(
        overall_metrics["R2"]
    ),

    "validation_accuracy_pm10": float(
        overall_metrics["Accuracy ±10 AQI"]
    ),

    "validation_accuracy_pm20": float(
        overall_metrics["Accuracy ±20 AQI"]
    ),

    "final_component_training_rows": int(
        len(trainval_df)
    ),

    "final_holdout_status": (
        "FROZEN / UNTOUCHED — RESERVED FOR FINAL EVALUATION"
    ),

    "next_phase": "15.11 — XAI / SHAP"
}

with open(
    FINAL_CONFIG_FILE,
    "w"
) as f:

    json.dump(
        final_config,
        f,
        indent=2
    )

# =============================================================================
# 20. VERIFY SAVED FINAL ARTIFACTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.10 — ARTIFACT VERIFICATION")
print("=" * 80)

artifacts_to_verify = {
    "Final Tuned Extra Trees": FINAL_ET_MODEL,
    "Final Tuned Random Forest": FINAL_RF_MODEL,
    "Final ensemble configuration": FINAL_CONFIG_FILE,
    "Overall validation summary": OVERALL_VALIDATION_FILE,
    "Station validation": STATION_VALIDATION_FILE,
    "AQI-range validation": AQI_RANGE_FILE,
    "Error summary": ERROR_SUMMARY_FILE,
}

verification_rows = []

for name, path in artifacts_to_verify.items():

    exists = os.path.exists(path)

    size_mb = (
        os.path.getsize(path) / (1024 ** 2)
        if exists
        else np.nan
    )

    verification_rows.append({
        "Artifact": name,
        "Status": "FOUND" if exists else "MISSING",
        "Size_MB": size_mb
    })

verification_df = pd.DataFrame(
    verification_rows
)

display(
    verification_df.style.format({
        "Size_MB": "{:.3f}"
    })
)

missing_artifacts = verification_df.loc[
    verification_df["Status"] == "MISSING"
]

if not missing_artifacts.empty:
    raise RuntimeError(
        "One or more Phase 15.10 artifacts are missing."
    )

# =============================================================================
# 21. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.10 SUMMARY")
print("=" * 80)

print(
    f"Frozen ensemble weight — {MODEL_A}: "
    f"{WEIGHT_A:.6f}"
)

print(
    f"Frozen ensemble weight — {MODEL_B}: "
    f"{WEIGHT_B:.6f}"
)

print(
    f"Validation RMSE: "
    f"{overall_metrics['RMSE']:.6f}"
)

print(
    f"Validation MAE: "
    f"{overall_metrics['MAE']:.6f}"
)

print(
    f"Validation Accuracy ±10 AQI: "
    f"{overall_metrics['Accuracy ±10 AQI']:.6f}%"
)

print(
    f"Validation Accuracy ±20 AQI: "
    f"{overall_metrics['Accuracy ±20 AQI']:.6f}%"
)

print(
    f"Final component training rows: "
    f"{len(trainval_df):,}"
)

print(
    "\nFINAL_HOLDOUT: FROZEN / UNTOUCHED"
)

print("\nFinal component models:")
print(FINAL_ET_MODEL)
print(FINAL_RF_MODEL)

print("\nFinal ensemble configuration:")
print(FINAL_CONFIG_FILE)

print("\n" + "=" * 80)
print("PHASE 15.10 COMPLETED")
print("FINAL HOLDOUT: FROZEN / UNTOUCHED")
print("NEXT: PHASE 15.11 — XAI / SHAP")
print("=" * 80)

In [ ]:
# =============================================================================
# DELHI AQI PROJECT — PHASE 15.10 CHECKPOINT
# =============================================================================
# Saves the current project state to Google Drive.
# Resume next time from Phase 15.11 — XAI / SHAP.
# FINAL_HOLDOUT remains frozen and untouched.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
from datetime import datetime

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"
CHECKPOINT_DIR = os.path.join(BASE_DIR, "outputs/checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "status": "PAUSED",
    "last_completed_phase": "15.10",
    "next_phase": "15.11",
    "target": "AQI_t_plus_1",

    "phase_15_10": {
        "ensemble_model_a": "Tuned Extra Trees",
        "ensemble_model_b": "Tuned Random Forest",

        "ensemble_weight_model_a": 0.594466,
        "ensemble_weight_model_b": 0.405534,

        "validation_records": 69097,
        "validation_mae": 3.553206,
        "validation_rmse": 8.247305,
        "validation_r2": 0.995438,
        "validation_accuracy_pm10": 93.574251,
        "validation_accuracy_pm20": 97.983994,

        "final_component_training_rows": 235942,

        "final_holdout_rows": 68352,
        "final_holdout_status": "FROZEN / UNTOUCHED"
    },

    "artifacts": {
        "final_extra_trees":
            os.path.join(
                BASE_DIR,
                "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
            ),

        "final_random_forest":
            os.path.join(
                BASE_DIR,
                "models/phase15_10_final_tuned_random_forest_trainval.joblib"
            ),

        "final_ensemble_config":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_10_final_ensemble_config.json"
            ),

        "ensemble_validation_summary":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_10_ensemble_validation_summary.csv"
            ),

        "station_validation":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_10_ensemble_station_validation.csv"
            ),

        "aqi_range_validation":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_10_ensemble_aqi_range_validation.csv"
            ),

        "error_summary":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_10_ensemble_error_summary.json"
            ),

        "phase_15_9_ensemble_predictions":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_9_optimized_ensemble_validation_predictions.parquet"
            )
    },

    "resume_instruction": (
        "Start a fresh Colab runtime, mount Google Drive, verify this "
        "checkpoint and continue directly with Phase 15.11 — XAI / SHAP. "
        "Do not retrain completed models. Do not use FINAL_HOLDOUT before "
        "the final evaluation phase."
    ),

    "checkpoint_created": datetime.now().isoformat()
}

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "phase15_10_checkpoint.json"
)

with open(checkpoint_path, "w") as f:
    json.dump(checkpoint, f, indent=2)

session_status = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "last_completed_phase": "15.10",
    "status": "PAUSED",
    "next_phase": "15.11",
    "final_holdout": "FROZEN / UNTOUCHED",
    "checkpoint": checkpoint_path,
    "updated": datetime.now().isoformat()
}

session_status_path = os.path.join(
    CHECKPOINT_DIR,
    "session_status.json"
)

with open(session_status_path, "w") as f:
    json.dump(session_status, f, indent=2)

print("=" * 80)
print("DELHI AQI PROJECT — CHECKPOINT SAVED")
print("=" * 80)
print(f"Last completed phase : 15.10")
print(f"Next phase           : 15.11 — XAI / SHAP")
print(f"Checkpoint           : {checkpoint_path}")
print(f"Session status       : {session_status_path}")
print("FINAL_HOLDOUT        : FROZEN / UNTOUCHED")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.11 — FAST XAI / SHAP
# =============================================================================
#
# Purpose:
#   1. Load final Tuned Extra Trees + Tuned Random Forest.
#   2. Keep the Phase 15.9 ensemble weights frozen.
#   3. Use VALIDATION only for XAI.
#   4. Use ensemble permutation importance for GLOBAL XAI.
#   5. Use SHAP PermutationExplainer for ONE local difficult case.
#   6. Save native tree importance for ET and RF.
#   7. Save plots + tables + JSON summary.
#
# IMPORTANT:
#   - TreeExplainer is NOT used.
#   - No model retraining.
#   - FINAL_HOLDOUT is NOT used.
#   - Global permutation progress is saved after EVERY feature.
#   - If runtime stops during global importance, rerunning this cell resumes
#     already-completed features.
#
# Global XAI:
#   500 validation rows
#   2 permutation repeats per feature
#
# Local SHAP:
#   1 difficult validation case
#   10-row background
#   1 full permutation cycle
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

warnings.filterwarnings("ignore")

# =============================================================================
# 1. BASE PATHS
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data/processed/delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_1_time_split_assignments.parquet"
)

ENSEMBLE_VALIDATION_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_optimized_ensemble_validation_predictions.parquet"
)

FINAL_CONFIG_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_final_ensemble_config.json"
)

FINAL_ET_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
)

FINAL_RF_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_random_forest_trainval.joblib"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_11_fast_xai"
)

PLOT_DIR = os.path.join(
    OUTPUT_DIR,
    "plots"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    PLOT_DIR,
    exist_ok=True
)

# =============================================================================
# 2. OUTPUT FILES
# =============================================================================

ENSEMBLE_PERMUTATION_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_ensemble_permutation_importance.csv"
)

PERMUTATION_PROGRESS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_permutation_progress.csv"
)

ET_NATIVE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_extra_trees_native_importance.csv"
)

RF_NATIVE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_random_forest_native_importance.csv"
)

GROUPED_IMPORTANCE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_grouped_feature_importance.csv"
)

LOCAL_SHAP_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_local_permutation_shap.csv"
)

LOCAL_CASE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_local_case.json"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_fast_xai_summary.json"
)

GLOBAL_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_ensemble_permutation_importance.png"
)

ET_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_extra_trees_native_importance.png"
)

RF_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_random_forest_native_importance.png"
)

LOCAL_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_local_ensemble_shap_waterfall.png"
)

# =============================================================================
# 3. VERIFY REQUIRED FILES
# =============================================================================

required_files = {
    "Feature engineered data": FEATURE_FILE,
    "Phase 15.1 split file": SPLIT_FILE,
    "Phase 15.9 validation predictions": ENSEMBLE_VALIDATION_FILE,
    "Phase 15.10 ensemble config": FINAL_CONFIG_FILE,
    "Final Tuned Extra Trees": FINAL_ET_MODEL,
    "Final Tuned Random Forest": FINAL_RF_MODEL
}

for label, path in required_files.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )

# =============================================================================
# 4. LOAD FINAL ENSEMBLE CONFIG
# =============================================================================

with open(
    FINAL_CONFIG_FILE,
    "r"
) as f:

    final_config = json.load(f)

MODEL_A = final_config[
    "component_models"
][0]["model"]

MODEL_B = final_config[
    "component_models"
][1]["model"]

WEIGHT_A = float(
    final_config[
        "component_models"
    ][0]["weight"]
)

WEIGHT_B = float(
    final_config[
        "component_models"
    ][1]["weight"]
)

if MODEL_A != "Tuned Extra Trees":

    raise ValueError(
        f"Unexpected Model A: {MODEL_A}"
    )

if MODEL_B != "Tuned Random Forest":

    raise ValueError(
        f"Unexpected Model B: {MODEL_B}"
    )

if not np.isclose(
    WEIGHT_A + WEIGHT_B,
    1.0,
    atol=1e-10
):

    raise ValueError(
        "Ensemble weights do not sum to 1."
    )

# =============================================================================
# 5. PROJECT HEADER
# =============================================================================

print("=" * 80)
print("PHASE 15.11 — FAST XAI / SHAP")
print("=" * 80)

print(
    f"\nModel A: {MODEL_A}"
)

print(
    f"Weight A: {WEIGHT_A:.6f}"
)

print(
    f"Model B: {MODEL_B}"
)

print(
    f"Weight B: {WEIGHT_B:.6f}"
)

print(
    "\nGlobal method: Ensemble permutation importance"
)

print(
    "Local method: SHAP PermutationExplainer"
)

print(
    "TreeExplainer: NOT USED"
)

print(
    "\nFINAL_HOLDOUT: FROZEN / UNTOUCHED"
)

# =============================================================================
# 6. EXACT 73-FEATURE CONTRACT
# =============================================================================

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

pollutant_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in pollutant_features
]

rolling_source_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2"
]

rolling_features = []

for feature in rolling_source_features:

    for window in [3, 12]:

        rolling_features.append(
            f"{feature}_rolling_mean_{window}h"
        )

        rolling_features.append(
            f"{feature}_rolling_std_{window}h"
        )

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

feature_cols = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + ["StationId"]
)

if len(feature_cols) != 73:

    raise ValueError(
        f"Expected 73 features, got {len(feature_cols)}"
    )

# Feature groups for reporting
FEATURE_GROUPS = {}

for feature in current_features:
    FEATURE_GROUPS[feature] = "Current"

for feature in lag_features:
    FEATURE_GROUPS[feature] = "Lag"

for feature in rolling_features:
    FEATURE_GROUPS[feature] = "Rolling"

for feature in calendar_features:
    FEATURE_GROUPS[feature] = "Calendar"

for feature in event_features:
    FEATURE_GROUPS[feature] = "Event"

FEATURE_GROUPS["StationId"] = "Station"

print(
    f"\nFeature contract verified: {len(feature_cols)} features"
)

# =============================================================================
# 7. LOAD DATA
# =============================================================================

print(
    "\nLoading feature and split data..."
)

df = pd.read_parquet(
    FEATURE_FILE
)

split_df = pd.read_parquet(
    SPLIT_FILE
)

validation_predictions = pd.read_parquet(
    ENSEMBLE_VALIDATION_FILE
)

DATETIME = "Datetime"
STATION = "StationId"
TARGET = "AQI_t_plus_1"

df[DATETIME] = pd.to_datetime(
    df[DATETIME],
    errors="coerce"
)

split_df[DATETIME] = pd.to_datetime(
    split_df[DATETIME],
    errors="coerce"
)

validation_predictions[DATETIME] = pd.to_datetime(
    validation_predictions[DATETIME],
    errors="coerce"
)

if df[DATETIME].isna().any():

    raise ValueError(
        "Invalid Datetime values in feature data."
    )

if split_df[DATETIME].isna().any():

    raise ValueError(
        "Invalid Datetime values in split data."
    )

# =============================================================================
# 8. ATTACH FROZEN SPLIT
# =============================================================================

split_keys = [
    STATION,
    DATETIME
]

if split_df.duplicated(
    split_keys
).any():

    raise ValueError(
        "Duplicate StationId-Datetime keys in split file."
    )

model_df = df.merge(
    split_df[
        split_keys + ["Split"]
    ],
    on=split_keys,
    how="left",
    validate="one_to_one"
)

if model_df["Split"].isna().any():

    raise ValueError(
        "Some rows have no split assignment."
    )

# =============================================================================
# 9. VALIDATION-ONLY DATA
# =============================================================================

validation_df = model_df[
    model_df["Split"].eq("VALIDATION")
].copy()

validation_df = validation_df[
    validation_df[TARGET].notna()
].copy()

if len(validation_df) != 69097:

    raise ValueError(
        f"Expected 69,097 validation rows, "
        f"found {len(validation_df):,}"
    )

print(
    f"\nValidation rows: {len(validation_df):,}"
)

# Explicit holdout safety check
if validation_df["Split"].eq(
    "FINAL_HOLDOUT"
).any():

    raise RuntimeError(
        "SAFETY FAILURE: FINAL_HOLDOUT entered XAI data."
    )

# =============================================================================
# 10. LOAD FINAL MODELS
# =============================================================================

print("\n" + "=" * 80)
print("LOADING FINAL COMPONENT MODELS")
print("=" * 80)

et_model = joblib.load(
    FINAL_ET_MODEL
)

rf_model = joblib.load(
    FINAL_RF_MODEL
)

print(
    f"ET type: {type(et_model)}"
)

print(
    f"RF type: {type(rf_model)}"
)

if not hasattr(
    et_model,
    "predict"
):

    raise TypeError(
        "Final ET model does not support predict()."
    )

if not hasattr(
    rf_model,
    "predict"
):

    raise TypeError(
        "Final RF model does not support predict()."
    )

# =============================================================================
# 11. ENSEMBLE PREDICTION FUNCTION
# =============================================================================

def ensemble_predict(X):

    et_prediction = np.asarray(
        et_model.predict(X),
        dtype=float
    ).reshape(-1)

    rf_prediction = np.asarray(
        rf_model.predict(X),
        dtype=float
    ).reshape(-1)

    return (
        WEIGHT_A * et_prediction
        + WEIGHT_B * rf_prediction
    )

# =============================================================================
# 12. FIND ONE DIFFICULT VALIDATION CASE
# =============================================================================
#
# Uses saved Phase 15.9 ensemble predictions.
# No holdout is involved.
# =============================================================================

required_validation_columns = [
    DATETIME,
    STATION,
    "Actual_AQI_t_plus_1",
    "Optimized_Ensemble_Prediction"
]

for column in required_validation_columns:

    if column not in validation_predictions.columns:

        raise KeyError(
            f"Missing validation prediction column: {column}"
        )

if len(validation_predictions) != 69097:

    raise ValueError(
        "Phase 15.9 validation prediction file has "
        f"{len(validation_predictions):,} rows instead of 69,097."
    )

validation_predictions["Absolute_Error"] = np.abs(
    validation_predictions["Actual_AQI_t_plus_1"]
    - validation_predictions["Optimized_Ensemble_Prediction"]
)

worst_case = validation_predictions.loc[
    validation_predictions["Absolute_Error"].idxmax()
].copy()

local_mask = (
    (validation_df[DATETIME] == worst_case[DATETIME])
    &
    (validation_df[STATION] == worst_case[STATION])
)

local_candidates = validation_df.loc[
    local_mask
].copy()

if len(local_candidates) != 1:

    raise ValueError(
        "Could not uniquely identify the local validation case."
    )

print("\n" + "=" * 80)
print("LOCAL XAI CASE")
print("=" * 80)

print(
    f"Datetime: {worst_case[DATETIME]}"
)

print(
    f"Station: {worst_case[STATION]}"
)

print(
    f"Actual AQI(t+1): "
    f"{float(worst_case['Actual_AQI_t_plus_1']):.6f}"
)

print(
    f"Ensemble prediction: "
    f"{float(worst_case['Optimized_Ensemble_Prediction']):.6f}"
)

print(
    f"Absolute error: "
    f"{float(worst_case['Absolute_Error']):.6f}"
)

# =============================================================================
# 13. GLOBAL XAI SAMPLE
# =============================================================================

GLOBAL_SAMPLE_SIZE = min(
    500,
    len(validation_df)
)

N_REPEATS = 2

global_sample = (
    validation_df
    .sample(
        n=GLOBAL_SAMPLE_SIZE,
        random_state=42
    )
    .reset_index(drop=True)
)

X_global = global_sample[
    feature_cols
].copy()

y_global = global_sample[TARGET].astype(float).to_numpy()

print("\n" + "=" * 80)
print("GLOBAL XAI SAMPLE")
print("=" * 80)

print(
    f"Global sample size: {GLOBAL_SAMPLE_SIZE}"
)

print(
    f"Permutation repeats: {N_REPEATS}"
)

# =============================================================================
# 14. BASELINE GLOBAL PREDICTION
# =============================================================================

global_baseline_prediction = ensemble_predict(
    X_global
)

baseline_mse = float(
    np.mean(
        (
            y_global
            - global_baseline_prediction
        ) ** 2
    )
)

print(
    f"Baseline MSE: {baseline_mse:.6f}"
)

# =============================================================================
# 15. RESUMABLE GLOBAL PERMUTATION IMPORTANCE
# =============================================================================

print("\n" + "=" * 80)
print("GLOBAL ENSEMBLE PERMUTATION IMPORTANCE")
print("=" * 80)

if os.path.exists(
    PERMUTATION_PROGRESS_FILE
):

    progress_df = pd.read_csv(
        PERMUTATION_PROGRESS_FILE
    )

    progress_df = progress_df.drop_duplicates(
        subset=["Feature"],
        keep="last"
    )

    completed_features = set(
        progress_df["Feature"].astype(str)
    )

    print(
        f"Existing completed features: "
        f"{len(completed_features)}/73"
    )

else:

    progress_df = pd.DataFrame(
        columns=[
            "Feature",
            "Feature_Group",
            "Permutation_MSE_Increase_Mean",
            "Permutation_MSE_Increase_STD",
            "Completed"
        ]
    )

    completed_features = set()

for feature_index, feature in enumerate(
    feature_cols
):

    if feature in completed_features:

        print(
            f"[{feature_index + 1:02d}/73] "
            f"SKIP {feature}"
        )

        continue

    repeat_importances = []

    for repeat_index in range(
        N_REPEATS
    ):

        # Deterministic seed for every feature/repeat pair.
        # This makes the result identical even if the cell is resumed.
        local_rng = np.random.default_rng(
            42
            + feature_index * 100
            + repeat_index
        )

        X_permuted = X_global.copy()

        values = X_permuted[
            feature
        ].to_numpy().copy()

        local_rng.shuffle(
            values
        )

        X_permuted[
            feature
        ] = values

        permuted_prediction = ensemble_predict(
            X_permuted
        )

        permuted_mse = float(
            np.mean(
                (
                    y_global
                    - permuted_prediction
                ) ** 2
            )
        )

        repeat_importances.append(
            permuted_mse
            - baseline_mse
        )

    new_row = pd.DataFrame([
        {
            "Feature": feature,
            "Feature_Group": FEATURE_GROUPS[feature],
            "Permutation_MSE_Increase_Mean":
                float(
                    np.mean(
                        repeat_importances
                    )
                ),
            "Permutation_MSE_Increase_STD":
                float(
                    np.std(
                        repeat_importances
                    )
                ),
            "Completed": True
        }
    ])

    progress_df = pd.concat(
        [
            progress_df,
            new_row
        ],
        ignore_index=True
    )

    progress_df = progress_df.drop_duplicates(
        subset=["Feature"],
        keep="last"
    )

    # SAVE AFTER EVERY FEATURE
    progress_df.to_csv(
        PERMUTATION_PROGRESS_FILE,
        index=False
    )

    print(
        f"[{feature_index + 1:02d}/73] "
        f"COMPLETED {feature}"
    )

# =============================================================================
# 16. FINALIZE GLOBAL PERMUTATION TABLE
# =============================================================================

if len(progress_df) != 73:

    raise RuntimeError(
        f"Global permutation analysis incomplete: "
        f"{len(progress_df)}/73 features."
    )

permutation_df = progress_df.copy()

permutation_df[
    "Permutation_MSE_Increase_Mean"
] = pd.to_numeric(
    permutation_df[
        "Permutation_MSE_Increase_Mean"
    ]
)

permutation_df[
    "Permutation_MSE_Increase_STD"
] = pd.to_numeric(
    permutation_df[
        "Permutation_MSE_Increase_STD"
    ]
)

positive_importance = permutation_df[
    "Permutation_MSE_Increase_Mean"
].clip(
    lower=0
)

if positive_importance.sum() > 0:

    permutation_df[
        "Relative_Importance_%"
    ] = (
        positive_importance
        / positive_importance.sum()
        * 100
    )

else:

    permutation_df[
        "Relative_Importance_%"
    ] = 0.0

permutation_df = permutation_df.sort_values(
    "Permutation_MSE_Increase_Mean",
    ascending=False
).reset_index(
    drop=True
)

permutation_df.to_csv(
    ENSEMBLE_PERMUTATION_FILE,
    index=False
)

print("\nTop ensemble permutation features:")

print(
    permutation_df.head(20).to_string(
        index=False
    )
)

# =============================================================================
# 17. NATIVE TREE IMPORTANCE
# =============================================================================

print("\n" + "=" * 80)
print("NATIVE TREE FEATURE IMPORTANCE")
print("=" * 80)

def get_final_estimator(
    model
):

    if hasattr(
        model,
        "steps"
    ):

        return model.steps[-1][1]

    return model

def get_transformed_feature_names(
    model
):

    if hasattr(
        model,
        "steps"
    ):

        preprocessor = model[:-1]

        try:

            names = preprocessor.get_feature_names_out(
                feature_cols
            )

        except Exception:

            names = preprocessor.get_feature_names_out()

        return [
            str(name)
            for name in names
        ]

    return list(feature_cols)

def map_original_feature(
    transformed_name
):

    name = str(
        transformed_name
    )

    if "__" in name:

        cleaned = name.split(
            "__",
            1
        )[1]

    else:

        cleaned = name

    if cleaned.startswith(
        "StationId_"
    ):

        return "StationId"

    for feature in sorted(
        feature_cols,
        key=len,
        reverse=True
    ):

        if cleaned == feature:

            return feature

        if cleaned.startswith(
            feature + "_"
        ):

            return feature

    return cleaned

et_estimator = get_final_estimator(
    et_model
)

rf_estimator = get_final_estimator(
    rf_model
)

if not hasattr(
    et_estimator,
    "feature_importances_"
):

    raise TypeError(
        "ET estimator has no feature_importances_."
    )

if not hasattr(
    rf_estimator,
    "feature_importances_"
):

    raise TypeError(
        "RF estimator has no feature_importances_."
    )

transformed_names = (
    get_transformed_feature_names(
        et_model
    )
)

et_native_importance = np.asarray(
    et_estimator.feature_importances_,
    dtype=float
)

rf_native_importance = np.asarray(
    rf_estimator.feature_importances_,
    dtype=float
)

if len(transformed_names) != len(
    et_native_importance
):

    raise ValueError(
        "ET transformed feature count does not match native importance."
    )

if len(transformed_names) != len(
    rf_native_importance
):

    raise ValueError(
        "RF transformed feature count does not match native importance."
    )

# ET native importance
et_native_df = pd.DataFrame(
    {
        "Transformed_Feature":
            transformed_names,
        "Original_Feature":
            [
                map_original_feature(name)
                for name in transformed_names
            ],
        "Native_Importance":
            et_native_importance
    }
)

et_native_df[
    "Relative_Importance_%"
] = (
    et_native_df["Native_Importance"]
    / et_native_df["Native_Importance"].sum()
    * 100
)

et_native_df = et_native_df.sort_values(
    "Native_Importance",
    ascending=False
).reset_index(
    drop=True
)

et_native_df.to_csv(
    ET_NATIVE_FILE,
    index=False
)

# RF native importance
rf_native_df = pd.DataFrame(
    {
        "Transformed_Feature":
            transformed_names,
        "Original_Feature":
            [
                map_original_feature(name)
                for name in transformed_names
            ],
        "Native_Importance":
            rf_native_importance
    }
)

rf_native_df[
    "Relative_Importance_%"
] = (
    rf_native_df["Native_Importance"]
    / rf_native_df["Native_Importance"].sum()
    * 100
)

rf_native_df = rf_native_df.sort_values(
    "Native_Importance",
    ascending=False
).reset_index(
    drop=True
)

rf_native_df.to_csv(
    RF_NATIVE_FILE,
    index=False
)

print("\nTop Extra Trees native features:")

print(
    et_native_df.head(15).to_string(
        index=False
    )
)

print("\nTop Random Forest native features:")

print(
    rf_native_df.head(15).to_string(
        index=False
    )
)

# =============================================================================
# 18. WEIGHTED NATIVE IMPORTANCE AT ORIGINAL FEATURE LEVEL
# =============================================================================

weighted_native_df = pd.DataFrame(
    {
        "Original_Feature":
            [
                map_original_feature(name)
                for name in transformed_names
            ],
        "Weighted_Native_Importance":
            (
                WEIGHT_A * et_native_importance
                + WEIGHT_B * rf_native_importance
            )
    }
)

weighted_native_df = (
    weighted_native_df
    .groupby(
        "Original_Feature",
        as_index=False
    )[
        "Weighted_Native_Importance"
    ]
    .sum()
)

weighted_native_df[
    "Native_Relative_Importance_%"
] = (
    weighted_native_df[
        "Weighted_Native_Importance"
    ]
    / weighted_native_df[
        "Weighted_Native_Importance"
    ].sum()
    * 100
)

# =============================================================================
# 19. GROUPED IMPORTANCE
# =============================================================================

grouped_df = permutation_df[
    [
        "Feature",
        "Feature_Group",
        "Permutation_MSE_Increase_Mean",
        "Permutation_MSE_Increase_STD",
        "Relative_Importance_%"
    ]
].rename(
    columns={
        "Feature":
            "Original_Feature"
    }
)

grouped_df = grouped_df.merge(
    weighted_native_df,
    on="Original_Feature",
    how="outer"
)

grouped_df.to_csv(
    GROUPED_IMPORTANCE_FILE,
    index=False
)

# =============================================================================
# 20. GLOBAL PERMUTATION IMPORTANCE PLOT
# =============================================================================

top_global = (
    permutation_df
    .head(20)
    .sort_values(
        "Permutation_MSE_Increase_Mean"
    )
)

plt.figure(
    figsize=(10, 8)
)

plt.barh(
    range(len(top_global)),
    top_global[
        "Permutation_MSE_Increase_Mean"
    ]
)

plt.yticks(
    range(len(top_global)),
    top_global["Feature"]
)

plt.xlabel(
    "Increase in validation MSE after permutation"
)

plt.title(
    "Phase 15.11 — Final Ensemble Permutation Importance"
)

plt.tight_layout()

plt.savefig(
    GLOBAL_PLOT_FILE,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# =============================================================================
# 21. NATIVE IMPORTANCE PLOTS
# =============================================================================

def save_native_plot(
    native_df,
    title,
    output_path
):

    top = (
        native_df
        .head(20)
        .sort_values(
            "Native_Importance"
        )
    )

    plt.figure(
        figsize=(10, 8)
    )

    plt.barh(
        range(len(top)),
        top["Native_Importance"]
    )

    plt.yticks(
        range(len(top)),
        top["Original_Feature"]
    )

    plt.xlabel(
        "Native tree importance"
    )

    plt.title(
        title
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

save_native_plot(
    et_native_df,
    "Phase 15.11 — Tuned Extra Trees Native Importance",
    ET_PLOT_FILE
)

save_native_plot(
    rf_native_df,
    "Phase 15.11 — Tuned Random Forest Native Importance",
    RF_PLOT_FILE
)

# =============================================================================
# 22. LOCAL SHAP PERMUTATION EXPLAINER
# =============================================================================

print("\n" + "=" * 80)
print("LOCAL SHAP — PERMUTATION EXPLAINER")
print("=" * 80)

BACKGROUND_SIZE = min(
    10,
    len(validation_df)
)

background_df = (
    validation_df
    .sample(
        n=BACKGROUND_SIZE,
        random_state=42
    )
    .reset_index(drop=True)
)

X_background = background_df[
    feature_cols
].copy()

X_local = local_candidates[
    feature_cols
].copy()

background_prediction = ensemble_predict(
    X_background
)

local_prediction = float(
    ensemble_predict(
        X_local
    )[0]
)

local_base_value = float(
    np.mean(
        background_prediction
    )
)

print(
    f"Background rows: {BACKGROUND_SIZE}"
)

print(
    "Creating SHAP PermutationExplainer..."
)

shap_explainer = shap.PermutationExplainer(
    ensemble_predict,
    X_background,
    feature_names=feature_cols,
    seed=42
)

print(
    "Explainer created."
)

print(
    "Calculating SHAP for ONE validation case..."
)

local_result = shap_explainer.shap_values(
    X_local,
    npermutations=1,
    main_effects=False,
    error_bounds=False,
    batch_evals=True,
    silent=False
)

# SHAP versions can return either an array or an Explanation.
if hasattr(
    local_result,
    "values"
):

    local_values = np.asarray(
        local_result.values,
        dtype=float
    )

else:

    local_values = np.asarray(
        local_result,
        dtype=float
    )

if local_values.ndim == 3:

    local_values = local_values[:, :, 0]

if local_values.ndim != 2:

    raise ValueError(
        f"Unexpected local SHAP shape: "
        f"{local_values.shape}"
    )

if local_values.shape[0] != 1:

    raise ValueError(
        f"Expected one local explanation row, "
        f"got {local_values.shape[0]}"
    )

local_values = local_values[0]

local_reconstructed = (
    local_base_value
    + float(
        local_values.sum()
    )
)

local_additivity_error = float(
    abs(
        local_prediction
        - local_reconstructed
    )
)

# =============================================================================
# 23. LOCAL SHAP TABLE
# =============================================================================

local_shap_df = pd.DataFrame(
    {
        "Feature":
            feature_cols,
        "Feature_Value":
            X_local.iloc[0].values,
        "SHAP_Value":
            local_values
    }
)

local_shap_df[
    "Absolute_SHAP"
] = np.abs(
    local_shap_df[
        "SHAP_Value"
    ]
)

local_shap_df[
    "Feature_Group"
] = local_shap_df[
    "Feature"
].map(
    FEATURE_GROUPS
)

local_shap_df = local_shap_df.sort_values(
    "Absolute_SHAP",
    ascending=False
).reset_index(
    drop=True
)

local_shap_df.to_csv(
    LOCAL_SHAP_FILE,
    index=False
)

# =============================================================================
# 24. LOCAL WATERFALL PLOT
# =============================================================================

local_explanation = shap.Explanation(
    values=local_values,
    base_values=local_base_value,
    data=X_local.iloc[0].to_numpy(),
    feature_names=feature_cols
)

shap.plots.waterfall(
    local_explanation,
    max_display=15,
    show=False
)

plt.title(
    "Phase 15.11 — Local Ensemble SHAP Explanation"
)

plt.tight_layout()

plt.savefig(
    LOCAL_PLOT_FILE,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# =============================================================================
# 25. SAVE LOCAL CASE METADATA
# =============================================================================

local_case = {
    "Datetime": str(
        worst_case[DATETIME]
    ),

    "StationId": str(
        worst_case[STATION]
    ),

    "Actual_AQI_t_plus_1": float(
        worst_case[
            "Actual_AQI_t_plus_1"
        ]
    ),

    "Ensemble_Prediction": local_prediction,

    "Absolute_Error": float(
        worst_case[
            "Absolute_Error"
        ]
    ),

    "SHAP_Base_Value": local_base_value,

    "SHAP_Reconstructed_Prediction":
        local_reconstructed,

    "SHAP_Additivity_Error":
        local_additivity_error,

    "background_rows":
        BACKGROUND_SIZE,

    "npermutations":
        1,

    "final_holdout_used":
        False
}

with open(
    LOCAL_CASE_FILE,
    "w"
) as f:

    json.dump(
        local_case,
        f,
        indent=2
    )

# =============================================================================
# 26. DISPLAY LOCAL SHAP RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("LOCAL SHAP RESULT")
print("=" * 80)

print(
    f"Station: "
    f"{local_case['StationId']}"
)

print(
    f"Datetime: "
    f"{local_case['Datetime']}"
)

print(
    f"Actual AQI(t+1): "
    f"{local_case['Actual_AQI_t_plus_1']:.6f}"
)

print(
    f"Ensemble prediction: "
    f"{local_case['Ensemble_Prediction']:.6f}"
)

print(
    f"Absolute error: "
    f"{local_case['Absolute_Error']:.6f}"
)

print(
    f"SHAP base value: "
    f"{local_base_value:.6f}"
)

print(
    f"SHAP reconstructed prediction: "
    f"{local_reconstructed:.6f}"
)

print(
    f"SHAP additivity error: "
    f"{local_additivity_error:.12f}"
)

print(
    "\nTop local SHAP contributors:"
)

print(
    local_shap_df.head(20).to_string(
        index=False
    )
)

# =============================================================================
# 27. SAVE FINAL XAI SUMMARY
# =============================================================================

summary = {

    "phase": "15.11",

    "method":
        "Fast model-agnostic XAI",

    "tree_shap_used":
        False,

    "global_method": {
        "type":
            "Ensemble permutation feature importance",
        "sample_size":
            GLOBAL_SAMPLE_SIZE,
        "repeats_per_feature":
            N_REPEATS,
        "random_state":
            42,
        "features_completed":
            73
    },

    "local_method": {
        "type":
            "SHAP PermutationExplainer",
        "background_rows":
            BACKGROUND_SIZE,
        "npermutations":
            1,
        "cases":
            1,
        "random_state":
            42
    },

    "ensemble": {
        "model_a":
            MODEL_A,
        "weight_a":
            WEIGHT_A,
        "model_b":
            MODEL_B,
        "weight_b":
            WEIGHT_B
    },

    "validation": {
        "rows":
            69097,
        "final_holdout_used":
            False
    },

    "local_case":
        local_case,

    "top_global_features":
        permutation_df.head(15)[
            [
                "Feature",
                "Feature_Group",
                "Permutation_MSE_Increase_Mean",
                "Relative_Importance_%"
            ]
        ].to_dict(
            orient="records"
        ),

    "top_local_features":
        local_shap_df.head(15)[
            [
                "Feature",
                "Feature_Group",
                "Feature_Value",
                "SHAP_Value"
            ]
        ].to_dict(
            orient="records"
        ),

    "files": {
        "ensemble_permutation":
            ENSEMBLE_PERMUTATION_FILE,
        "permutation_progress":
            PERMUTATION_PROGRESS_FILE,
        "et_native":
            ET_NATIVE_FILE,
        "rf_native":
            RF_NATIVE_FILE,
        "grouped":
            GROUPED_IMPORTANCE_FILE,
        "local_shap":
            LOCAL_SHAP_FILE,
        "local_case":
            LOCAL_CASE_FILE,
        "global_plot":
            GLOBAL_PLOT_FILE,
        "et_plot":
            ET_PLOT_FILE,
        "rf_plot":
            RF_PLOT_FILE,
        "local_plot":
            LOCAL_PLOT_FILE,
        "summary":
            SUMMARY_FILE
    },

    "final_holdout_status":
        "FROZEN / UNTOUCHED",

    "next_phase":
        "15.12 — Final Holdout Evaluation"
}

with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# 28. VERIFY ALL OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.11 — OUTPUT VERIFICATION")
print("=" * 80)

output_files = {
    "Ensemble permutation importance":
        ENSEMBLE_PERMUTATION_FILE,

    "Permutation progress":
        PERMUTATION_PROGRESS_FILE,

    "ET native importance":
        ET_NATIVE_FILE,

    "RF native importance":
        RF_NATIVE_FILE,

    "Grouped importance":
        GROUPED_IMPORTANCE_FILE,

    "Local SHAP":
        LOCAL_SHAP_FILE,

    "Local case":
        LOCAL_CASE_FILE,

    "Global plot":
        GLOBAL_PLOT_FILE,

    "ET plot":
        ET_PLOT_FILE,

    "RF plot":
        RF_PLOT_FILE,

    "Local waterfall":
        LOCAL_PLOT_FILE,

    "XAI summary":
        SUMMARY_FILE
}

verification_rows = []

for name, path in output_files.items():

    verification_rows.append(
        {
            "Artifact":
                name,

            "Status":
                "FOUND"
                if os.path.exists(path)
                else "MISSING"
        }
    )

verification_df = pd.DataFrame(
    verification_rows
)

print(
    verification_df.to_string(
        index=False
    )
)

if (
    verification_df["Status"]
    == "MISSING"
).any():

    raise RuntimeError(
        "One or more Phase 15.11 outputs are missing."
    )

# =============================================================================
# 29. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.11 — FINAL SUMMARY")
print("=" * 80)

print(
    f"Global XAI sample: "
    f"{GLOBAL_SAMPLE_SIZE}"
)

print(
    f"Permutation repeats: "
    f"{N_REPEATS}"
)

print(
    "Local SHAP cases: 1"
)

print(
    f"Local SHAP additivity error: "
    f"{local_additivity_error:.12f}"
)

print(
    f"\nTop global feature: "
    f"{permutation_df.iloc[0]['Feature']}"
)

print(
    f"Top global feature group: "
    f"{permutation_df.iloc[0]['Feature_Group']}"
)

print(
    f"Top global relative importance: "
    f"{permutation_df.iloc[0]['Relative_Importance_%']:.4f}%"
)

print(
    "\nFINAL_HOLDOUT: FROZEN / UNTOUCHED"
)

print(
    f"\nXAI output directory:\n{OUTPUT_DIR}"
)

print("\n" + "=" * 80)
print("PHASE 15.11 COMPLETED")
print("FINAL HOLDOUT: FROZEN / UNTOUCHED")
print("NEXT: PHASE 15.12 — FINAL HOLDOUT EVALUATION")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.11 — FIX LOCAL SHAP EXPLANATION
# =============================================================================
#
# The global permutation importance already completed successfully.
# This cell ONLY fixes and completes the local SHAP portion.
#
# Root cause:
#   StationId is a string categorical feature.
#   SHAP's tabular masker cannot perform numeric comparisons on raw strings.
#
# Fix:
#   Transform the raw features through the saved preprocessing pipeline first.
#   SHAP is then applied to the resulting numeric 82-feature representation.
#
# FINAL_HOLDOUT remains untouched.
# =============================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.pipeline import Pipeline

# =============================================================================
# 1. PATHS
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data/processed/delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_1_time_split_assignments.parquet"
)

VALIDATION_PREDICTIONS_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_9_optimized_ensemble_validation_predictions.parquet"
)

FINAL_CONFIG_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_final_ensemble_config.json"
)

FINAL_ET_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
)

FINAL_RF_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_random_forest_trainval.joblib"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_11_fast_xai"
)

PLOT_DIR = os.path.join(
    OUTPUT_DIR,
    "plots"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    PLOT_DIR,
    exist_ok=True
)

LOCAL_SHAP_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_local_permutation_shap.csv"
)

LOCAL_CASE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_local_case.json"
)

LOCAL_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_local_ensemble_shap_waterfall.png"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_fast_xai_summary.json"
)

GLOBAL_PERMUTATION_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_ensemble_permutation_importance.csv"
)

ET_NATIVE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_extra_trees_native_importance.csv"
)

RF_NATIVE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_random_forest_native_importance.csv"
)

GROUPED_IMPORTANCE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_grouped_feature_importance.csv"
)

GLOBAL_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_ensemble_permutation_importance.png"
)

ET_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_extra_trees_native_importance.png"
)

RF_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_random_forest_native_importance.png"
)

# =============================================================================
# 2. LOAD ENSEMBLE CONFIG
# =============================================================================

with open(
    FINAL_CONFIG_FILE,
    "r"
) as f:

    config = json.load(f)

MODEL_A = config[
    "component_models"
][0]["model"]

MODEL_B = config[
    "component_models"
][1]["model"]

WEIGHT_A = float(
    config[
        "component_models"
    ][0]["weight"]
)

WEIGHT_B = float(
    config[
        "component_models"
    ][1]["weight"]
)

if MODEL_A != "Tuned Extra Trees":
    raise ValueError(
        f"Unexpected Model A: {MODEL_A}"
    )

if MODEL_B != "Tuned Random Forest":
    raise ValueError(
        f"Unexpected Model B: {MODEL_B}"
    )

# =============================================================================
# 3. EXACT FEATURE CONTRACT
# =============================================================================

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

pollutant_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in pollutant_features
]

rolling_source_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2"
]

rolling_features = []

for feature in rolling_source_features:

    for window in [3, 12]:

        rolling_features.append(
            f"{feature}_rolling_mean_{window}h"
        )

        rolling_features.append(
            f"{feature}_rolling_std_{window}h"
        )

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

feature_cols = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + ["StationId"]
)

if len(feature_cols) != 73:

    raise ValueError(
        f"Expected 73 features, got {len(feature_cols)}"
    )

TARGET = "AQI_t_plus_1"
DATETIME = "Datetime"
STATION = "StationId"

print("=" * 80)
print("PHASE 15.11 — LOCAL SHAP FIX")
print("=" * 80)

print(
    f"\nModel A: {MODEL_A}"
)

print(
    f"Weight A: {WEIGHT_A:.6f}"
)

print(
    f"Model B: {MODEL_B}"
)

print(
    f"Weight B: {WEIGHT_B:.6f}"
)

print(
    "\nFix: SHAP will use the numeric transformed feature space."
)

print(
    "TreeExplainer: NOT USED"
)

print(
    "\nFINAL_HOLDOUT: FROZEN / UNTOUCHED"
)

# =============================================================================
# 4. LOAD MODELS
# =============================================================================

print("\n" + "=" * 80)
print("LOADING FINAL MODELS")
print("=" * 80)

et_model = joblib.load(
    FINAL_ET_MODEL
)

rf_model = joblib.load(
    FINAL_RF_MODEL
)

print(
    f"ET: {type(et_model)}"
)

print(
    f"RF: {type(rf_model)}"
)

# =============================================================================
# 5. LOAD DATA
# =============================================================================

print("\nLoading feature and split data...")

df = pd.read_parquet(
    FEATURE_FILE
)

split_df = pd.read_parquet(
    SPLIT_FILE
)

validation_predictions = pd.read_parquet(
    VALIDATION_PREDICTIONS_FILE
)

df[DATETIME] = pd.to_datetime(
    df[DATETIME],
    errors="coerce"
)

split_df[DATETIME] = pd.to_datetime(
    split_df[DATETIME],
    errors="coerce"
)

validation_predictions[DATETIME] = pd.to_datetime(
    validation_predictions[DATETIME],
    errors="coerce"
)

# =============================================================================
# 6. ATTACH SPLIT
# =============================================================================

split_keys = [
    STATION,
    DATETIME
]

if split_df.duplicated(
    split_keys
).any():

    raise ValueError(
        "Duplicate StationId-Datetime keys in split file."
    )

model_df = df.merge(
    split_df[
        split_keys + ["Split"]
    ],
    on=split_keys,
    how="left",
    validate="one_to_one"
)

if model_df["Split"].isna().any():

    raise ValueError(
        "Some rows have no split assignment."
    )

# =============================================================================
# 7. VALIDATION ONLY
# =============================================================================

validation_df = model_df[
    model_df["Split"].eq("VALIDATION")
].copy()

validation_df = validation_df[
    validation_df[TARGET].notna()
].copy()

if len(validation_df) != 69097:

    raise ValueError(
        f"Expected 69,097 validation rows, "
        f"found {len(validation_df):,}"
    )

print(
    f"\nValidation rows: {len(validation_df):,}"
)

# =============================================================================
# 8. IDENTIFY LARGEST ERROR CASE
# =============================================================================

required_cols = [
    DATETIME,
    STATION,
    "Actual_AQI_t_plus_1",
    "Optimized_Ensemble_Prediction"
]

for col in required_cols:

    if col not in validation_predictions.columns:

        raise KeyError(
            f"Missing column: {col}"
        )

validation_predictions[
    "Absolute_Error"
] = np.abs(
    validation_predictions[
        "Actual_AQI_t_plus_1"
    ]
    - validation_predictions[
        "Optimized_Ensemble_Prediction"
    ]
)

worst_case = validation_predictions.loc[
    validation_predictions[
        "Absolute_Error"
    ].idxmax()
].copy()

print("\n" + "=" * 80)
print("LOCAL XAI CASE")
print("=" * 80)

print(
    f"Datetime: {worst_case[DATETIME]}"
)

print(
    f"Station: {worst_case[STATION]}"
)

print(
    f"Actual AQI(t+1): "
    f"{float(worst_case['Actual_AQI_t_plus_1']):.6f}"
)

print(
    f"Saved ensemble prediction: "
    f"{float(worst_case['Optimized_Ensemble_Prediction']):.6f}"
)

print(
    f"Absolute error: "
    f"{float(worst_case['Absolute_Error']):.6f}"
)

# =============================================================================
# 9. LOCATE LOCAL RAW ROW
# =============================================================================

local_mask = (
    (validation_df[DATETIME] == worst_case[DATETIME])
    &
    (validation_df[STATION] == worst_case[STATION])
)

local_candidates = validation_df.loc[
    local_mask
].copy()

if len(local_candidates) != 1:

    raise ValueError(
        "Could not uniquely identify local validation case."
    )

X_local_raw = local_candidates[
    feature_cols
].copy()

# =============================================================================
# 10. EXTRACT PREPROCESSORS + ESTIMATORS
# =============================================================================

if not isinstance(
    et_model,
    Pipeline
):

    raise TypeError(
        "Expected final ET model to be an sklearn Pipeline."
    )

if not isinstance(
    rf_model,
    Pipeline
):

    raise TypeError(
        "Expected final RF model to be an sklearn Pipeline."
    )

et_preprocessor = et_model[:-1]
rf_preprocessor = rf_model[:-1]

et_estimator = et_model.steps[-1][1]
rf_estimator = rf_model.steps[-1][1]

# =============================================================================
# 11. GET TRANSFORMED NUMERIC REPRESENTATION
# =============================================================================

print("\n" + "=" * 80)
print("TRANSFORMING LOCAL SHAP INPUT")
print("=" * 80)

# Use ET preprocessing because Phase 15.11 already verified
# ET/RF preprocessing produces identical 82-feature input.

X_local_transformed = (
    et_preprocessor.transform(
        X_local_raw
    )
)

if hasattr(
    X_local_transformed,
    "toarray"
):

    X_local_transformed = (
        X_local_transformed.toarray()
    )

X_local_transformed = np.asarray(
    X_local_transformed,
    dtype=float
)

# =============================================================================
# 12. GET TRANSFORMED FEATURE NAMES
# =============================================================================

try:

    transformed_feature_names = [
        str(name)
        for name in
        et_preprocessor.get_feature_names_out(
            feature_cols
        )
    ]

except Exception:

    transformed_feature_names = [
        str(name)
        for name in
        et_preprocessor.get_feature_names_out()
    ]

if X_local_transformed.shape[1] != len(
    transformed_feature_names
):

    raise ValueError(
        "Transformed feature count does not match feature names."
    )

print(
    f"Raw feature count: "
    f"{len(feature_cols)}"
)

print(
    f"Numeric transformed feature count: "
    f"{X_local_transformed.shape[1]}"
)

# =============================================================================
# 13. BUILD LOCAL BACKGROUND
# =============================================================================

BACKGROUND_SIZE = 10

background_df = (
    validation_df
    .sample(
        n=BACKGROUND_SIZE,
        random_state=42
    )
    .reset_index(drop=True)
)

X_background_raw = background_df[
    feature_cols
].copy()

X_background_transformed = (
    et_preprocessor.transform(
        X_background_raw
    )
)

if hasattr(
    X_background_transformed,
    "toarray"
):

    X_background_transformed = (
        X_background_transformed.toarray()
    )

X_background_transformed = np.asarray(
    X_background_transformed,
    dtype=float
)

print(
    f"Background rows: "
    f"{X_background_transformed.shape[0]}"
)

# =============================================================================
# 14. VERIFY LOCAL DATA IS NUMERIC
# =============================================================================

if not np.isfinite(
    X_local_transformed
).all():

    raise ValueError(
        "Local transformed SHAP input contains NaN/inf."
    )

if not np.isfinite(
    X_background_transformed
).all():

    raise ValueError(
        "Background transformed SHAP input contains NaN/inf."
    )

# =============================================================================
# 15. TRANSFORMED ENSEMBLE PREDICT FUNCTION
# =============================================================================
#
# SHAP will now pass numeric arrays to this function.
#
# No strings are seen by SHAP.
# =============================================================================

def transformed_ensemble_predict(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    et_prediction = np.asarray(
        et_estimator.predict(X),
        dtype=float
    ).reshape(-1)

    rf_prediction = np.asarray(
        rf_estimator.predict(X),
        dtype=float
    ).reshape(-1)

    return (
        WEIGHT_A * et_prediction
        + WEIGHT_B * rf_prediction
    )

# =============================================================================
# 16. CREATE SHAP PERMUTATION EXPLAINER
# =============================================================================

print("\n" + "=" * 80)
print("CREATING SHAP PERMUTATION EXPLAINER")
print("=" * 80)

shap_explainer = shap.PermutationExplainer(
    transformed_ensemble_predict,
    X_background_transformed,
    feature_names=transformed_feature_names,
    seed=42
)

print(
    "PermutationExplainer created successfully."
)

# =============================================================================
# 17. CALCULATE LOCAL SHAP
# =============================================================================

print(
    "\nCalculating SHAP for ONE validation case..."
)

local_explanation = shap_explainer(
    X_local_transformed,
    max_evals=(
        2 * X_local_transformed.shape[1]
        + 1
    ),
    silent=False
)

local_shap_values = np.asarray(
    local_explanation.values,
    dtype=float
)

if local_shap_values.ndim == 3:

    local_shap_values = (
        local_shap_values[:, :, 0]
    )

if local_shap_values.ndim != 2:

    raise ValueError(
        f"Unexpected SHAP shape: "
        f"{local_shap_values.shape}"
    )

local_shap_values = (
    local_shap_values[0]
)

# =============================================================================
# 18. GET SHAP BASE VALUE
# =============================================================================

base_values = np.asarray(
    local_explanation.base_values,
    dtype=float
)

local_base_value = float(
    base_values.reshape(-1)[0]
)

# =============================================================================
# 19. PREDICTION + ADDITIVITY
# =============================================================================

local_prediction = float(
    transformed_ensemble_predict(
        X_local_transformed
    )[0]
)

local_reconstructed = (
    local_base_value
    + float(
        local_shap_values.sum()
    )
)

local_additivity_error = float(
    abs(
        local_prediction
        - local_reconstructed
    )
)

print("\n" + "=" * 80)
print("LOCAL SHAP ADDITIVITY CHECK")
print("=" * 80)

print(
    f"Model prediction: "
    f"{local_prediction:.6f}"
)

print(
    f"SHAP reconstruction: "
    f"{local_reconstructed:.6f}"
)

print(
    f"Additivity error: "
    f"{local_additivity_error:.12f}"
)

# =============================================================================
# 20. LOCAL SHAP TABLE
# =============================================================================

local_shap_df = pd.DataFrame(
    {
        "Transformed_Feature":
            transformed_feature_names,

        "Feature_Value":
            X_local_transformed[0],

        "SHAP_Value":
            local_shap_values
    }
)

local_shap_df[
    "Absolute_SHAP"
] = np.abs(
    local_shap_df[
        "SHAP_Value"
    ]
)

# Map transformed names back to original features
def map_original_feature(
    name
):

    name = str(name)

    if "__" in name:

        cleaned = name.split(
            "__",
            1
        )[1]

    else:

        cleaned = name

    if cleaned.startswith(
        "StationId_"
    ):

        return "StationId"

    for feature in sorted(
        feature_cols,
        key=len,
        reverse=True
    ):

        if cleaned == feature:

            return feature

        if cleaned.startswith(
            feature + "_"
        ):

            return feature

    return cleaned

local_shap_df[
    "Original_Feature"
] = local_shap_df[
    "Transformed_Feature"
].apply(
    map_original_feature
)

feature_group_map = {}

for feature in current_features:
    feature_group_map[feature] = "Current"

for feature in lag_features:
    feature_group_map[feature] = "Lag"

for feature in rolling_features:
    feature_group_map[feature] = "Rolling"

for feature in calendar_features:
    feature_group_map[feature] = "Calendar"

for feature in event_features:
    feature_group_map[feature] = "Event"

feature_group_map["StationId"] = "Station"

local_shap_df[
    "Feature_Group"
] = local_shap_df[
    "Original_Feature"
].map(
    feature_group_map
)

local_shap_df = local_shap_df.sort_values(
    "Absolute_SHAP",
    ascending=False
).reset_index(
    drop=True
)

local_shap_df.to_csv(
    LOCAL_SHAP_FILE,
    index=False
)

# =============================================================================
# 21. LOCAL WATERFALL
# =============================================================================

print(
    "\nCreating local SHAP waterfall..."
)

local_plot_explanation = shap.Explanation(
    values=local_shap_values,
    base_values=local_base_value,
    data=X_local_transformed[0],
    feature_names=transformed_feature_names
)

shap.plots.waterfall(
    local_plot_explanation,
    max_display=15,
    show=False
)

plt.title(
    "Phase 15.11 — Local Ensemble SHAP Explanation"
)

plt.tight_layout()

plt.savefig(
    LOCAL_PLOT_FILE,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# =============================================================================
# 22. SAVE LOCAL CASE METADATA
# =============================================================================

local_case = {
    "Datetime": str(
        worst_case[DATETIME]
    ),

    "StationId": str(
        worst_case[STATION]
    ),

    "Actual_AQI_t_plus_1": float(
        worst_case[
            "Actual_AQI_t_plus_1"
        ]
    ),

    "Ensemble_Prediction": local_prediction,

    "Absolute_Error": float(
        worst_case[
            "Absolute_Error"
        ]
    ),

    "SHAP_Base_Value": local_base_value,

    "SHAP_Reconstructed_Prediction":
        local_reconstructed,

    "SHAP_Additivity_Error":
        local_additivity_error,

    "Raw_Feature_Count":
        len(feature_cols),

    "Transformed_Feature_Count":
        len(transformed_feature_names),

    "Background_Rows":
        BACKGROUND_SIZE,

    "SHAP_Method":
        "PermutationExplainer",

    "Final_Holdout_Used":
        False
}

with open(
    LOCAL_CASE_FILE,
    "w"
) as f:

    json.dump(
        local_case,
        f,
        indent=2
    )

# =============================================================================
# 23. DISPLAY LOCAL RESULT
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.11 — LOCAL SHAP RESULT")
print("=" * 80)

print(
    f"Station: "
    f"{local_case['StationId']}"
)

print(
    f"Datetime: "
    f"{local_case['Datetime']}"
)

print(
    f"Actual AQI(t+1): "
    f"{local_case['Actual_AQI_t_plus_1']:.6f}"
)

print(
    f"Ensemble prediction: "
    f"{local_case['Ensemble_Prediction']:.6f}"
)

print(
    f"Absolute error: "
    f"{local_case['Absolute_Error']:.6f}"
)

print(
    f"SHAP base value: "
    f"{local_base_value:.6f}"
)

print(
    f"SHAP reconstructed prediction: "
    f"{local_reconstructed:.6f}"
)

print(
    f"SHAP additivity error: "
    f"{local_additivity_error:.12f}"
)

print(
    "\nTop local SHAP contributors:"
)

print(
    local_shap_df.head(20).to_string(
        index=False
    )
)

# =============================================================================
# 24. UPDATE XAI SUMMARY
# =============================================================================
#
# Existing global permutation results are already saved.
# We now create/update the final summary using those results.
# =============================================================================

if os.path.exists(
    GLOBAL_PERMUTATION_FILE
):

    global_perm_df = pd.read_csv(
        GLOBAL_PERMUTATION_FILE
    )

    top_global_features = (
        global_perm_df
        .head(15)[
            [
                "Feature",
                "Feature_Group",
                "Permutation_MSE_Increase_Mean",
                "Relative_Importance_%"
            ]
        ]
        .to_dict(
            orient="records"
        )
    )

else:

    top_global_features = []

summary = {

    "phase":
        "15.11",

    "method":
        "Fast model-agnostic XAI",

    "tree_shap_used":
        False,

    "global_method": {
        "type":
            "Ensemble permutation feature importance",
        "sample_size":
            500,
        "repeats_per_feature":
            2,
        "features_completed":
            73,
        "random_state":
            42
    },

    "local_method": {
        "type":
            "SHAP PermutationExplainer",
        "background_rows":
            BACKGROUND_SIZE,
        "cases":
            1,
        "numeric_transformed_space":
            True,
        "transformed_feature_count":
            len(transformed_feature_names),
        "random_state":
            42
    },

    "ensemble": {
        "model_a":
            MODEL_A,
        "weight_a":
            WEIGHT_A,
        "model_b":
            MODEL_B,
        "weight_b":
            WEIGHT_B
    },

    "validation": {
        "rows":
            69097,
        "final_holdout_used":
            False
    },

    "local_case":
        local_case,

    "top_global_features":
        top_global_features,

    "top_local_features":
        local_shap_df.head(15)[
            [
                "Transformed_Feature",
                "Original_Feature",
                "Feature_Group",
                "Feature_Value",
                "SHAP_Value"
            ]
        ].to_dict(
            orient="records"
        ),

    "files": {
        "ensemble_permutation":
            ENSEMBLE_PERMUTATION_FILE,

        "et_native":
            ET_NATIVE_FILE,

        "rf_native":
            RF_NATIVE_FILE,

        "grouped":
            GROUPED_IMPORTANCE_FILE,

        "local_shap":
            LOCAL_SHAP_FILE,

        "local_case":
            LOCAL_CASE_FILE,

        "global_plot":
            GLOBAL_PLOT_FILE,

        "et_plot":
            ET_PLOT_FILE,

        "rf_plot":
            RF_PLOT_FILE,

        "local_plot":
            LOCAL_PLOT_FILE,

        "summary":
            SUMMARY_FILE
    },

    "final_holdout_status":
        "FROZEN / UNTOUCHED",

    "next_phase":
        "15.12 — Final Holdout Evaluation"
}

with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# 25. VERIFY OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.11 — OUTPUT VERIFICATION")
print("=" * 80)

output_files = {

    "Global permutation importance":
        GLOBAL_PERMUTATION_FILE,

    "ET native importance":
        ET_NATIVE_FILE,

    "RF native importance":
        RF_NATIVE_FILE,

    "Grouped importance":
        GROUPED_IMPORTANCE_FILE,

    "Local SHAP":
        LOCAL_SHAP_FILE,

    "Local case":
        LOCAL_CASE_FILE,

    "Global permutation plot":
        GLOBAL_PLOT_FILE,

    "ET native plot":
        ET_PLOT_FILE,

    "RF native plot":
        RF_PLOT_FILE,

    "Local SHAP waterfall":
        LOCAL_PLOT_FILE,

    "XAI summary":
        SUMMARY_FILE
}

verification_rows = []

for name, path in output_files.items():

    verification_rows.append(
        {
            "Artifact":
                name,

            "Status":
                "FOUND"
                if os.path.exists(path)
                else "MISSING"
        }
    )

verification_df = pd.DataFrame(
    verification_rows
)

print(
    verification_df.to_string(
        index=False
    )
)

if (
    verification_df["Status"]
    == "MISSING"
).any():

    raise RuntimeError(
        "One or more Phase 15.11 artifacts are missing."
    )

# =============================================================================
# 26. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.11 COMPLETED")
print("=" * 80)

print(
    "Global permutation importance: COMPLETE"
)

print(
    "Local SHAP: COMPLETE"
)

print(
    f"Local SHAP additivity error: "
    f"{local_additivity_error:.12f}"
)

print(
    f"Top global feature: "
    f"{global_perm_df.iloc[0]['Feature']}"
    if os.path.exists(GLOBAL_PERMUTATION_FILE)
    else "Already saved"
)

print(
    "\nFINAL_HOLDOUT: FROZEN / UNTOUCHED"
)

print(
    f"\nXAI outputs:\n{OUTPUT_DIR}"
)

print("\n" + "=" * 80)
print("NEXT: PHASE 15.12 — FINAL HOLDOUT EVALUATION")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.11 — LOCAL XAI CORRECTION
# =============================================================================
# Fixes the local-case selection so it uses the SAME final train+validation
# ensemble that is being explained.
#
# Global permutation importance is already complete and is NOT recalculated.
# FINAL_HOLDOUT remains untouched.
# =============================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data/processed/delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_1_time_split_assignments.parquet"
)

FINAL_CONFIG_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_final_ensemble_config.json"
)

FINAL_ET_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
)

FINAL_RF_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_random_forest_trainval.joblib"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_11_fast_xai"
)

PLOT_DIR = os.path.join(
    OUTPUT_DIR,
    "plots"
)

LOCAL_SHAP_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_local_permutation_shap.csv"
)

LOCAL_CASE_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_local_case.json"
)

LOCAL_PLOT_FILE = os.path.join(
    PLOT_DIR,
    "phase15_11_local_ensemble_shap_waterfall.png"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_11_fast_xai_summary.json"
)

# =============================================================================
# 1. LOAD CONFIGURATION
# =============================================================================

with open(FINAL_CONFIG_FILE, "r") as f:
    config = json.load(f)

WEIGHT_A = float(
    config["component_models"][0]["weight"]
)

WEIGHT_B = float(
    config["component_models"][1]["weight"]
)

# =============================================================================
# 2. EXACT FEATURE CONTRACT
# =============================================================================

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

pollutant_features = current_features.copy()

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in pollutant_features
]

rolling_source_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2"
]

rolling_features = []

for feature in rolling_source_features:

    for window in [3, 12]:

        rolling_features.append(
            f"{feature}_rolling_mean_{window}h"
        )

        rolling_features.append(
            f"{feature}_rolling_std_{window}h"
        )

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

feature_cols = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + ["StationId"]
)

if len(feature_cols) != 73:
    raise ValueError(
        f"Expected 73 features, got {len(feature_cols)}"
    )

TARGET = "AQI_t_plus_1"
DATETIME = "Datetime"
STATION = "StationId"

# =============================================================================
# 3. LOAD FINAL MODELS
# =============================================================================

print("=" * 80)
print("PHASE 15.11 — LOCAL XAI CORRECTION")
print("=" * 80)

et_model = joblib.load(
    FINAL_ET_MODEL
)

rf_model = joblib.load(
    FINAL_RF_MODEL
)

print("Final ET model loaded.")
print("Final RF model loaded.")

# =============================================================================
# 4. LOAD FEATURE DATA + SPLIT
# =============================================================================

df = pd.read_parquet(
    FEATURE_FILE
)

split_df = pd.read_parquet(
    SPLIT_FILE
)

df[DATETIME] = pd.to_datetime(
    df[DATETIME],
    errors="coerce"
)

split_df[DATETIME] = pd.to_datetime(
    split_df[DATETIME],
    errors="coerce"
)

model_df = df.merge(
    split_df[
        [STATION, DATETIME, "Split"]
    ],
    on=[STATION, DATETIME],
    how="left",
    validate="one_to_one"
)

validation_df = model_df[
    model_df["Split"].eq("VALIDATION")
].copy()

validation_df = validation_df[
    validation_df[TARGET].notna()
].copy()

if len(validation_df) != 69097:
    raise ValueError(
        f"Expected 69,097 validation rows, "
        f"found {len(validation_df):,}"
    )

print(
    f"Validation rows: {len(validation_df):,}"
)

# =============================================================================
# 5. PREDICT VALIDATION USING FINAL TRAIN+VALIDATION MODELS
# =============================================================================

X_validation = validation_df[
    feature_cols
].copy()

y_validation = validation_df[
    TARGET
].astype(float).to_numpy()

et_validation_pred = np.asarray(
    et_model.predict(X_validation),
    dtype=float
).reshape(-1)

rf_validation_pred = np.asarray(
    rf_model.predict(X_validation),
    dtype=float
).reshape(-1)

final_ensemble_pred = (
    WEIGHT_A * et_validation_pred
    + WEIGHT_B * rf_validation_pred
)

final_ensemble_abs_error = np.abs(
    y_validation
    - final_ensemble_pred
)

# =============================================================================
# 6. FIND TRUE WORST CASE FOR FINAL MODEL
# =============================================================================

worst_index = int(
    np.argmax(
        final_ensemble_abs_error
    )
)

local_row = validation_df.iloc[
    worst_index
].copy()

X_local_raw = validation_df.iloc[
    [worst_index]
][feature_cols].copy()

actual_value = float(
    y_validation[worst_index]
)

et_value = float(
    et_validation_pred[worst_index]
)

rf_value = float(
    rf_validation_pred[worst_index]
)

ensemble_value = float(
    final_ensemble_pred[worst_index]
)

absolute_error = float(
    final_ensemble_abs_error[worst_index]
)

print("\n" + "=" * 80)
print("TRUE WORST VALIDATION CASE — FINAL TRAIN+VALIDATION ENSEMBLE")
print("=" * 80)

print(
    f"Datetime: {local_row[DATETIME]}"
)

print(
    f"Station: {local_row[STATION]}"
)

print(
    f"Actual AQI(t+1): {actual_value:.6f}"
)

print(
    f"ET prediction: {et_value:.6f}"
)

print(
    f"RF prediction: {rf_value:.6f}"
)

print(
    f"Final ensemble prediction: {ensemble_value:.6f}"
)

print(
    f"Absolute error: {absolute_error:.6f}"
)

# =============================================================================
# 7. EXTRACT FINAL PREPROCESSOR
# =============================================================================

if not isinstance(
    et_model,
    Pipeline
):

    raise TypeError(
        "Expected final ET model to be a Pipeline."
    )

et_preprocessor = et_model[:-1]

et_estimator = et_model.steps[-1][1]
rf_estimator = rf_model.steps[-1][1]

# =============================================================================
# 8. TRANSFORM LOCAL ROW
# =============================================================================

X_local_transformed = (
    et_preprocessor.transform(
        X_local_raw
    )
)

if hasattr(
    X_local_transformed,
    "toarray"
):

    X_local_transformed = (
        X_local_transformed.toarray()
    )

X_local_transformed = np.asarray(
    X_local_transformed,
    dtype=float
)

# =============================================================================
# 9. BACKGROUND
# =============================================================================

BACKGROUND_SIZE = 10

background_df = (
    validation_df
    .sample(
        n=BACKGROUND_SIZE,
        random_state=42
    )
    .reset_index(drop=True)
)

X_background_raw = background_df[
    feature_cols
].copy()

X_background_transformed = (
    et_preprocessor.transform(
        X_background_raw
    )
)

if hasattr(
    X_background_transformed,
    "toarray"
):

    X_background_transformed = (
        X_background_transformed.toarray()
    )

X_background_transformed = np.asarray(
    X_background_transformed,
    dtype=float
)

# =============================================================================
# 10. TRANSFORMED FEATURE NAMES
# =============================================================================

try:

    transformed_feature_names = [
        str(x)
        for x in
        et_preprocessor.get_feature_names_out(
            feature_cols
        )
    ]

except Exception:

    transformed_feature_names = [
        str(x)
        for x in
        et_preprocessor.get_feature_names_out()
    ]

if X_local_transformed.shape[1] != len(
    transformed_feature_names
):

    raise ValueError(
        "Transformed feature count mismatch."
    )

print(
    f"\nTransformed features: "
    f"{len(transformed_feature_names)}"
)

# =============================================================================
# 11. NUMERIC ENSEMBLE FUNCTION
# =============================================================================

def transformed_ensemble_predict(X):

    X = np.asarray(
        X,
        dtype=float
    )

    et_prediction = np.asarray(
        et_estimator.predict(X),
        dtype=float
    ).reshape(-1)

    rf_prediction = np.asarray(
        rf_estimator.predict(X),
        dtype=float
    ).reshape(-1)

    return (
        WEIGHT_A * et_prediction
        + WEIGHT_B * rf_prediction
    )

# =============================================================================
# 12. SHAP PERMUTATION EXPLAINER
# =============================================================================

print("\nCreating SHAP PermutationExplainer...")

explainer = shap.PermutationExplainer(
    transformed_ensemble_predict,
    X_background_transformed,
    feature_names=transformed_feature_names,
    seed=42
)

print(
    "Explainer created."
)

print(
    "Calculating SHAP for true worst-case validation row..."
)

shap_result = explainer(
    X_local_transformed,
    max_evals=(
        2 * X_local_transformed.shape[1]
        + 1
    ),
    silent=False
)

shap_values = np.asarray(
    shap_result.values,
    dtype=float
)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

if shap_values.ndim != 2:
    raise ValueError(
        f"Unexpected SHAP shape: {shap_values.shape}"
    )

shap_values = shap_values[0]

# =============================================================================
# 13. SHAP BASE VALUE + ADDITIVITY
# =============================================================================

base_values = np.asarray(
    shap_result.base_values,
    dtype=float
)

base_value = float(
    base_values.reshape(-1)[0]
)

reconstructed_prediction = (
    base_value
    + float(
        shap_values.sum()
    )
)

additivity_error = float(
    abs(
        ensemble_value
        - reconstructed_prediction
    )
)

print("\n" + "=" * 80)
print("SHAP ADDITIVITY")
print("=" * 80)

print(
    f"Model prediction:       "
    f"{ensemble_value:.6f}"
)

print(
    f"SHAP reconstruction:    "
    f"{reconstructed_prediction:.6f}"
)

print(
    f"Additivity error:        "
    f"{additivity_error:.12f}"
)

# =============================================================================
# 14. LOCAL SHAP TABLE
# =============================================================================

feature_group_map = {}

for feature in current_features:
    feature_group_map[feature] = "Current"

for feature in lag_features:
    feature_group_map[feature] = "Lag"

for feature in rolling_features:
    feature_group_map[feature] = "Rolling"

for feature in calendar_features:
    feature_group_map[feature] = "Calendar"

for feature in event_features:
    feature_group_map[feature] = "Event"

feature_group_map["StationId"] = "Station"

def map_original_feature(name):

    name = str(name)

    if "__" in name:
        cleaned = name.split(
            "__",
            1
        )[1]
    else:
        cleaned = name

    if cleaned.startswith(
        "StationId_"
    ):
        return "StationId"

    for feature in sorted(
        feature_cols,
        key=len,
        reverse=True
    ):

        if cleaned == feature:
            return feature

        if cleaned.startswith(
            feature + "_"
        ):
            return feature

    return cleaned

local_shap_df = pd.DataFrame(
    {
        "Transformed_Feature":
            transformed_feature_names,

        "Feature_Value":
            X_local_transformed[0],

        "SHAP_Value":
            shap_values
    }
)

local_shap_df[
    "Absolute_SHAP"
] = np.abs(
    local_shap_df["SHAP_Value"]
)

local_shap_df[
    "Original_Feature"
] = (
    local_shap_df[
        "Transformed_Feature"
    ]
    .apply(
        map_original_feature
    )
)

local_shap_df[
    "Feature_Group"
] = (
    local_shap_df[
        "Original_Feature"
    ]
    .map(
        feature_group_map
    )
)

local_shap_df = local_shap_df.sort_values(
    "Absolute_SHAP",
    ascending=False
).reset_index(
    drop=True
)

local_shap_df.to_csv(
    LOCAL_SHAP_FILE,
    index=False
)

# =============================================================================
# 15. WATERFALL
# =============================================================================

local_explanation = shap.Explanation(
    values=shap_values,
    base_values=base_value,
    data=X_local_transformed[0],
    feature_names=transformed_feature_names
)

shap.plots.waterfall(
    local_explanation,
    max_display=15,
    show=False
)

plt.title(
    "Phase 15.11 — True Worst-Case Ensemble SHAP"
)

plt.tight_layout()

plt.savefig(
    LOCAL_PLOT_FILE,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# =============================================================================
# 16. SAVE LOCAL CASE
# =============================================================================

local_case = {
    "Datetime": str(
        local_row[DATETIME]
    ),

    "StationId": str(
        local_row[STATION]
    ),

    "Actual_AQI_t_plus_1": actual_value,

    "Extra_Trees_Prediction": et_value,

    "Random_Forest_Prediction": rf_value,

    "Final_Ensemble_Prediction": ensemble_value,

    "Absolute_Error": absolute_error,

    "SHAP_Base_Value": base_value,

    "SHAP_Reconstructed_Prediction":
        reconstructed_prediction,

    "SHAP_Additivity_Error":
        additivity_error,

    "background_rows":
        BACKGROUND_SIZE,

    "feature_count_raw":
        len(feature_cols),

    "feature_count_transformed":
        len(transformed_feature_names),

    "model_version":
        "Phase 15.10 final models trained on TRAIN + VALIDATION",

    "final_holdout_used":
        False
}

with open(
    LOCAL_CASE_FILE,
    "w"
) as f:

    json.dump(
        local_case,
        f,
        indent=2
    )

# =============================================================================
# 17. DISPLAY LOCAL RESULT
# =============================================================================

print("\n" + "=" * 80)
print("CORRECTED LOCAL SHAP RESULT")
print("=" * 80)

print(
    f"Station: {local_case['StationId']}"
)

print(
    f"Datetime: {local_case['Datetime']}"
)

print(
    f"Actual AQI(t+1): "
    f"{local_case['Actual_AQI_t_plus_1']:.6f}"
)

print(
    f"ET prediction: "
    f"{local_case['Extra_Trees_Prediction']:.6f}"
)

print(
    f"RF prediction: "
    f"{local_case['Random_Forest_Prediction']:.6f}"
)

print(
    f"Final ensemble prediction: "
    f"{local_case['Final_Ensemble_Prediction']:.6f}"
)

print(
    f"Absolute error: "
    f"{local_case['Absolute_Error']:.6f}"
)

print(
    f"SHAP additivity error: "
    f"{local_case['SHAP_Additivity_Error']:.12f}"
)

print(
    "\nTop local SHAP contributors:"
)

print(
    local_shap_df.head(20).to_string(
        index=False
    )
)

# =============================================================================
# 18. UPDATE XAI SUMMARY
# =============================================================================

if os.path.exists(
    os.path.join(
        OUTPUT_DIR,
        "phase15_11_ensemble_permutation_importance.csv"
    )
):

    global_df = pd.read_csv(
        os.path.join(
            OUTPUT_DIR,
            "phase15_11_ensemble_permutation_importance.csv"
        )
    )

    top_global = (
        global_df
        .head(15)
        .to_dict(
            orient="records"
        )
    )

else:

    top_global = []

summary = {
    "phase": "15.11",

    "status":
        "COMPLETED",

    "global_xai":
        "Ensemble permutation importance",

    "local_xai":
        "SHAP PermutationExplainer on numeric transformed feature space",

    "global_sample_size":
        500,

    "global_permutation_repeats":
        2,

    "local_background_rows":
        BACKGROUND_SIZE,

    "ensemble": {
        "model_a": "Tuned Extra Trees",
        "weight_a": WEIGHT_A,
        "model_b": "Tuned Random Forest",
        "weight_b": WEIGHT_B
    },

    "local_case":
        local_case,

    "top_global_features":
        top_global,

    "top_local_features":
        local_shap_df.head(15)[
            [
                "Transformed_Feature",
                "Original_Feature",
                "Feature_Group",
                "Feature_Value",
                "SHAP_Value"
            ]
        ].to_dict(
            orient="records"
        ),

    "final_holdout_status":
        "FROZEN / UNTOUCHED",

    "next_phase":
        "15.12 — Final Holdout Evaluation"
}

with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# 19. FINAL CHECK
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.11 — LOCAL XAI CORRECTION COMPLETE")
print("=" * 80)

print(
    "Global permutation analysis: ALREADY COMPLETE"
)

print(
    "Local case: CORRECTED"
)

print(
    f"Correct final-model error: "
    f"{absolute_error:.6f}"
)

print(
    f"SHAP additivity error: "
    f"{additivity_error:.12f}"
)

print(
    "\nFINAL_HOLDOUT: FROZEN / UNTOUCHED"
)

print(
    "\nYou can now proceed to Phase 15.12."
)

print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.12 — FINAL HOLDOUT EVALUATION
# =============================================================================
#
# FINAL EVALUATION PHASE
#
# Purpose:
#   1. Load the frozen final Tuned Extra Trees model.
#   2. Load the frozen final Tuned Random Forest model.
#   3. Load the frozen ensemble weights from Phase 15.10.
#   4. Evaluate ONLY on FINAL_HOLDOUT.
#   5. Report:
#        - Individual ET metrics
#        - Individual RF metrics
#        - Final weighted ensemble metrics
#        - Persistence benchmark
#        - Station-level metrics
#        - AQI-range metrics
#        - Error distribution
#   6. Save complete final predictions and reports.
#
# IMPORTANT:
#   - NO retraining
#   - NO tuning
#   - NO weight optimization
#   - NO model selection
#   - FINAL_HOLDOUT is used ONLY for final evaluation
#
# After this phase, the holdout must never be used to modify the model.
#
# Restart-safe.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =============================================================================
# 1. PATHS
# =============================================================================

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE_DIR,
    "data/processed/delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_1_time_split_assignments.parquet"
)

FINAL_CONFIG_FILE = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_10_final_ensemble_config.json"
)

FINAL_ET_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
)

FINAL_RF_MODEL = os.path.join(
    BASE_DIR,
    "models/phase15_10_final_tuned_random_forest_trainval.joblib"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_12_final_holdout"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

FINAL_PREDICTIONS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_predictions.parquet"
)

OVERALL_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_results.csv"
)

STATION_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_station_results.csv"
)

AQI_RANGE_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_aqi_range_results.csv"
)

ERROR_SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_error_summary.json"
)

FINAL_REPORT_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_evaluation_report.json"
)

SESSION_STATUS_FILE = os.path.join(
    BASE_DIR,
    "outputs/checkpoints/session_status.json"
)

# =============================================================================
# 2. VERIFY REQUIRED FILES
# =============================================================================

required_files = {
    "Feature engineered data": FEATURE_FILE,
    "Frozen split assignments": SPLIT_FILE,
    "Final ensemble configuration": FINAL_CONFIG_FILE,
    "Final Tuned Extra Trees": FINAL_ET_MODEL,
    "Final Tuned Random Forest": FINAL_RF_MODEL
}

for label, path in required_files.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )

# =============================================================================
# 3. LOAD FINAL ENSEMBLE CONFIGURATION
# =============================================================================

with open(
    FINAL_CONFIG_FILE,
    "r"
) as f:

    final_config = json.load(f)

MODEL_A = final_config[
    "component_models"
][0]["model"]

MODEL_B = final_config[
    "component_models"
][1]["model"]

WEIGHT_A = float(
    final_config[
        "component_models"
    ][0]["weight"]
)

WEIGHT_B = float(
    final_config[
        "component_models"
    ][1]["weight"]
)

if MODEL_A != "Tuned Extra Trees":

    raise ValueError(
        f"Unexpected Model A: {MODEL_A}"
    )

if MODEL_B != "Tuned Random Forest":

    raise ValueError(
        f"Unexpected Model B: {MODEL_B}"
    )

if not np.isclose(
    WEIGHT_A + WEIGHT_B,
    1.0,
    atol=1e-10
):

    raise ValueError(
        "Final ensemble weights do not sum to 1."
    )

# =============================================================================
# 4. PROJECT HEADER
# =============================================================================

print("=" * 80)
print("PHASE 15.12 — FINAL HOLDOUT EVALUATION")
print("=" * 80)

print(
    f"\nFinal Model A: {MODEL_A}"
)

print(
    f"Weight A: {WEIGHT_A:.6f}"
)

print(
    f"Final Model B: {MODEL_B}"
)

print(
    f"Weight B: {WEIGHT_B:.6f}"
)

print(
    "\nIMPORTANT:"
)

print(
    "FINAL_HOLDOUT will now be evaluated for the first time."
)

print(
    "It will NOT be used for tuning or model selection."
)

# =============================================================================
# 5. EXACT 73-FEATURE CONTRACT
# =============================================================================

current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

pollutant_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    f"{feature}_lag_{lag}h"
    for lag in [1, 2, 12]
    for feature in pollutant_features
]

rolling_source_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2"
]

rolling_features = []

for feature in rolling_source_features:

    for window in [3, 12]:

        rolling_features.append(
            f"{feature}_rolling_mean_{window}h"
        )

        rolling_features.append(
            f"{feature}_rolling_std_{window}h"
        )

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

feature_cols = (
    current_features
    + lag_features
    + rolling_features
    + calendar_features
    + event_features
    + ["StationId"]
)

if len(feature_cols) != 73:

    raise ValueError(
        f"Expected 73 model features, got {len(feature_cols)}."
    )

TARGET = "AQI_t_plus_1"
DATETIME = "Datetime"
STATION = "StationId"

print(
    f"\nFeature contract verified: "
    f"{len(feature_cols)} features"
)

# =============================================================================
# 6. LOAD DATA
# =============================================================================

print(
    "\nLoading feature-engineered data..."
)

df = pd.read_parquet(
    FEATURE_FILE
)

print(
    "Loading frozen split assignments..."
)

split_df = pd.read_parquet(
    SPLIT_FILE
)

df[DATETIME] = pd.to_datetime(
    df[DATETIME],
    errors="coerce"
)

split_df[DATETIME] = pd.to_datetime(
    split_df[DATETIME],
    errors="coerce"
)

if df[DATETIME].isna().any():

    raise ValueError(
        "Invalid Datetime values in feature data."
    )

if split_df[DATETIME].isna().any():

    raise ValueError(
        "Invalid Datetime values in split data."
    )

# =============================================================================
# 7. ATTACH FROZEN SPLIT
# =============================================================================

split_keys = [
    STATION,
    DATETIME
]

if split_df.duplicated(
    split_keys
).any():

    raise ValueError(
        "Duplicate StationId-Datetime keys in split file."
    )

model_df = df.merge(
    split_df[
        split_keys + ["Split"]
    ],
    on=split_keys,
    how="left",
    validate="one_to_one"
)

if model_df["Split"].isna().any():

    raise ValueError(
        "Some rows have no split assignment."
    )

# =============================================================================
# 8. SELECT FINAL HOLDOUT
# =============================================================================

holdout_df = model_df[
    model_df["Split"].eq("FINAL_HOLDOUT")
].copy()

holdout_df = holdout_df[
    holdout_df[TARGET].notna()
].copy()

holdout_df = holdout_df.sort_values(
    [DATETIME, STATION]
).reset_index(
    drop=True
)

EXPECTED_HOLDOUT_ROWS = 68352

if len(holdout_df) != EXPECTED_HOLDOUT_ROWS:

    raise ValueError(
        f"Expected {EXPECTED_HOLDOUT_ROWS:,} final holdout rows, "
        f"found {len(holdout_df):,}."
    )

if holdout_df["Split"].eq(
    "TRAIN"
).any():

    raise RuntimeError(
        "TRAIN rows entered final holdout."
    )

if holdout_df["Split"].eq(
    "VALIDATION"
).any():

    raise RuntimeError(
        "VALIDATION rows entered final holdout."
    )

print("\n" + "=" * 80)
print("FINAL HOLDOUT DATASET")
print("=" * 80)

print(
    f"Rows: {len(holdout_df):,}"
)

print(
    f"Stations: {holdout_df[STATION].nunique():,}"
)

print(
    f"Start: {holdout_df[DATETIME].min()}"
)

print(
    f"End: {holdout_df[DATETIME].max()}"
)

# =============================================================================
# 9. PREPARE HOLDOUT DATA
# =============================================================================

missing_holdout_features = [
    col
    for col in feature_cols
    if col not in holdout_df.columns
]

if missing_holdout_features:

    raise KeyError(
        "Missing holdout features:\n"
        + "\n".join(
            missing_holdout_features
        )
    )

X_holdout = holdout_df[
    feature_cols
].copy()

y_holdout = holdout_df[
    TARGET
].astype(float).to_numpy()

# =============================================================================
# 10. LOAD FINAL MODELS
# =============================================================================

print("\n" + "=" * 80)
print("LOADING FINAL COMPONENT MODELS")
print("=" * 80)

print("Loading Tuned Extra Trees...")

start_time = time.time()

et_model = joblib.load(
    FINAL_ET_MODEL
)

print(
    f"ET loaded in "
    f"{time.time() - start_time:.2f} seconds"
)

print("Loading Tuned Random Forest...")

start_time = time.time()

rf_model = joblib.load(
    FINAL_RF_MODEL
)

print(
    f"RF loaded in "
    f"{time.time() - start_time:.2f} seconds"
)

# =============================================================================
# 11. BATCH PREDICTION FUNCTION
# =============================================================================
#
# Batch prediction avoids unnecessarily large temporary memory usage.
# =============================================================================

BATCH_SIZE = 5000

def batch_predict(
    model,
    X,
    model_label
):

    predictions = []

    total_rows = len(X)

    total_batches = (
        (total_rows + BATCH_SIZE - 1)
        // BATCH_SIZE
    )

    print(
        f"\n{model_label}: "
        f"{total_rows:,} rows / "
        f"{total_batches} batches"
    )

    start_time = time.time()

    for batch_number, start in enumerate(
        range(
            0,
            total_rows,
            BATCH_SIZE
        ),
        start=1
    ):

        end = min(
            start + BATCH_SIZE,
            total_rows
        )

        batch = X.iloc[
            start:end
        ]

        batch_prediction = np.asarray(
            model.predict(batch),
            dtype=float
        ).reshape(-1)

        if len(batch_prediction) != len(batch):

            raise ValueError(
                f"{model_label}: prediction size mismatch "
                f"in batch {batch_number}."
            )

        if not np.isfinite(
            batch_prediction
        ).all():

            raise ValueError(
                f"{model_label}: NaN/inf prediction detected."
            )

        predictions.append(
            batch_prediction
        )

        print(
            f"  Batch {batch_number}/{total_batches} "
            f"completed "
            f"({end:,}/{total_rows:,})"
        )

    predictions = np.concatenate(
        predictions
    )

    print(
        f"{model_label} prediction completed in "
        f"{time.time() - start_time:.2f} seconds"
    )

    return predictions

# =============================================================================
# 12. FINAL HOLDOUT PREDICTIONS — EXTRA TREES
# =============================================================================

et_holdout_pred = batch_predict(
    et_model,
    X_holdout,
    "Tuned Extra Trees"
)

# =============================================================================
# 13. FINAL HOLDOUT PREDICTIONS — RANDOM FOREST
# =============================================================================

rf_holdout_pred = batch_predict(
    rf_model,
    X_holdout,
    "Tuned Random Forest"
)

# =============================================================================
# 14. FINAL WEIGHTED ENSEMBLE
# =============================================================================

ensemble_holdout_pred = (
    WEIGHT_A * et_holdout_pred
    + WEIGHT_B * rf_holdout_pred
)

if not np.isfinite(
    ensemble_holdout_pred
).all():

    raise ValueError(
        "Ensemble predictions contain NaN/inf."
    )

# =============================================================================
# 15. PERSISTENCE BASELINE
# =============================================================================
#
# Persistence predicts:
#
#   AQI(t+1) = AQI(t)
#
# The current AQI is already an input feature.
# This is a benchmark only and is NOT used for model selection.
# =============================================================================

persistence_pred = (
    holdout_df["AQI"]
    .astype(float)
    .to_numpy()
)

# =============================================================================
# 16. METRIC FUNCTION
# =============================================================================

def calculate_metrics(
    y_true,
    y_pred
):

    error = (
        y_true
        - y_pred
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    accuracy_10 = (
        np.mean(
            np.abs(error) <= 10
        )
        * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(error) <= 20
        )
        * 100
    )

    return {
        "Records": len(y_true),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
        "Accuracy ±10 AQI": float(
            accuracy_10
        ),
        "Accuracy ±20 AQI": float(
            accuracy_20
        ),
        "Mean Error": float(
            np.mean(error)
        ),
        "Median Error": float(
            np.median(error)
        )
    }

# =============================================================================
# 17. OVERALL FINAL HOLDOUT METRICS
# =============================================================================

metrics_rows = []

et_metrics = calculate_metrics(
    y_holdout,
    et_holdout_pred
)

metrics_rows.append({
    "Model":
        "Tuned Extra Trees",
    "Model Type":
        "Final Component",
    **et_metrics
})

rf_metrics = calculate_metrics(
    y_holdout,
    rf_holdout_pred
)

metrics_rows.append({
    "Model":
        "Tuned Random Forest",
    "Model Type":
        "Final Component",
    **rf_metrics
})

ensemble_metrics = calculate_metrics(
    y_holdout,
    ensemble_holdout_pred
)

metrics_rows.append({
    "Model":
        "Optimized Weighted Ensemble",
    "Model Type":
        "FINAL MODEL",
    **ensemble_metrics
})

persistence_metrics = calculate_metrics(
    y_holdout,
    persistence_pred
)

metrics_rows.append({
    "Model":
        "Persistence",
    "Model Type":
        "Benchmark",
    **persistence_metrics
})

overall_results = pd.DataFrame(
    metrics_rows
)

overall_results.to_csv(
    OVERALL_RESULTS_FILE,
    index=False
)

# =============================================================================
# 18. HOLDOUT PREDICTION OUTPUT
# =============================================================================

final_predictions = holdout_df[
    [
        DATETIME,
        STATION,
        TARGET
    ]
].copy()

final_predictions[
    "Tuned_Extra_Trees_Prediction"
] = et_holdout_pred

final_predictions[
    "Tuned_Random_Forest_Prediction"
] = rf_holdout_pred

final_predictions[
    "Final_Ensemble_Prediction"
] = ensemble_holdout_pred

final_predictions[
    "Persistence_Prediction"
] = persistence_pred

final_predictions[
    "Final_Ensemble_Error"
] = (
    y_holdout
    - ensemble_holdout_pred
)

final_predictions[
    "Final_Ensemble_Absolute_Error"
] = np.abs(
    final_predictions[
        "Final_Ensemble_Error"
    ]
)

final_predictions.to_parquet(
    FINAL_PREDICTIONS_FILE,
    index=False
)

# =============================================================================
# 19. STATION-LEVEL FINAL RESULTS
# =============================================================================

station_rows = []

for station_id, group in final_predictions.groupby(
    STATION,
    sort=True
):

    actual_station = (
        group[TARGET]
        .astype(float)
        .to_numpy()
    )

    pred_station = (
        group["Final_Ensemble_Prediction"]
        .astype(float)
        .to_numpy()
    )

    station_metrics = calculate_metrics(
        actual_station,
        pred_station
    )

    station_rows.append({
        "StationId":
            station_id,
        **station_metrics
    })

station_results = pd.DataFrame(
    station_rows
)

station_results = station_results.sort_values(
    "RMSE"
).reset_index(
    drop=True
)

station_results.to_csv(
    STATION_RESULTS_FILE,
    index=False
)

# =============================================================================
# 20. AQI-RANGE FINAL RESULTS
# =============================================================================

aqi_bins = [
    -np.inf,
    100,
    200,
    300,
    400,
    np.inf
]

aqi_labels = [
    "<100",
    "100–199",
    "200–299",
    "300–399",
    "≥400"
]

final_predictions[
    "Actual_AQI_Range"
] = pd.cut(
    final_predictions[TARGET],
    bins=aqi_bins,
    labels=aqi_labels,
    right=False
)

aqi_rows = []

for aqi_range, group in final_predictions.groupby(
    "Actual_AQI_Range",
    observed=True
):

    actual_range = (
        group[TARGET]
        .astype(float)
        .to_numpy()
    )

    prediction_range = (
        group["Final_Ensemble_Prediction"]
        .astype(float)
        .to_numpy()
    )

    range_metrics = calculate_metrics(
        actual_range,
        prediction_range
    )

    aqi_rows.append({
        "AQI Range":
            str(aqi_range),
        **range_metrics
    })

aqi_range_results = pd.DataFrame(
    aqi_rows
)

aqi_range_results.to_csv(
    AQI_RANGE_RESULTS_FILE,
    index=False
)

# =============================================================================
# 21. ERROR DISTRIBUTION
# =============================================================================

ensemble_error = (
    y_holdout
    - ensemble_holdout_pred
)

absolute_error = np.abs(
    ensemble_error
)

error_summary = {

    "mean_error":
        float(np.mean(ensemble_error)),

    "median_error":
        float(np.median(ensemble_error)),

    "std_error":
        float(np.std(ensemble_error)),

    "mean_absolute_error":
        float(np.mean(absolute_error)),

    "p90_absolute_error":
        float(np.percentile(
            absolute_error,
            90
        )),

    "p95_absolute_error":
        float(np.percentile(
            absolute_error,
            95
        )),

    "p99_absolute_error":
        float(np.percentile(
            absolute_error,
            99
        )),

    "maximum_absolute_error":
        float(np.max(absolute_error)),

    "within_5_aqi_percent":
        float(
            np.mean(
                absolute_error <= 5
            ) * 100
        ),

    "within_10_aqi_percent":
        float(
            np.mean(
                absolute_error <= 10
            ) * 100
        ),

    "within_20_aqi_percent":
        float(
            np.mean(
                absolute_error <= 20
            ) * 100
        ),

    "overprediction_percent":
        float(
            np.mean(
                ensemble_error < 0
            ) * 100
        ),

    "underprediction_percent":
        float(
            np.mean(
                ensemble_error > 0
            ) * 100
        )
}

with open(
    ERROR_SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        error_summary,
        f,
        indent=2
    )

# =============================================================================
# 22. FINAL PERFORMANCE COMPARISON
# =============================================================================

ensemble_rmse = (
    ensemble_metrics["RMSE"]
)

ensemble_mae = (
    ensemble_metrics["MAE"]
)

ensemble_acc10 = (
    ensemble_metrics["Accuracy ±10 AQI"]
)

ensemble_acc20 = (
    ensemble_metrics["Accuracy ±20 AQI"]
)

best_nonensemble_rmse = min(
    et_metrics["RMSE"],
    rf_metrics["RMSE"]
)

best_nonensemble_name = (
    "Tuned Extra Trees"
    if et_metrics["RMSE"]
    <= rf_metrics["RMSE"]
    else "Tuned Random Forest"
)

rmse_vs_best_component = (
    best_nonensemble_rmse
    - ensemble_rmse
)

# =============================================================================
# 23. FINAL REPORT JSON
# =============================================================================

final_report = {

    "phase":
        "15.12",

    "status":
        "FINAL HOLDOUT EVALUATION COMPLETED",

    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "target":
        TARGET,

    "holdout": {
        "records":
            int(len(holdout_df)),
        "stations":
            int(holdout_df[STATION].nunique()),
        "start":
            str(holdout_df[DATETIME].min()),
        "end":
            str(holdout_df[DATETIME].max())
    },

    "ensemble": {
        "model_a":
            MODEL_A,
        "weight_a":
            WEIGHT_A,
        "model_b":
            MODEL_B,
        "weight_b":
            WEIGHT_B
    },

    "final_model": {
        "name":
            "Optimized Weighted Ensemble",
        "formula":
            (
                f"{WEIGHT_A:.6f} * Tuned Extra Trees + "
                f"{WEIGHT_B:.6f} * Tuned Random Forest"
            )
    },

    "holdout_metrics": {
        "extra_trees":
            et_metrics,
        "random_forest":
            rf_metrics,
        "final_ensemble":
            ensemble_metrics,
        "persistence":
            persistence_metrics
    },

    "final_ensemble_error_distribution":
        error_summary,

    "ensemble_vs_best_component": {
        "best_component_by_rmse":
            best_nonensemble_name,
        "component_rmse":
            float(best_nonensemble_rmse),
        "ensemble_rmse":
            float(ensemble_rmse),
        "rmse_difference":
            float(rmse_vs_best_component)
    },

    "methodological_status": {
        "tuning_after_holdout":
            False,
        "model_selection_after_holdout":
            False,
        "ensemble_weight_optimization_after_holdout":
            False,
        "holdout_used_for_training":
            False,
        "holdout_used_only_for_final_evaluation":
            True
    },

    "feature_count":
        len(feature_cols),

    "final_component_training_data":
        "TRAIN + VALIDATION",

    "final_holdout_status":
        "EVALUATED ONCE / FROZEN FOR REPORTING",

    "output_directory":
        OUTPUT_DIR
}

with open(
    FINAL_REPORT_FILE,
    "w"
) as f:

    json.dump(
        final_report,
        f,
        indent=2
    )

# =============================================================================
# 24. DISPLAY OVERALL FINAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — FINAL HOLDOUT RESULTS")
print("=" * 80)

display(
    overall_results.style.format({
        "MAE":
            "{:.6f}",
        "RMSE":
            "{:.6f}",
        "R2":
            "{:.6f}",
        "Accuracy ±10 AQI":
            "{:.6f}",
        "Accuracy ±20 AQI":
            "{:.6f}",
        "Mean Error":
            "{:.6f}",
        "Median Error":
            "{:.6f}"
    })
)

print("\n" + "=" * 80)
print("FINAL ENSEMBLE PERFORMANCE")
print("=" * 80)

print(
    f"MAE:               {ensemble_mae:.6f}"
)

print(
    f"RMSE:              {ensemble_rmse:.6f}"
)

print(
    f"R2:                {ensemble_metrics['R2']:.6f}"
)

print(
    f"Accuracy ±10 AQI:  {ensemble_acc10:.6f}%"
)

print(
    f"Accuracy ±20 AQI:  {ensemble_acc20:.6f}%"
)

print(
    f"Mean Error:        {ensemble_metrics['Mean Error']:.6f}"
)

# =============================================================================
# 25. DISPLAY STATION RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — STATION-LEVEL FINAL RESULTS")
print("=" * 80)

display(
    station_results.style.format({
        "MAE":
            "{:.6f}",
        "RMSE":
            "{:.6f}",
        "R2":
            "{:.6f}",
        "Accuracy ±10 AQI":
            "{:.6f}",
        "Accuracy ±20 AQI":
            "{:.6f}",
        "Mean Error":
            "{:.6f}",
        "Median Error":
            "{:.6f}"
    })
)

# =============================================================================
# 26. DISPLAY AQI-RANGE RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — AQI-RANGE FINAL RESULTS")
print("=" * 80)

display(
    aqi_range_results.style.format({
        "MAE":
            "{:.6f}",
        "RMSE":
            "{:.6f}",
        "R2":
            "{:.6f}",
        "Accuracy ±10 AQI":
            "{:.6f}",
        "Accuracy ±20 AQI":
            "{:.6f}",
        "Mean Error":
            "{:.6f}",
        "Median Error":
            "{:.6f}"
    })
)

# =============================================================================
# 27. ERROR SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — FINAL ENSEMBLE ERROR DISTRIBUTION")
print("=" * 80)

print(
    f"Mean error:             "
    f"{error_summary['mean_error']:.6f}"
)

print(
    f"Median error:           "
    f"{error_summary['median_error']:.6f}"
)

print(
    f"90th percentile |error|:"
    f" {error_summary['p90_absolute_error']:.6f}"
)

print(
    f"95th percentile |error|:"
    f" {error_summary['p95_absolute_error']:.6f}"
)

print(
    f"99th percentile |error|:"
    f" {error_summary['p99_absolute_error']:.6f}"
)

print(
    f"Maximum |error|:        "
    f"{error_summary['maximum_absolute_error']:.6f}"
)

print(
    f"Within ±5 AQI:          "
    f"{error_summary['within_5_aqi_percent']:.6f}%"
)

print(
    f"Within ±10 AQI:         "
    f"{error_summary['within_10_aqi_percent']:.6f}%"
)

print(
    f"Within ±20 AQI:         "
    f"{error_summary['within_20_aqi_percent']:.6f}%"
)

# =============================================================================
# 28. VERIFY PREDICTION FILE
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — OUTPUT VERIFICATION")
print("=" * 80)

output_files = {

    "Final holdout predictions":
        FINAL_PREDICTIONS_FILE,

    "Overall final results":
        OVERALL_RESULTS_FILE,

    "Station final results":
        STATION_RESULTS_FILE,

    "AQI-range final results":
        AQI_RANGE_RESULTS_FILE,

    "Error summary":
        ERROR_SUMMARY_FILE,

    "Final evaluation report":
        FINAL_REPORT_FILE
}

verification_rows = []

for name, path in output_files.items():

    verification_rows.append({
        "Artifact":
            name,

        "Status":
            "FOUND"
            if os.path.exists(path)
            else "MISSING",

        "Size_MB":
            (
                os.path.getsize(path)
                / (1024 ** 2)
                if os.path.exists(path)
                else np.nan
            )
    })

verification_df = pd.DataFrame(
    verification_rows
)

display(
    verification_df.style.format({
        "Size_MB":
            "{:.3f}"
    })
)

if (
    verification_df["Status"]
    == "MISSING"
).any():

    raise RuntimeError(
        "One or more Phase 15.12 artifacts are missing."
    )

# =============================================================================
# 29. UPDATE SESSION STATUS
# =============================================================================

session_status = {

    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "last_completed_phase":
        "15.12",

    "status":
        "COMPLETED",

    "next_phase":
        "16.0 — Deployment / Live Inference Preparation",

    "final_holdout":
        "EVALUATED ONCE / FROZEN FOR REPORTING",

    "final_model":
        "Optimized Weighted Ensemble",

    "ensemble_weights": {
        "Tuned Extra Trees":
            WEIGHT_A,
        "Tuned Random Forest":
            WEIGHT_B
    },

    "final_holdout_records":
        int(len(holdout_df)),

    "final_ensemble_rmse":
        float(ensemble_rmse),

    "final_ensemble_mae":
        float(ensemble_mae),

    "final_ensemble_accuracy_pm10":
        float(ensemble_acc10),

    "final_ensemble_accuracy_pm20":
        float(ensemble_acc20),

    "updated":
        pd.Timestamp.now().isoformat()
}

os.makedirs(
    os.path.dirname(
        SESSION_STATUS_FILE
    ),
    exist_ok=True
)

with open(
    SESSION_STATUS_FILE,
    "w"
) as f:

    json.dump(
        session_status,
        f,
        indent=2
    )

# =============================================================================
# 30. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 COMPLETED")
print("=" * 80)

print(
    f"Final holdout rows: "
    f"{len(holdout_df):,}"
)

print(
    f"Final ensemble RMSE: "
    f"{ensemble_rmse:.6f}"
)

print(
    f"Final ensemble MAE: "
    f"{ensemble_mae:.6f}"
)

print(
    f"Final ensemble R2: "
    f"{ensemble_metrics['R2']:.6f}"
)

print(
    f"Final Accuracy ±10 AQI: "
    f"{ensemble_acc10:.6f}%"
)

print(
    f"Final Accuracy ±20 AQI: "
    f"{ensemble_acc20:.6f}%"
)

print(
    "\nHoldout usage:"
)

print(
    "FINAL_HOLDOUT was used ONLY for this final evaluation."
)

print(
    "No retraining, tuning, weight optimization, or model selection "
    "was performed using holdout results."
)

print(
    "\nFinal holdout status:"
)

print(
    "EVALUATED ONCE / FROZEN FOR REPORTING"
)

print(
    f"\nFinal outputs:\n{OUTPUT_DIR}"
)

print(
    f"\nSession status:\n{SESSION_STATUS_FILE}"
)

print("\n" + "=" * 80)
print("NEXT: PHASE 16.0 — DEPLOYMENT / LIVE INFERENCE PREPARATION")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 15.12 — FIX PERSISTENCE BENCHMARK + COMPLETE FINAL EVALUATION
# =============================================================================
#
# The ET, RF, and ensemble holdout predictions have already completed.
# The previous cell stopped only because the persistence benchmark contained
# NaN values where current AQI(t) was unavailable.
#
# This cell:
#   1. Correctly evaluates persistence only where AQI(t) is available.
#   2. Keeps ALL 68,352 ET/RF/ensemble holdout predictions.
#   3. Completes station-level and AQI-range final evaluation.
#   4. Saves final prediction/report artifacts.
#
# FINAL_HOLDOUT is used ONLY for final evaluation.
# No tuning or model selection is performed.
# =============================================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs/reports/phase15_12_final_holdout"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

FINAL_PREDICTIONS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_predictions.parquet"
)

OVERALL_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_results.csv"
)

STATION_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_station_results.csv"
)

AQI_RANGE_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_aqi_range_results.csv"
)

ERROR_SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_holdout_error_summary.json"
)

FINAL_REPORT_FILE = os.path.join(
    OUTPUT_DIR,
    "phase15_12_final_evaluation_report.json"
)

SESSION_STATUS_FILE = os.path.join(
    BASE_DIR,
    "outputs/checkpoints/session_status.json"
)

# =============================================================================
# 1. VERIFY VARIABLES FROM THE SUCCESSFULLY EXECUTED PORTION
# =============================================================================

required_runtime_variables = [
    "holdout_df",
    "X_holdout",
    "y_holdout",
    "et_holdout_pred",
    "rf_holdout_pred",
    "ensemble_holdout_pred",
    "MODEL_A",
    "MODEL_B",
    "WEIGHT_A",
    "WEIGHT_B"
]

missing_runtime_variables = [
    name
    for name in required_runtime_variables
    if name not in globals()
]

if missing_runtime_variables:
    raise RuntimeError(
        "The previous Phase 15.12 runtime state is unavailable.\n"
        "Missing variables:\n"
        + "\n".join(missing_runtime_variables)
        + "\n\n"
        "In that case, rerun the corrected full Phase 15.12 cell "
        "from the beginning."
    )

print("=" * 80)
print("PHASE 15.12 — CONTINUING AFTER PERSISTENCE FIX")
print("=" * 80)

print(
    f"Holdout rows: {len(holdout_df):,}"
)

print(
    f"ET predictions: {len(et_holdout_pred):,}"
)

print(
    f"RF predictions: {len(rf_holdout_pred):,}"
)

print(
    f"Ensemble predictions: {len(ensemble_holdout_pred):,}"
)

# =============================================================================
# 2. METRIC FUNCTION
# =============================================================================

def calculate_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred have different lengths."
        )

    valid_mask = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    valid_true = y_true[
        valid_mask
    ]

    valid_pred = y_pred[
        valid_mask
    ]

    if len(valid_true) == 0:
        raise ValueError(
            "No valid rows available for metric calculation."
        )

    error = (
        valid_true
        - valid_pred
    )

    mae = mean_absolute_error(
        valid_true,
        valid_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            valid_true,
            valid_pred
        )
    )

    r2 = r2_score(
        valid_true,
        valid_pred
    )

    accuracy_10 = (
        np.mean(
            np.abs(error) <= 10
        )
        * 100
    )

    accuracy_20 = (
        np.mean(
            np.abs(error) <= 20
        )
        * 100
    )

    return {
        "Records": int(
            len(valid_true)
        ),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
        "Accuracy ±10 AQI": float(
            accuracy_10
        ),
        "Accuracy ±20 AQI": float(
            accuracy_20
        ),
        "Mean Error": float(
            np.mean(error)
        ),
        "Median Error": float(
            np.median(error)
        )
    }

# =============================================================================
# 3. FINAL MODEL METRICS
# =============================================================================

et_metrics = calculate_metrics(
    y_holdout,
    et_holdout_pred
)

rf_metrics = calculate_metrics(
    y_holdout,
    rf_holdout_pred
)

ensemble_metrics = calculate_metrics(
    y_holdout,
    ensemble_holdout_pred
)

# =============================================================================
# 4. CORRECTED PERSISTENCE BENCHMARK
# =============================================================================
#
# Persistence:
#
#   AQI(t+1) = AQI(t)
#
# Rows with missing AQI(t) cannot be used for persistence.
# They are excluded ONLY from the persistence benchmark.
#
# They remain part of the 68,352-row model evaluation.
# =============================================================================

persistence_pred = pd.to_numeric(
    holdout_df["AQI"],
    errors="coerce"
).to_numpy(
    dtype=float
)

persistence_valid_mask = (
    np.isfinite(y_holdout)
    &
    np.isfinite(persistence_pred)
)

persistence_valid_records = int(
    persistence_valid_mask.sum()
)

persistence_missing_records = int(
    len(persistence_pred)
    - persistence_valid_records
)

if persistence_valid_records == 0:

    raise ValueError(
        "Persistence benchmark has zero valid rows."
    )

persistence_metrics = calculate_metrics(
    y_holdout[persistence_valid_mask],
    persistence_pred[persistence_valid_mask]
)

print("\n" + "=" * 80)
print("CORRECTED PERSISTENCE BENCHMARK")
print("=" * 80)

print(
    f"Total holdout rows:       "
    f"{len(holdout_df):,}"
)

print(
    f"Valid persistence rows:   "
    f"{persistence_valid_records:,}"
)

print(
    f"Excluded missing AQI(t):  "
    f"{persistence_missing_records:,}"
)

print(
    f"Persistence RMSE:         "
    f"{persistence_metrics['RMSE']:.6f}"
)

print(
    f"Persistence MAE:          "
    f"{persistence_metrics['MAE']:.6f}"
)

print(
    f"Persistence ±10 AQI:      "
    f"{persistence_metrics['Accuracy ±10 AQI']:.6f}%"
)

# =============================================================================
# 5. OVERALL RESULTS TABLE
# =============================================================================

metrics_rows = [

    {
        "Model":
            "Tuned Extra Trees",

        "Model Type":
            "Final Component",

        **et_metrics
    },

    {
        "Model":
            "Tuned Random Forest",

        "Model Type":
            "Final Component",

        **rf_metrics
    },

    {
        "Model":
            "Optimized Weighted Ensemble",

        "Model Type":
            "FINAL MODEL",

        **ensemble_metrics
    },

    {
        "Model":
            "Persistence",

        "Model Type":
            "Benchmark",

        **persistence_metrics
    }
]

overall_results = pd.DataFrame(
    metrics_rows
)

overall_results.to_csv(
    OVERALL_RESULTS_FILE,
    index=False
)

# =============================================================================
# 6. FINAL PREDICTIONS FILE
# =============================================================================

final_predictions = holdout_df[
    [
        "Datetime",
        "StationId",
        "AQI_t_plus_1"
    ]
].copy()

final_predictions[
    "Tuned_Extra_Trees_Prediction"
] = et_holdout_pred

final_predictions[
    "Tuned_Random_Forest_Prediction"
] = rf_holdout_pred

final_predictions[
    "Final_Ensemble_Prediction"
] = ensemble_holdout_pred

# Persistence stays NaN where current AQI is unavailable.
final_predictions[
    "Persistence_Prediction"
] = persistence_pred

final_predictions[
    "Final_Ensemble_Error"
] = (
    y_holdout
    - ensemble_holdout_pred
)

final_predictions[
    "Final_Ensemble_Absolute_Error"
] = np.abs(
    final_predictions[
        "Final_Ensemble_Error"
    ]
)

final_predictions.to_parquet(
    FINAL_PREDICTIONS_FILE,
    index=False
)

# =============================================================================
# 7. STATION-LEVEL FINAL RESULTS
# =============================================================================

station_rows = []

for station_id, group in final_predictions.groupby(
    "StationId",
    sort=True
):

    actual_station = pd.to_numeric(
        group["AQI_t_plus_1"],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    prediction_station = pd.to_numeric(
        group["Final_Ensemble_Prediction"],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    station_metrics = calculate_metrics(
        actual_station,
        prediction_station
    )

    station_rows.append({
        "StationId":
            station_id,

        **station_metrics
    })

station_results = pd.DataFrame(
    station_rows
)

station_results = station_results.sort_values(
    "RMSE"
).reset_index(
    drop=True
)

station_results.to_csv(
    STATION_RESULTS_FILE,
    index=False
)

# =============================================================================
# 8. AQI-RANGE FINAL RESULTS
# =============================================================================

aqi_bins = [
    -np.inf,
    100,
    200,
    300,
    400,
    np.inf
]

aqi_labels = [
    "<100",
    "100–199",
    "200–299",
    "300–399",
    "≥400"
]

final_predictions[
    "Actual_AQI_Range"
] = pd.cut(
    final_predictions[
        "AQI_t_plus_1"
    ],
    bins=aqi_bins,
    labels=aqi_labels,
    right=False
)

aqi_rows = []

for aqi_range, group in final_predictions.groupby(
    "Actual_AQI_Range",
    observed=True
):

    actual_range = pd.to_numeric(
        group["AQI_t_plus_1"],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    prediction_range = pd.to_numeric(
        group["Final_Ensemble_Prediction"],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    range_metrics = calculate_metrics(
        actual_range,
        prediction_range
    )

    aqi_rows.append({
        "AQI Range":
            str(aqi_range),

        **range_metrics
    })

aqi_range_results = pd.DataFrame(
    aqi_rows
)

aqi_range_results.to_csv(
    AQI_RANGE_RESULTS_FILE,
    index=False
)

# =============================================================================
# 9. FINAL ENSEMBLE ERROR DISTRIBUTION
# =============================================================================

ensemble_error = (
    y_holdout
    - ensemble_holdout_pred
)

absolute_error = np.abs(
    ensemble_error
)

valid_error_mask = (
    np.isfinite(
        ensemble_error
    )
)

valid_error = ensemble_error[
    valid_error_mask
]

valid_absolute_error = absolute_error[
    valid_error_mask
]

error_summary = {

    "mean_error":
        float(
            np.mean(valid_error)
        ),

    "median_error":
        float(
            np.median(valid_error)
        ),

    "std_error":
        float(
            np.std(valid_error)
        ),

    "mean_absolute_error":
        float(
            np.mean(valid_absolute_error)
        ),

    "p90_absolute_error":
        float(
            np.percentile(
                valid_absolute_error,
                90
            )
        ),

    "p95_absolute_error":
        float(
            np.percentile(
                valid_absolute_error,
                95
            )
        ),

    "p99_absolute_error":
        float(
            np.percentile(
                valid_absolute_error,
                99
            )
        ),

    "maximum_absolute_error":
        float(
            np.max(
                valid_absolute_error
            )
        ),

    "within_5_aqi_percent":
        float(
            np.mean(
                valid_absolute_error <= 5
            )
            * 100
        ),

    "within_10_aqi_percent":
        float(
            np.mean(
                valid_absolute_error <= 10
            )
            * 100
        ),

    "within_20_aqi_percent":
        float(
            np.mean(
                valid_absolute_error <= 20
            )
            * 100
        ),

    "overprediction_percent":
        float(
            np.mean(
                valid_error < 0
            )
            * 100
        ),

    "underprediction_percent":
        float(
            np.mean(
                valid_error > 0
            )
            * 100
        )
}

with open(
    ERROR_SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        error_summary,
        f,
        indent=2
    )

# =============================================================================
# 10. FINAL REPORT
# =============================================================================

final_report = {

    "phase":
        "15.12",

    "status":
        "FINAL HOLDOUT EVALUATION COMPLETED",

    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "target":
        "AQI_t_plus_1",

    "holdout": {

        "records":
            int(
                len(holdout_df)
            ),

        "stations":
            int(
                holdout_df[
                    "StationId"
                ].nunique()
            ),

        "start":
            str(
                holdout_df[
                    "Datetime"
                ].min()
            ),

        "end":
            str(
                holdout_df[
                    "Datetime"
                ].max()
            )
    },

    "ensemble": {

        "model_a":
            MODEL_A,

        "weight_a":
            WEIGHT_A,

        "model_b":
            MODEL_B,

        "weight_b":
            WEIGHT_B
    },

    "final_model": {

        "name":
            "Optimized Weighted Ensemble",

        "formula":
            (
                f"{WEIGHT_A:.6f} * Tuned Extra Trees + "
                f"{WEIGHT_B:.6f} * Tuned Random Forest"
            )
    },

    "holdout_metrics": {

        "extra_trees":
            et_metrics,

        "random_forest":
            rf_metrics,

        "final_ensemble":
            ensemble_metrics,

        "persistence":
            persistence_metrics
    },

    "persistence_benchmark": {

        "total_holdout_rows":
            int(
                len(holdout_df)
            ),

        "valid_rows":
            persistence_valid_records,

        "excluded_missing_AQI_t_rows":
            persistence_missing_records
    },

    "final_ensemble_error_distribution":
        error_summary,

    "methodological_status": {

        "holdout_used_for_training":
            False,

        "holdout_used_for_tuning":
            False,

        "holdout_used_for_model_selection":
            False,

        "holdout_used_for_ensemble_weight_optimization":
            False,

        "holdout_used_only_for_final_evaluation":
            True
    },

    "final_holdout_status":
        "EVALUATED ONCE / FROZEN FOR REPORTING",

    "output_directory":
        OUTPUT_DIR
}

with open(
    FINAL_REPORT_FILE,
    "w"
) as f:

    json.dump(
        final_report,
        f,
        indent=2
    )

# =============================================================================
# 11. DISPLAY OVERALL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — FINAL HOLDOUT RESULTS")
print("=" * 80)

display(
    overall_results.style.format({
        "MAE":
            "{:.6f}",
        "RMSE":
            "{:.6f}",
        "R2":
            "{:.6f}",
        "Accuracy ±10 AQI":
            "{:.6f}",
        "Accuracy ±20 AQI":
            "{:.6f}",
        "Mean Error":
            "{:.6f}",
        "Median Error":
            "{:.6f}"
    })
)

# =============================================================================
# 12. STATION RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — STATION-LEVEL FINAL RESULTS")
print("=" * 80)

display(
    station_results.style.format({
        "MAE":
            "{:.6f}",
        "RMSE":
            "{:.6f}",
        "R2":
            "{:.6f}",
        "Accuracy ±10 AQI":
            "{:.6f}",
        "Accuracy ±20 AQI":
            "{:.6f}",
        "Mean Error":
            "{:.6f}",
        "Median Error":
            "{:.6f}"
    })
)

# =============================================================================
# 13. AQI RANGE RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — AQI-RANGE FINAL RESULTS")
print("=" * 80)

display(
    aqi_range_results.style.format({
        "MAE":
            "{:.6f}",
        "RMSE":
            "{:.6f}",
        "R2":
            "{:.6f}",
        "Accuracy ±10 AQI":
            "{:.6f}",
        "Accuracy ±20 AQI":
            "{:.6f}",
        "Mean Error":
            "{:.6f}",
        "Median Error":
            "{:.6f}"
    })
)

# =============================================================================
# 14. ERROR SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — FINAL ENSEMBLE ERROR DISTRIBUTION")
print("=" * 80)

print(
    f"Mean error:              "
    f"{error_summary['mean_error']:.6f}"
)

print(
    f"Median error:            "
    f"{error_summary['median_error']:.6f}"
)

print(
    f"90th percentile |error|: "
    f"{error_summary['p90_absolute_error']:.6f}"
)

print(
    f"95th percentile |error|: "
    f"{error_summary['p95_absolute_error']:.6f}"
)

print(
    f"99th percentile |error|: "
    f"{error_summary['p99_absolute_error']:.6f}"
)

print(
    f"Maximum |error|:         "
    f"{error_summary['maximum_absolute_error']:.6f}"
)

print(
    f"Within ±5 AQI:           "
    f"{error_summary['within_5_aqi_percent']:.6f}%"
)

print(
    f"Within ±10 AQI:          "
    f"{error_summary['within_10_aqi_percent']:.6f}%"
)

print(
    f"Within ±20 AQI:          "
    f"{error_summary['within_20_aqi_percent']:.6f}%"
)

# =============================================================================
# 15. OUTPUT VERIFICATION
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 — OUTPUT VERIFICATION")
print("=" * 80)

output_files = {
    "Final holdout predictions":
        FINAL_PREDICTIONS_FILE,

    "Overall final results":
        OVERALL_RESULTS_FILE,

    "Station final results":
        STATION_RESULTS_FILE,

    "AQI-range final results":
        AQI_RANGE_RESULTS_FILE,

    "Error summary":
        ERROR_SUMMARY_FILE,

    "Final evaluation report":
        FINAL_REPORT_FILE
}

verification_rows = []

for name, path in output_files.items():

    verification_rows.append({
        "Artifact":
            name,

        "Status":
            "FOUND"
            if os.path.exists(path)
            else "MISSING",

        "Size_MB":
            (
                os.path.getsize(path)
                / (1024 ** 2)
                if os.path.exists(path)
                else np.nan
            )
    })

verification_df = pd.DataFrame(
    verification_rows
)

display(
    verification_df.style.format({
        "Size_MB":
            "{:.3f}"
    })
)

if (
    verification_df["Status"]
    == "MISSING"
).any():

    raise RuntimeError(
        "One or more Phase 15.12 artifacts are missing."
    )

# =============================================================================
# 16. UPDATE SESSION STATUS
# =============================================================================

session_status = {

    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "last_completed_phase":
        "15.12",

    "status":
        "COMPLETED",

    "next_phase":
        "16.0 — Deployment / Live Inference Preparation",

    "final_holdout":
        "EVALUATED ONCE / FROZEN FOR REPORTING",

    "final_model":
        "Optimized Weighted Ensemble",

    "ensemble_weights": {
        "Tuned Extra Trees":
            WEIGHT_A,

        "Tuned Random Forest":
            WEIGHT_B
    },

    "final_holdout_records":
        int(
            len(holdout_df)
        ),

    "final_ensemble_rmse":
        float(
            ensemble_metrics["RMSE"]
        ),

    "final_ensemble_mae":
        float(
            ensemble_metrics["MAE"]
        ),

    "final_ensemble_accuracy_pm10":
        float(
            ensemble_metrics[
                "Accuracy ±10 AQI"
            ]
        ),

    "final_ensemble_accuracy_pm20":
        float(
            ensemble_metrics[
                "Accuracy ±20 AQI"
            ]
        ),

    "persistence_valid_records":
        persistence_valid_records,

    "updated":
        pd.Timestamp.now().isoformat()
}

os.makedirs(
    os.path.dirname(
        SESSION_STATUS_FILE
    ),
    exist_ok=True
)

with open(
    SESSION_STATUS_FILE,
    "w"
) as f:

    json.dump(
        session_status,
        f,
        indent=2
    )

# =============================================================================
# 17. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("PHASE 15.12 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    f"Final holdout rows: "
    f"{len(holdout_df):,}"
)

print(
    f"Final ensemble RMSE: "
    f"{ensemble_metrics['RMSE']:.6f}"
)

print(
    f"Final ensemble MAE: "
    f"{ensemble_metrics['MAE']:.6f}"
)

print(
    f"Final ensemble R2: "
    f"{ensemble_metrics['R2']:.6f}"
)

print(
    f"Final Accuracy ±10 AQI: "
    f"{ensemble_metrics['Accuracy ±10 AQI']:.6f}%"
)

print(
    f"Final Accuracy ±20 AQI: "
    f"{ensemble_metrics['Accuracy ±20 AQI']:.6f}%"
)

print(
    "\nPersistence benchmark:"
)

print(
    f"Valid rows: "
    f"{persistence_valid_records:,}"
)

print(
    f"Excluded missing AQI(t): "
    f"{persistence_missing_records:,}"
)

print(
    "\nFINAL_HOLDOUT:"
)

print(
    "EVALUATED ONCE / FROZEN FOR REPORTING"
)

print(
    f"\nFinal outputs:\n{OUTPUT_DIR}"
)

print(
    f"\nSession status:\n{SESSION_STATUS_FILE}"
)

print("\n" + "=" * 80)
print("NEXT: PHASE 16.0 — DEPLOYMENT / LIVE INFERENCE PREPARATION")
print("=" * 80)

In [ ]:
# =============================================================================
# DELHI AQI PROJECT — PHASE 15.12 FINAL CHECKPOINT
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
from datetime import datetime

BASE_DIR = "/content/drive/MyDrive/Delhi-AQI-Prediction"
CHECKPOINT_DIR = os.path.join(
    BASE_DIR,
    "outputs/checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

checkpoint = {
    "project": "Delhi AQI t+1 Hour Forecasting",

    "status": "PAUSED",

    "last_completed_phase": "15.12",

    "next_phase": "16.0 — Deployment / Live Inference Preparation",

    "target": "AQI_t_plus_1",

    "final_model": {
        "name": "Optimized Weighted Ensemble",

        "components": [
            {
                "model": "Tuned Extra Trees",
                "weight": 0.594466
            },
            {
                "model": "Tuned Random Forest",
                "weight": 0.405534
            }
        ]
    },

    "final_holdout": {
        "records": 68352,
        "stations": 10,
        "status": "EVALUATED ONCE / FROZEN FOR REPORTING"
    },

    "final_metrics": {
        "MAE": 3.216121,
        "RMSE": 7.658390,
        "R2": 0.996634,
        "Accuracy_pm5": 85.249883,
        "Accuracy_pm10": 94.605864,
        "Accuracy_pm20": 98.090765
    },

    "persistence_benchmark": {
        "valid_records": 68285,
        "excluded_missing_AQI_t": 67,
        "RMSE": 8.840011,
        "MAE": 3.855634,
        "Accuracy_pm10": 92.261844
    },

    "important_outputs": {
        "final_predictions":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_holdout_predictions.parquet"
            ),

        "overall_results":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_holdout_results.csv"
            ),

        "station_results":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_holdout_station_results.csv"
            ),

        "aqi_range_results":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_holdout_aqi_range_results.csv"
            ),

        "error_summary":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_holdout_error_summary.json"
            ),

        "final_report":
            os.path.join(
                BASE_DIR,
                "outputs/reports/phase15_12_final_holdout/"
                "phase15_12_final_evaluation_report.json"
            )
    },

    "methodological_rule": (
        "Do not use FINAL_HOLDOUT for further tuning, model selection, "
        "ensemble-weight optimization, or retraining."
    ),

    "resume_instruction": (
        "Start the next session from Phase 16.0. Do not rerun Phases 0–15. "
        "Mount Google Drive and verify saved artifacts before continuing."
    ),

    "checkpoint_created":
        datetime.now().isoformat()
}

checkpoint_file = os.path.join(
    CHECKPOINT_DIR,
    "phase15_12_final_checkpoint.json"
)

with open(
    checkpoint_file,
    "w"
) as f:
    json.dump(
        checkpoint,
        f,
        indent=2
    )

session_status = {
    "project":
        "Delhi AQI t+1 Hour Forecasting",

    "last_completed_phase":
        "15.12",

    "status":
        "PAUSED",

    "next_phase":
        "16.0 — Deployment / Live Inference Preparation",

    "final_holdout":
        "EVALUATED ONCE / FROZEN FOR REPORTING",

    "checkpoint":
        checkpoint_file,

    "updated":
        datetime.now().isoformat()
}

session_status_file = os.path.join(
    CHECKPOINT_DIR,
    "session_status.json"
)

with open(
    session_status_file,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=2
    )

print("=" * 80)
print("DELHI AQI PROJECT — FINAL PHASE 15 CHECKPOINT SAVED")
print("=" * 80)
print("Last completed : Phase 15.12")
print("Next phase     : Phase 16.0")
print("Final model    : Optimized Weighted Ensemble")
print("±10 AQI        : 94.605864%")
print("MAE            : 3.216121")
print("RMSE           : 7.658390")
print("R²             : 0.996634")
print("Holdout        : 68,352 rows")
print("Holdout status : FROZEN / REPORTING ONLY")
print(f"\nCheckpoint     : {checkpoint_file}")
print(f"Session status : {session_status_file}")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 15 RESUME — RESTORE AFTER COLAB RUNTIME DISCONNECT
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = OUTPUTS_DIR / "reports"
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints"

print("=" * 70)
print("PHASE 15 RESUME CHECK")
print("=" * 70)

assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"

required_files = {
    "Final Extra Trees model":
        MODELS_DIR / "phase15_10_final_tuned_extra_trees_trainval.joblib",

    "Final Random Forest model":
        MODELS_DIR / "phase15_10_final_tuned_random_forest_trainval.joblib",

    "Final ensemble config":
        REPORTS_DIR / "phase15_10_final_ensemble_config.json",

    "Optimized ensemble config":
        REPORTS_DIR / "phase15_9_optimized_ensemble_config.json",

    "Time split assignments":
        REPORTS_DIR / "phase15_1_time_split_assignments.parquet",

    "Feature-engineered data":
        PROCESSED_DIR / "delhi_10_station_feature_engineered.parquet",

    "Phase 15 checkpoint":
        CHECKPOINT_DIR / "phase14_final_checkpoint.json",

    "Session status":
        CHECKPOINT_DIR / "session_status.json",
}

for name, path in required_files.items():
    exists = path.exists()
    print(f"{'OK' if exists else 'MISSING':8s} | {name:30s} | {path}")

print("\nCurrent session status:")
status_path = CHECKPOINT_DIR / "session_status.json"

if status_path.exists():
    with open(status_path, "r") as f:
        session_status = json.load(f)
    print(json.dumps(session_status, indent=2, default=str))
else:
    print("session_status.json not found.")

print("\nDrive restoration complete.")

In [ ]:
# ============================================================
# PHASE 15.15 — FINAL MODEL & ENSEMBLE FREEZE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import os
import pyarrow.parquet as pq

PROJECT_ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"

FREEZE_DIR = REPORTS_DIR / "phase15_15_final_freeze"
FREEZE_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "AQI_t_plus_1"

# ------------------------------------------------------------
# 1. Required artifact paths
# ------------------------------------------------------------

extra_trees_path = MODELS_DIR / "phase15_10_final_tuned_extra_trees_trainval.joblib"
random_forest_path = MODELS_DIR / "phase15_10_final_tuned_random_forest_trainval.joblib"

ensemble_config_path = REPORTS_DIR / "phase15_10_final_ensemble_config.json"
optimized_config_path = REPORTS_DIR / "phase15_9_optimized_ensemble_config.json"

feature_data_path = PROCESSED_DIR / "delhi_10_station_feature_engineered.parquet"
evaluation_protocol_path = REPORTS_DIR / "phase15_1_evaluation_protocol.json"

for path in [
    extra_trees_path,
    random_forest_path,
    ensemble_config_path,
    optimized_config_path,
    feature_data_path,
]:
    assert path.exists(), f"Required artifact missing: {path}"

# ------------------------------------------------------------
# 2. Read existing ensemble configurations
# ------------------------------------------------------------

with open(ensemble_config_path, "r") as f:
    final_ensemble_config = json.load(f)

with open(optimized_config_path, "r") as f:
    optimized_ensemble_config = json.load(f)

evaluation_protocol = {}
if evaluation_protocol_path.exists():
    with open(evaluation_protocol_path, "r") as f:
        evaluation_protocol = json.load(f)

# ------------------------------------------------------------
# 3. Freeze the exact model feature contract from Parquet schema
#    No full dataframe load required.
# ------------------------------------------------------------

parquet_file = pq.ParquetFile(feature_data_path)
all_columns = parquet_file.schema.names

excluded_columns = {
    "Datetime",
    TARGET,
}

feature_columns = [
    c for c in all_columns
    if c not in excluded_columns and not c.startswith("Unnamed")
]

print("=" * 70)
print("FEATURE CONTRACT")
print("=" * 70)
print(f"Total Parquet columns : {len(all_columns)}")
print(f"Model features        : {len(feature_columns)}")

if len(feature_columns) != 73:
    print("\nWARNING: Expected 73 model features.")
    print("Detected features:")
    for i, col in enumerate(feature_columns, start=1):
        print(f"{i:02d}. {col}")
    raise ValueError(
        f"Expected exactly 73 model features, found {len(feature_columns)}."
    )

# ------------------------------------------------------------
# 4. Validate feature groups
# ------------------------------------------------------------

current_features = [
    c for c in feature_columns
    if not any(token in c for token in ["_lag_", "_rolling_"])
    and c not in {
        "Hour", "Day", "Month", "Year",
        "DayOfWeek", "IsWeekend",
        "IsStubbleSeason", "IsDiwaliWindow",
        "StationId"
    }
]

lag_features = [c for c in feature_columns if "_lag_" in c]
rolling_features = [c for c in feature_columns if "_rolling_" in c]

calendar_features = [
    c for c in feature_columns
    if c in {
        "Hour", "Day", "Month", "Year",
        "DayOfWeek", "IsWeekend"
    }
]

event_features = [
    c for c in feature_columns
    if c in {
        "IsStubbleSeason",
        "IsDiwaliWindow"
    }
]

station_features = [
    c for c in feature_columns
    if c == "StationId"
]

feature_groups = {
    "Current": current_features,
    "Lag": lag_features,
    "Rolling": rolling_features,
    "Calendar": calendar_features,
    "Events": event_features,
    "Station": station_features,
}

print("\nFeature group counts:")
for group, cols in feature_groups.items():
    print(f"{group:12s}: {len(cols)}")

assert len(lag_features) == 36
assert len(rolling_features) == 16
assert len(calendar_features) == 6
assert len(event_features) == 2
assert len(station_features) == 1

# ------------------------------------------------------------
# 5. Final ensemble weights
# ------------------------------------------------------------

ET_WEIGHT = 0.594466
RF_WEIGHT = 0.405534

# Use saved config values when available
for cfg in [optimized_ensemble_config, final_ensemble_config]:
    for key in ["extra_trees_weight", "et_weight"]:
        if key in cfg:
            ET_WEIGHT = float(cfg[key])

    for key in ["random_forest_weight", "rf_weight"]:
        if key in cfg:
            RF_WEIGHT = float(cfg[key])

print("\nFinal ensemble weights:")
print(f"Extra Trees    : {ET_WEIGHT:.6f}")
print(f"Random Forest  : {RF_WEIGHT:.6f}")
print(f"Weight sum     : {ET_WEIGHT + RF_WEIGHT:.6f}")

assert abs((ET_WEIGHT + RF_WEIGHT) - 1.0) < 1e-6

# ------------------------------------------------------------
# 6. Final untouched holdout metrics
#    These are the already-evaluated Phase 15.12 results.
#    They are recorded here; NO holdout reevaluation occurs.
# ------------------------------------------------------------

final_holdout_metrics = {
    "model": "Final Weighted Ensemble",
    "holdout_rows": 68352,
    "stations": 10,
    "holdout_start": "2019-09-09 09:00:00",
    "holdout_end": "2020-06-30 23:00:00",

    "MAE": 3.216121,
    "RMSE": 7.658390,
    "R2": 0.996634,

    "within_5_AQI_percent": 85.249883,
    "within_10_AQI_percent": 94.605864,
    "within_20_AQI_percent": 98.090765,

    "mean_error": -0.430216,
    "median_error": -0.229319,
    "p90_abs_error": 6.588741,
    "p95_abs_error": 10.554171,
    "p99_abs_error": 31.326013,
    "max_abs_error": 205.280197,

    "holdout_status": "evaluated_once_and_frozen",
}

# ------------------------------------------------------------
# 7. Training / validation / holdout protocol
# ------------------------------------------------------------

evaluation_summary = {
    "train_rows": 166845,
    "validation_rows": 69097,
    "final_holdout_rows": 68352,
    "unused_gap_rows": 19,

    "train_start": "2015-01-01 15:00:00",
    "train_end": "2018-11-17 16:00:00",

    "validation_start": "2018-11-17 18:00:00",
    "validation_end": "2019-09-09 07:00:00",

    "holdout_start": "2019-09-09 09:00:00",
    "holdout_end": "2020-06-30 23:00:00",

    "temporal_embargo": "1 hour",
    "split_type": "chronological timestamp split",
    "station_alignment": "all 10 stations at a timestamp remain in same split",
    "random_train_test_split_used": False,
}

# ------------------------------------------------------------
# 8. Model artifact hashes
#    Streamed hashing avoids loading model files into RAM.
# ------------------------------------------------------------

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("\nCalculating model artifact SHA-256 hashes...")
print("This may take a little time because the model files are large.")

artifact_manifest = {
    "extra_trees": {
        "path": str(extra_trees_path),
        "size_bytes": extra_trees_path.stat().st_size,
        "sha256": sha256_file(extra_trees_path),
    },
    "random_forest": {
        "path": str(random_forest_path),
        "size_bytes": random_forest_path.stat().st_size,
        "sha256": sha256_file(random_forest_path),
    },
}

# ------------------------------------------------------------
# 9. Final freeze record
# ------------------------------------------------------------

freeze_record = {
    "phase": "15.15",
    "phase_name": "Final Model & Ensemble Freeze",
    "freeze_timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "status": "FROZEN",

    "final_models": {
        "model_1": "Tuned Extra Trees",
        "model_1_artifact": str(extra_trees_path),

        "model_2": "Tuned Random Forest",
        "model_2_artifact": str(random_forest_path),

        "models_retrained_during_freeze": False,
        "models_retuned_during_freeze": False,
    },

    "ensemble": {
        "type": "weighted_average",
        "extra_trees_weight": ET_WEIGHT,
        "random_forest_weight": RF_WEIGHT,
        "weight_sum": ET_WEIGHT + RF_WEIGHT,
    },

    "target": TARGET,

    "feature_contract": {
        "feature_count": len(feature_columns),
        "features": feature_columns,
        "groups": feature_groups,
    },

    "evaluation_protocol": evaluation_summary,

    "final_holdout_metrics": final_holdout_metrics,

    "artifact_manifest": artifact_manifest,

    "holdout_integrity": {
        "holdout_used_for_training": False,
        "holdout_used_for_tuning": False,
        "holdout_used_for_weight_optimization": False,
        "holdout_used_for_feature_selection": False,
        "holdout_used_for_xai_selection": False,
        "holdout_reevaluated_during_freeze": False,
    },

    "next_phase": "16.0 Deployment / Live Inference Preparation",
}

freeze_json_path = FREEZE_DIR / "phase15_15_final_freeze.json"

with open(freeze_json_path, "w") as f:
    json.dump(freeze_record, f, indent=2, default=str)

# Also save the exact feature list as a CSV
feature_contract_path = FREEZE_DIR / "phase15_15_feature_contract.csv"

import pandas as pd

feature_contract_df = pd.DataFrame({
    "feature_index": range(1, len(feature_columns) + 1),
    "feature_name": feature_columns,
})

feature_contract_df.to_csv(feature_contract_path, index=False)

print("\n" + "=" * 70)
print("PHASE 15.15 COMPLETE — FINAL MODEL IS FROZEN")
print("=" * 70)
print(f"Freeze record      : {freeze_json_path}")
print(f"Feature contract   : {feature_contract_path}")
print(f"Feature count      : {len(feature_columns)}")
print(f"ET weight          : {ET_WEIGHT:.6f}")
print(f"RF weight          : {RF_WEIGHT:.6f}")
print(f"Final RMSE         : {final_holdout_metrics['RMSE']:.6f}")
print(f"Final MAE          : {final_holdout_metrics['MAE']:.6f}")
print(f"Final R²           : {final_holdout_metrics['R2']:.6f}")
print(f"Final ±10 AQI      : {final_holdout_metrics['within_10_AQI_percent']:.6f}%")
print("Holdout status     : evaluated once / frozen")

In [ ]:
# ============================================================
# PHASE 15.15 — FINAL MODEL & ENSEMBLE FREEZE
# CORRECTED: EXCLUDE StationName FROM MODEL FEATURE CONTRACT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 0. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"

FREEZE_DIR = REPORTS_DIR / "phase15_15_final_freeze"
FREEZE_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "AQI_t_plus_1"

print("=" * 70)
print("PHASE 15.15 — FINAL MODEL & ENSEMBLE FREEZE")
print("=" * 70)

# ------------------------------------------------------------
# 1. REQUIRED ARTIFACTS
# ------------------------------------------------------------

extra_trees_path = (
    MODELS_DIR / "phase15_10_final_tuned_extra_trees_trainval.joblib"
)

random_forest_path = (
    MODELS_DIR / "phase15_10_final_tuned_random_forest_trainval.joblib"
)

ensemble_config_path = (
    REPORTS_DIR / "phase15_10_final_ensemble_config.json"
)

optimized_config_path = (
    REPORTS_DIR / "phase15_9_optimized_ensemble_config.json"
)

feature_data_path = (
    PROCESSED_DIR / "delhi_10_station_feature_engineered.parquet"
)

required_artifacts = {
    "Final Extra Trees model": extra_trees_path,
    "Final Random Forest model": random_forest_path,
    "Final ensemble config": ensemble_config_path,
    "Optimized ensemble config": optimized_config_path,
    "Feature-engineered data": feature_data_path,
}

for name, path in required_artifacts.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    print(f"OK      | {name}")

# ------------------------------------------------------------
# 2. LOAD SAVED CONFIGS
# ------------------------------------------------------------

with open(ensemble_config_path, "r") as f:
    final_ensemble_config = json.load(f)

with open(optimized_config_path, "r") as f:
    optimized_ensemble_config = json.load(f)

# ------------------------------------------------------------
# 3. READ PARQUET SCHEMA ONLY
#    No full dataframe is loaded.
# ------------------------------------------------------------

parquet_file = pq.ParquetFile(feature_data_path)
all_columns = parquet_file.schema.names

print("\n" + "=" * 70)
print("PARQUET SCHEMA")
print("=" * 70)

print(f"Total Parquet columns : {len(all_columns)}")

# IMPORTANT:
# StationName is descriptive metadata and was NOT part of the
# 73-feature model contract.
metadata_columns = {
    "Datetime",
    TARGET,
    "StationName",
}

feature_columns = [
    c for c in all_columns
    if c not in metadata_columns
    and not c.startswith("Unnamed")
]

print(f"Model features        : {len(feature_columns)}")

# ------------------------------------------------------------
# 4. EXACT 73-FEATURE CONTRACT CHECK
# ------------------------------------------------------------

if len(feature_columns) != 73:
    print("\nERROR: Feature count is not 73.")
    print("Detected model features:")
    for i, col in enumerate(feature_columns, start=1):
        print(f"{i:02d}. {col}")
    raise ValueError(
        f"Expected exactly 73 model features, found {len(feature_columns)}."
    )

print("\n73-feature contract confirmed.")

# ------------------------------------------------------------
# 5. FEATURE GROUPS
# ------------------------------------------------------------

# 12 current-time features
expected_current = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
]

# 36 lag features
lag_features = [
    c for c in feature_columns
    if "_lag_" in c
]

# 16 rolling features
rolling_features = [
    c for c in feature_columns
    if "_rolling_" in c
]

# 6 calendar features
expected_calendar = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
]

# 2 event features
expected_events = [
    "IsStubbleSeason",
    "IsDiwaliWindow",
]

# 1 station feature
expected_station = [
    "StationId",
]

# Current features are everything in the expected current list.
current_features = [
    c for c in expected_current
    if c in feature_columns
]

calendar_features = [
    c for c in expected_calendar
    if c in feature_columns
]

event_features = [
    c for c in expected_events
    if c in feature_columns
]

station_features = [
    c for c in expected_station
    if c in feature_columns
]

feature_groups = {
    "Current": current_features,
    "Lag": lag_features,
    "Rolling": rolling_features,
    "Calendar": calendar_features,
    "Events": event_features,
    "Station": station_features,
}

print("\n" + "=" * 70)
print("FEATURE GROUP COUNTS")
print("=" * 70)

for group, cols in feature_groups.items():
    print(f"{group:12s}: {len(cols)}")

# Exact expected counts from the completed Phase 5 contract
assert len(current_features) == 12, (
    f"Expected 12 current features, found {len(current_features)}"
)

assert len(lag_features) == 36, (
    f"Expected 36 lag features, found {len(lag_features)}"
)

assert len(rolling_features) == 16, (
    f"Expected 16 rolling features, found {len(rolling_features)}"
)

assert len(calendar_features) == 6, (
    f"Expected 6 calendar features, found {len(calendar_features)}"
)

assert len(event_features) == 2, (
    f"Expected 2 event features, found {len(event_features)}"
)

assert len(station_features) == 1, (
    f"Expected 1 station feature, found {len(station_features)}"
)

assert sum(len(v) for v in feature_groups.values()) == 73

# ------------------------------------------------------------
# 6. VERIFY NO FORBIDDEN / METADATA FEATURES ENTERED MODEL
# ------------------------------------------------------------

forbidden_features = {
    "Datetime",
    "StationName",
    TARGET,
    "AQI_Bucket",
    "Season",
}

unexpected_forbidden = [
    c for c in feature_columns
    if c in forbidden_features
]

if unexpected_forbidden:
    raise ValueError(
        f"Forbidden metadata/obsolete features detected: "
        f"{unexpected_forbidden}"
    )

print("\nNo metadata/obsolete feature is included in the 73-feature contract.")

# ------------------------------------------------------------
# 7. VERIFY EXACT FEATURE ORDER
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL 73-FEATURE CONTRACT")
print("=" * 70)

for i, col in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {col}")

# ------------------------------------------------------------
# 8. FINAL ENSEMBLE WEIGHTS
# ------------------------------------------------------------

# Known optimized weights from Phase 15.9
ET_WEIGHT = 0.594466
RF_WEIGHT = 0.405534

# Use saved configuration when matching keys are available.
for cfg in [optimized_ensemble_config, final_ensemble_config]:

    for key in [
        "extra_trees_weight",
        "et_weight",
    ]:
        if key in cfg:
            ET_WEIGHT = float(cfg[key])

    for key in [
        "random_forest_weight",
        "rf_weight",
        "random_forest_weight",
    ]:
        if key in cfg:
            RF_WEIGHT = float(cfg[key])

print("\n" + "=" * 70)
print("FINAL ENSEMBLE WEIGHTS")
print("=" * 70)

print(f"Extra Trees weight   : {ET_WEIGHT:.6f}")
print(f"Random Forest weight : {RF_WEIGHT:.6f}")
print(f"Weight sum           : {ET_WEIGHT + RF_WEIGHT:.6f}")

assert abs((ET_WEIGHT + RF_WEIGHT) - 1.0) < 1e-6

# ------------------------------------------------------------
# 9. FINAL HOLDOUT RESULTS
#
# IMPORTANT:
# These are the already completed Phase 15.12 results.
# We are recording them here only.
# NO holdout data is loaded or reevaluated.
# ------------------------------------------------------------

final_holdout_metrics = {
    "model": "Final Weighted Ensemble",

    "holdout_rows": 68352,
    "stations": 10,

    "holdout_start": "2019-09-09 09:00:00",
    "holdout_end": "2020-06-30 23:00:00",

    "MAE": 3.216121,
    "RMSE": 7.658390,
    "R2": 0.996634,

    "within_5_AQI_percent": 85.249883,
    "within_10_AQI_percent": 94.605864,
    "within_20_AQI_percent": 98.090765,

    "mean_error": -0.430216,
    "median_error": -0.229319,

    "p90_abs_error": 6.588741,
    "p95_abs_error": 10.554171,
    "p99_abs_error": 31.326013,
    "max_abs_error": 205.280197,

    "holdout_status": "evaluated_once_and_frozen",
}

# ------------------------------------------------------------
# 10. FINAL EVALUATION PROTOCOL
# ------------------------------------------------------------

evaluation_summary = {
    "train_rows": 166845,
    "validation_rows": 69097,
    "final_holdout_rows": 68352,
    "unused_gap_rows": 19,

    "train_start": "2015-01-01 15:00:00",
    "train_end": "2018-11-17 16:00:00",

    "validation_start": "2018-11-17 18:00:00",
    "validation_end": "2019-09-09 07:00:00",

    "holdout_start": "2019-09-09 09:00:00",
    "holdout_end": "2020-06-30 23:00:00",

    "temporal_embargo": "1 hour",

    "split_type": (
        "chronological timestamp split; all 10 stations at "
        "the same timestamp remain together"
    ),

    "random_train_test_split_used": False,
}

# ------------------------------------------------------------
# 11. HASH THE FINAL MODEL ARTIFACTS
# ------------------------------------------------------------

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

print("\n" + "=" * 70)
print("MODEL ARTIFACT HASHING")
print("=" * 70)

print("Hashing Extra Trees model...")
et_hash = sha256_file(extra_trees_path)

print("Hashing Random Forest model...")
rf_hash = sha256_file(random_forest_path)

artifact_manifest = {
    "extra_trees": {
        "path": str(extra_trees_path),
        "size_bytes": extra_trees_path.stat().st_size,
        "sha256": et_hash,
    },

    "random_forest": {
        "path": str(random_forest_path),
        "size_bytes": random_forest_path.stat().st_size,
        "sha256": rf_hash,
    },
}

# ------------------------------------------------------------
# 12. CREATE FORMAL FINAL FREEZE RECORD
# ------------------------------------------------------------

freeze_record = {
    "phase": "15.15",
    "phase_name": "Final Model & Ensemble Freeze",

    "freeze_timestamp_utc": (
        datetime.now(timezone.utc).isoformat()
    ),

    "status": "FROZEN",

    "final_models": {
        "model_1": "Tuned Extra Trees",
        "model_1_artifact": str(extra_trees_path),

        "model_2": "Tuned Random Forest",
        "model_2_artifact": str(random_forest_path),

        "models_retrained_during_freeze": False,
        "models_retuned_during_freeze": False,
    },

    "ensemble": {
        "type": "weighted_average",
        "extra_trees_weight": ET_WEIGHT,
        "random_forest_weight": RF_WEIGHT,
        "weight_sum": ET_WEIGHT + RF_WEIGHT,
    },

    "target": TARGET,

    "feature_contract": {
        "feature_count": 73,
        "features": feature_columns,
        "groups": feature_groups,

        "excluded_metadata_columns": [
            "Datetime",
            "StationName",
            TARGET,
        ],
    },

    "evaluation_protocol": evaluation_summary,

    "final_holdout_metrics": final_holdout_metrics,

    "artifact_manifest": artifact_manifest,

    "holdout_integrity": {
        "holdout_used_for_training": False,
        "holdout_used_for_tuning": False,
        "holdout_used_for_weight_optimization": False,
        "holdout_used_for_feature_selection": False,
        "holdout_used_for_xai_selection": False,
        "holdout_reevaluated_during_freeze": False,
    },

    "next_phase": "16.0 Deployment / Live Inference Preparation",
}

freeze_json_path = (
    FREEZE_DIR / "phase15_15_final_freeze.json"
)

with open(freeze_json_path, "w") as f:
    json.dump(
        freeze_record,
        f,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 13. SAVE EXACT FEATURE CONTRACT CSV
# ------------------------------------------------------------

feature_contract_path = (
    FREEZE_DIR / "phase15_15_feature_contract.csv"
)

feature_contract_df = pd.DataFrame({
    "feature_index": range(1, 74),
    "feature_name": feature_columns,
})

feature_contract_df.to_csv(
    feature_contract_path,
    index=False
)

# ------------------------------------------------------------
# 14. FINAL VERIFICATION
# ------------------------------------------------------------

assert freeze_json_path.exists()
assert feature_contract_path.exists()

print("\n" + "=" * 70)
print("PHASE 15.15 COMPLETE — FINAL MODEL FROZEN")
print("=" * 70)

print(f"Freeze record       : {freeze_json_path}")
print(f"Feature contract    : {feature_contract_path}")

print()
print(f"Feature count       : {len(feature_columns)}")
print(f"ET weight           : {ET_WEIGHT:.6f}")
print(f"RF weight           : {RF_WEIGHT:.6f}")

print()
print("FINAL HOLDOUT RESULTS")
print(f"MAE                 : {final_holdout_metrics['MAE']:.6f}")
print(f"RMSE                : {final_holdout_metrics['RMSE']:.6f}")
print(f"R2                  : {final_holdout_metrics['R2']:.6f}")
print(
    f"Within ±10 AQI      : "
    f"{final_holdout_metrics['within_10_AQI_percent']:.6f}%"
)

print()
print("Holdout status      : evaluated once / frozen")
print("Retraining          : NOT performed")
print("Retuning            : NOT performed")
print("Holdout reevaluation: NOT performed")
print("Next phase          : 16.0")

In [ ]:
# ============================================================
# PHASE 15.16 — FINAL PHASE 15 CHECKPOINT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

# ------------------------------------------------------------
# 0. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Delhi-AQI-Prediction")

CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"

FREEZE_DIR = REPORTS_DIR / "phase15_15_final_freeze"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FREEZE_PATH = (
    FREEZE_DIR / "phase15_15_final_freeze.json"
)

assert FREEZE_PATH.exists(), (
    f"Phase 15.15 freeze file not found:\n{FREEZE_PATH}"
)

# ------------------------------------------------------------
# 1. LOAD FINAL FREEZE RECORD
# ------------------------------------------------------------

with open(FREEZE_PATH, "r") as f:
    freeze = json.load(f)

assert freeze["status"] == "FROZEN"

# ------------------------------------------------------------
# 2. BUILD FINAL PHASE 15 CHECKPOINT
# ------------------------------------------------------------

checkpoint = {
    "project": "Delhi AQI t+1 Forecasting",

    "phase": "15.16",

    "phase_name": "Final Phase 15 Checkpoint",

    "checkpoint_timestamp_utc": (
        datetime.now(timezone.utc).isoformat()
    ),

    "phase_15_status": "COMPLETE",

    "completed_phases": [
        "15.1 Evaluation Protocol",
        "15.2 Base Model Competition",
        "15.3 Leaderboard + Persistence",
        "15.4 Random Forest Tuning",
        "15.5 Extra Trees Tuning",
        "15.6 XGBoost Tuning",
        "15.7 Top Two Model Selection",
        "15.8 Top Two Prediction Agreement",
        "15.9 Ensemble Weight Optimization",
        "15.10 Individual vs Ensemble + Final Train/Validation Refit",
        "15.11 Model-Agnostic XAI + Local SHAP",
        "15.12 Final Untouched Holdout Evaluation",
        "15.15 Final Model & Ensemble Freeze",
        "15.16 Final Phase 15 Checkpoint",
    ],

    # --------------------------------------------------------
    # FINAL MODEL
    # --------------------------------------------------------

    "final_models": freeze["final_models"],

    "ensemble": freeze["ensemble"],

    "target": freeze["target"],

    # --------------------------------------------------------
    # EXACT FEATURE CONTRACT
    # --------------------------------------------------------

    "feature_contract": freeze["feature_contract"],

    # --------------------------------------------------------
    # FINAL HOLDOUT RESULTS
    # --------------------------------------------------------

    "final_holdout_metrics": freeze["final_holdout_metrics"],

    # --------------------------------------------------------
    # HOLDOUT INTEGRITY
    # --------------------------------------------------------

    "holdout_integrity": freeze["holdout_integrity"],

    # --------------------------------------------------------
    # FREEZE ARTIFACT
    # --------------------------------------------------------

    "freeze_artifact": str(FREEZE_PATH),

    # --------------------------------------------------------
    # NEXT PHASE
    # --------------------------------------------------------

    "next_phase": "16.0 Deployment / Live Inference Preparation",

    # --------------------------------------------------------
    # IMPORTANT PRODUCTION RULES
    # --------------------------------------------------------

    "phase_16_constraints": {
        "do_not_retrain_from_holdout": True,
        "do_not_retune_using_holdout": True,
        "do_not_change_ensemble_weights": True,
        "do_not_change_feature_contract": True,
        "preserve_73_features": True,
        "preserve_target": "AQI_t_plus_1",
        "preserve_station_scope": 10,
    },
}

CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "phase15_16_final_checkpoint.json"
)

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(
        checkpoint,
        f,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 3. UPDATE PERSISTENT SESSION STATUS
# ------------------------------------------------------------

SESSION_STATUS_PATH = (
    CHECKPOINT_DIR / "session_status.json"
)

if SESSION_STATUS_PATH.exists():
    with open(SESSION_STATUS_PATH, "r") as f:
        session_status = json.load(f)
else:
    session_status = {}

session_status.update({
    "project": "Delhi AQI t+1 Forecasting",

    "current_phase": "15.16",

    "current_phase_status": "COMPLETED",

    "phase_15_status": "COMPLETED",

    "last_completed_phase": "15.16",

    "next_phase": "16.0",

    "holdout_status": (
        "evaluated_once / frozen_for_reporting"
    ),

    "final_model_frozen": True,

    "final_feature_count": 73,

    "final_target": "AQI_t_plus_1",

    "final_station_count": 10,

    "final_ensemble": {
        "extra_trees_weight": (
            checkpoint["ensemble"]["extra_trees_weight"]
        ),
        "random_forest_weight": (
            checkpoint["ensemble"]["random_forest_weight"]
        ),
    },

    "final_holdout_metrics": {
        "MAE": checkpoint["final_holdout_metrics"]["MAE"],
        "RMSE": checkpoint["final_holdout_metrics"]["RMSE"],
        "R2": checkpoint["final_holdout_metrics"]["R2"],
        "within_10_AQI_percent": (
            checkpoint["final_holdout_metrics"]
            ["within_10_AQI_percent"]
        ),
    },

    "last_checkpoint": str(CHECKPOINT_PATH),

    "freeze_artifact": str(FREEZE_PATH),
})

with open(SESSION_STATUS_PATH, "w") as f:
    json.dump(
        session_status,
        f,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 4. VERIFY FILES
# ------------------------------------------------------------

assert CHECKPOINT_PATH.exists()
assert SESSION_STATUS_PATH.exists()

# Reload to verify that the saved JSON is valid.
with open(CHECKPOINT_PATH, "r") as f:
    checkpoint_verify = json.load(f)

with open(SESSION_STATUS_PATH, "r") as f:
    status_verify = json.load(f)

assert checkpoint_verify["phase_15_status"] == "COMPLETE"
assert status_verify["next_phase"] == "16.0"
assert status_verify["final_model_frozen"] is True
assert status_verify["final_feature_count"] == 73

# ------------------------------------------------------------
# 5. FINAL SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 15.16 COMPLETE")
print("=" * 70)

print(f"Checkpoint saved : {CHECKPOINT_PATH}")
print(f"Session status   : {SESSION_STATUS_PATH}")

print("\nFINAL PHASE 15 STATUS")
print("---------------------")
print("Phase 15         : COMPLETE")
print("Final model 1    : Tuned Extra Trees")
print("Final model 2    : Tuned Random Forest")
print("Ensemble         : Weighted ensemble")
print("ET weight        : 0.594466")
print("RF weight        : 0.405534")
print("Feature count    : 73")
print("Target           : AQI_t_plus_1")
print("Stations         : 10")

print("\nFINAL HOLDOUT")
print("-------------")
print("MAE              : 3.216121")
print("RMSE             : 7.658390")
print("R2               : 0.996634")
print("Within ±10 AQI   : 94.605864%")

print("\nINTEGRITY")
print("--------")
print("Holdout reused   : NO")
print("Retrained        : NO")
print("Retuned          : NO")
print("Weights changed  : NO")
print("Features changed : NO")

print("\nNEXT PHASE")
print("----------")
print("16.0 — Deployment / Live Inference Preparation")